# BTCUSD fused multi-TF transformer research (v11)

One shared encoder, five **independent** OHLCV streams (5m / 10m / 30m / 1h / 2h),
softmax TF fusion weights, and three 2-class heads: **+30m / +1h / +2h**.
Training labels are BEAR / BULL; NEUTRAL (0.5 ATR on h30m/h1h, 0.75 ATR on
h2h) is **ignored** in cross-entropy (not a class). HOLD at live time comes
from LOW grade or ``min_probability``. There are **no MFE/MAE path heads**.
Optuna searches dropout / weight_decay / lr on val CE **before** the main
train. 5m is an input timeframe, not a forecast head. 10m is two closed 5m
bars built **outside** the encoder — it is an input TF, not a trading head.

This notebook is the Colab trainer for the live fused bundle
(`JackSparrow_Transformer_BTCUSD_mtf_fusion`). Upload **this notebook only**,
or run it from Windows via WSL:

    powershell -File scripts/colab/run_colab_cli.ps1

Historical OHLCV comes from the Delta Exchange India public API.

Edit repo `.py` files and regenerate:

    python scripts/colab/build_next_candle_research_notebook.py

**Colab setup:** Runtime → Change runtime type → **T4 GPU**.


## 01 Env

In [ ]:
# Colab ships torch/pandas/numpy; install research extras.
!pip install -q --upgrade-strategy only-if-needed pyarrow onnx onnxruntime requests scikit-learn optuna shap matplotlib seaborn scipy

import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch

print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected — training will run on CPU and be much slower.")
    print("Runtime -> Change runtime type -> select a GPU (T4), then re-run this cell.")


## Feature contract (agent integration)

In [ ]:
"""Constants shared between per-TF Colab training and agent inference."""

from __future__ import annotations

from typing import Any, Dict, Tuple

# Bump when FEATURE_COLS, label heads, or ONNX outputs change (requires retrain).
FEATURE_CONTRACT_VERSION_V6 = "transformer_btcusd_per_tf_features_v6"
FEATURE_CONTRACT_VERSION_V7 = "transformer_btcusd_per_tf_features_v7"
FEATURE_CONTRACT_VERSION_V8 = "transformer_btcusd_per_tf_features_v8"
FEATURE_CONTRACT_VERSION_V9 = "transformer_btcusd_per_tf_features_v9"
FEATURE_CONTRACT_VERSION_V10 = "transformer_btcusd_mtf_fusion_v10"
FEATURE_CONTRACT_VERSION_V11 = "transformer_btcusd_mtf_fusion_v11"
# Live per-TF bundles remain v9. The fused model uses V11 (2-class, no h10m head).
FEATURE_CONTRACT_VERSION = FEATURE_CONTRACT_VERSION_V9

SUPPORTED_RESOLUTIONS: Tuple[str, ...] = ("5m", "15m", "30m", "1h", "2h")
FUSION_INPUT_RESOLUTIONS: Tuple[str, ...] = ("5m", "10m", "30m", "1h", "2h")
FUSION_BUNDLE_DIR_NAME = "JackSparrow_Transformer_BTCUSD_mtf_fusion"
FUSION_MODEL_FAMILY = "jacksparrow_transformer_btcusd_mtf_fusion"
FUSION_ONNX_FILENAME = "btcusd_mtf_fusion.onnx"
FUSION_WINDOW_LEN: int = 64
FUSION_EMBARGO_BARS: int = 24

RESOLUTION_MINUTES: Dict[str, int] = {
    "5m": 5,
    "10m": 10,
    "15m": 15,
    "30m": 30,
    "1h": 60,
    "2h": 120,
}

TF_KEYS: Tuple[str, ...] = tuple(f"tf_{r}" for r in SUPPORTED_RESOLUTIONS)

HTF_SOURCE_TFS: Tuple[str, ...] = ("15m", "30m", "1h", "2h")
HTF_FEATURE_FIELDS: Tuple[str, ...] = (
    "structure_bias",
    "trend_efficiency",
    "ema21_slope_atr",
    "dist_support_atr",
    "dist_resistance_atr",
    "range_width_atr",
)

NATIVE_STRUCTURE_COLS: Tuple[str, ...] = (
    "hh_count",
    "hl_count",
    "lh_count",
    "ll_count",
    "structure_bias",
    "last_swing_dir",
    "bars_since_swing",
    "swing_amp_atr",
    "unconfirmed_ext_atr",
    "trend_efficiency",
    "displacement_atr",
    "pct_with_trend",
    "price_vs_ema9_atr",
    "price_vs_ema21_atr",
    "price_vs_ema50_atr",
    "price_vs_ema200_atr",
    "ema9_vs_21_atr",
    "ema21_vs_50_atr",
    "ema50_vs_200_atr",
    "ema21_slope_atr",
    "ema50_slope_atr",
    "dist_to_support_atr",
    "dist_to_resistance_atr",
    "support_touch_count",
    "resistance_touch_count",
    "range_width_atr",
    "range_width_pctile",
    "atr_contraction",
    "breakout_size_atr",
    "breakout_vol_ratio",
    "pre_breakout_comp",
    "bars_since_breakout",
    "retest_dist_atr",
    "failed_break",
    "peak_diff_atr",
    "trough_diff_atr",
    "peak_sep_bars",
    "trough_sep_bars",
    "dist_neck_atr",
    "high_slope_atr",
    "low_slope_atr",
    "convergence",
    "width_now_atr",
    "pole_disp_atr",
    "flag_width_atr",
    "flag_slope_atr",
)

HTF_STRUCTURE_COLS: Tuple[str, ...] = tuple(
    f"htf_{tf}_{field}" for tf in HTF_SOURCE_TFS for field in HTF_FEATURE_FIELDS
)

_V4_FEATURE_COLS: Tuple[str, ...] = (
    "ret_1",
    "rv_16",
    "rv_96",
    "ema50_dist_pct",
    "macd_hist",
    "rsi_14",
    "adx_14",
    "obv_z",
    "vol_z",
    "body_ratio",
    "upper_wick_ratio",
    "lower_wick_ratio",
    "close_loc",
    "range_atr",
    "body_atr",
    "gap_atr",
    "inside_bar",
    "outside_bar",
    "engulf_score",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "dist_to_resistance_pct",
    "dist_to_support_pct",
    "funding_rate",
    "oi_z",
    "funding_zscore",
    "funding_mom",
    "funding_rate_roc",
    "oi_change_2",
    "oi_delta_z",
    "oi_price_divergence",
    "oi_acceleration",
    "funding_x_oi",
)

# Native all-TF continuous columns (v4 plus causal structure/geometry).
FEATURE_COLS: Tuple[str, ...] = _V4_FEATURE_COLS + NATIVE_STRUCTURE_COLS

PATH_LABEL_COLS: Tuple[str, ...] = (
    "mfe",
    "mae",
    "future_volatility",
    "trend_strength",
    "drawdown_before_mfe",
    "future_oi_change_pct",
    "future_volume_change_pct",
    "candle_follow_through_atr",
    "structure_delta",
)

CONTINUOUS_LABEL_COLS: Tuple[str, ...] = PATH_LABEL_COLS

PATH_LABEL_HORIZON_BARS: int = 8

# Legacy 8h reference (btcusd_15m_transformer used 32 bars on 15m).
REFERENCE_LABEL_HORIZON_MINUTES: int = 480

DEFAULT_PATH_LABEL_HORIZON_MINUTES: Dict[str, int] = {
    "5m": 240,
    "15m": 240,
    "30m": 480,
    "1h": 480,
    "2h": 480,
}

# Volume change excluded from loss (dominates shared encoder).
DEFAULT_CONTINUOUS_LOSS_WEIGHTS: Dict[str, float] = {
    "mfe": 0.5,
    "mae": 0.5,
    "future_volatility": 1.0,
    "trend_strength": 0.5,
    "drawdown_before_mfe": 0.5,
    "future_oi_change_pct": 0.25,
    "future_volume_change_pct": 0.0,
    "candle_follow_through_atr": 0.5,
    "structure_delta": 0.5,
}

REGIME_NAMES: Dict[int, str] = {
    0: "LOW",
    1: "NORMAL",
    2: "HIGH",
    3: "EXTREME",
}

CANDLE_CLASS_COL = "candle_class_id"
CANDLE_CLASS_CARDINALITY = 13
CANDLE_EMBED_DIM = 8

CANDLE_CLASS_NAMES: Dict[int, str] = {
    0: "FLAT_ZERO_RANGE",
    1: "DOJI_DRAGONFLY",
    2: "DOJI_GRAVESTONE",
    3: "DOJI_STANDARD",
    4: "MARUBOZU_BULL",
    5: "MARUBOZU_BEAR",
    6: "HAMMER_SHAPE",
    7: "INV_HAMMER_SHAPE",
    8: "SPINNING_TOP",
    9: "BELT_HOLD_BULL",
    10: "BELT_HOLD_BEAR",
    11: "STANDARD_BULL",
    12: "STANDARD_BEAR",
}

# Discrete training/inference columns (not continuous ONNX heads).
STRUCTURE_OUTCOME_COL = "future_structure_outcome"
FUTURE_CANDLE_COL = "future_candle_class"
STRUCTURE_OUTCOME_CARDINALITY = 6
N_AUX_CLASS_HEADS = 3  # vol regime + structure outcome + next-bar candle

# v7 next-candle structure (kept for legacy 5m ONNX decode).
NEXT_DIRECTION_COL = "next_direction"
NEXT_BODY_COL = "next_body"
NEXT_WICK_COL = "next_wick"
NEXT_RANGE_COL = "next_range"
CHART_PATTERN_COL = "chart_pattern_id"
VOLUME_STATE_COL = "volume_state"
VOLUME_CONFIRMS_COL = "volume_confirms"
PATTERN_ACTIVE_COL = "pattern_active"
SAMPLE_WEIGHT_COL = "sample_weight"

# v8 wall-clock behavior packets on the 5m grid (bars).
HORIZON_SPECS: Tuple[Tuple[str, int], ...] = (
    ("h5m", 1),
    ("h10m", 2),
    ("h15m", 3),
    ("h30m", 6),
    ("h1h", 12),
    ("h2h", 24),
)
HORIZON_KEYS: Tuple[str, ...] = tuple(key for key, _ in HORIZON_SPECS)
HORIZON_BARS_5M: Tuple[int, ...] = tuple(bars for _, bars in HORIZON_SPECS)
N_HORIZONS: int = len(HORIZON_SPECS)
MAX_V8_HORIZON_BARS: int = max(HORIZON_BARS_5M)
HORIZON_CONTINUOUS_FIELDS: Tuple[str, ...] = ("mfe", "mae", "vol", "trend_strength")
HORIZON_DIR_COLS: Tuple[str, ...] = tuple(f"{key}_dir" for key in HORIZON_KEYS)
HORIZON_STRUCTURE_COLS: Tuple[str, ...] = tuple(
    f"{key}_structure" for key in HORIZON_KEYS
)
V8_CONTINUOUS_LABEL_COLS: Tuple[str, ...] = tuple(
    f"{key}_{field}"
    for key in HORIZON_KEYS
    for field in HORIZON_CONTINUOUS_FIELDS
)
HORIZON_DIR_COLS_V7: Tuple[str, ...] = (
    "horizon_t1_dir",
    "horizon_t3_dir",
    "horizon_t6_dir",
    "horizon_t12_dir",
    "horizon_t24_dir",
)

NEXT_DIRECTION_CARDINALITY_V8 = 3
NEXT_DIRECTION_CARDINALITY = 5
NEXT_BODY_CARDINALITY = 3
NEXT_WICK_CARDINALITY = 4
NEXT_RANGE_CARDINALITY = 3
VOLUME_STATE_CARDINALITY = 3
CHART_PATTERN_CARDINALITY = 9

NEXT_DIRECTION_NAMES_V8: Dict[int, str] = {0: "BEARISH", 1: "NEUTRAL", 2: "BULLISH"}
NEXT_DIRECTION_NAMES: Dict[int, str] = {
    0: "STRONG_DOWN",
    1: "DOWN",
    2: "NEUTRAL",
    3: "UP",
    4: "STRONG_UP",
}
BULLISH_DIRECTION_NAMES = frozenset({"UP", "STRONG_UP", "BULLISH"})
BEARISH_DIRECTION_NAMES = frozenset({"DOWN", "STRONG_DOWN", "BEARISH"})
NEUTRAL_DIRECTION_NAMES = frozenset({"NEUTRAL"})
NEXT_BODY_NAMES: Dict[int, str] = {0: "SMALL", 1: "MEDIUM", 2: "LARGE"}
NEXT_WICK_NAMES: Dict[int, str] = {
    0: "BALANCED",
    1: "UPPER_REJECTION",
    2: "LOWER_REJECTION",
    3: "BOTH_REJECTION",
}
NEXT_RANGE_NAMES: Dict[int, str] = {0: "COMPRESSED", 1: "NORMAL", 2: "EXPANDED"}
VOLUME_STATE_NAMES: Dict[int, str] = {0: "DRY", 1: "NORMAL", 2: "EXPANSION"}
CHART_PATTERN_NAMES: Dict[int, str] = {
    0: "NONE",
    1: "FLAG_BULL",
    2: "FLAG_BEAR",
    3: "TRIANGLE",
    4: "DOUBLE_TOP",
    5: "DOUBLE_BOTTOM",
    6: "CHANNEL",
    7: "BREAKOUT",
    8: "FAILED_BREAK",
}

VOL_Z_EXPANSION = 0.5
VOL_Z_DRY = -0.5
BREAKOUT_VOL_CONFIRM = 1.2
HORIZON_DIR_ATR_WEAK = 0.5
HORIZON_DIR_ATR_STRONG = 2.0
HORIZON_DIR_ATR_DEADZONE = HORIZON_DIR_ATR_WEAK
# Fusion heads: h2h uses a wider ignore band so chop does not train the trunk.
FUSION_HORIZON_ATR_WEAK: Dict[str, float] = {
    "h30m": 0.5,
    "h1h": 0.5,
    "h2h": 0.75,
}

V7_STRUCTURE_LOSS_WEIGHTS: Dict[str, float] = {
    "direction": 1.0,
    "body": 0.5,
    "wick": 0.5,
    "range": 0.5,
    "pattern": 0.5,
    "path": 1.0,
    "volume_state": 0.25,
    "pattern_validates": 0.25,
    "horizon": 0.35,
}

V8_STRUCTURE_LOSS_WEIGHTS: Dict[str, float] = {
    "path": 1.0,
    "direction": 1.0,
    "structure": 0.5,
    "volume_state": 0.25,
}
V9_STRUCTURE_LOSS_WEIGHTS: Dict[str, float] = {
    "path": 1.0,
    "direction": 1.0,
    "structure": 0.5,
    "volume_state": 0.25,
    "pattern": 0.25,
}

V7_CONTINUOUS_LOSS_WEIGHTS: Dict[str, float] = {
    "mfe": 0.5,
    "mae": 0.5,
    "future_volatility": 1.0,
    "trend_strength": 0.5,
    "drawdown_before_mfe": 0.5,
    "future_oi_change_pct": 0.25,
    "future_volume_change_pct": 0.25,
    "candle_follow_through_atr": 0.5,
    "structure_delta": 0.5,
}

STRUCTURE_OUTCOME_NAMES: Dict[int, str] = {
    0: "RANGE",
    1: "CONTINUATION_LONG",
    2: "CONTINUATION_SHORT",
    3: "BREAKOUT",
    4: "FAILED_BREAK",
    5: "REVERSAL",
}

ONNX_OUTPUT_NAMES_V6: Tuple[str, ...] = (
    "continuous_pred",
    "regime_logits",
    "structure_outcome_logits",
    "future_candle_logits",
)
# Live 15m–2h bundles still export v6 heads. 5m research is v8.
ONNX_OUTPUT_NAMES: Tuple[str, ...] = ONNX_OUTPUT_NAMES_V6

ONNX_OUTPUT_NAMES_V7_BASE: Tuple[str, ...] = (
    "next_direction_logits",
    "next_body_logits",
    "next_wick_logits",
    "next_range_logits",
    "next_pattern_logits",
    "continuous_pred",
    "volume_state_logits",
    "pattern_validates_logit",
)
ONNX_OUTPUT_NAMES_V7_HORIZONS: Tuple[str, ...] = (
    "horizon_t3_dir_logits",
    "horizon_t6_dir_logits",
    "horizon_t12_dir_logits",
    "horizon_t24_dir_logits",
)
ONNX_OUTPUT_NAMES_V7: Tuple[str, ...] = (
    ONNX_OUTPUT_NAMES_V7_BASE + ONNX_OUTPUT_NAMES_V7_HORIZONS
)
ONNX_OUTPUT_NAMES_V8_DIR: Tuple[str, ...] = tuple(
    f"{key}_dir_logits" for key in HORIZON_KEYS
)
ONNX_OUTPUT_NAMES_V8_STRUCTURE: Tuple[str, ...] = tuple(
    f"{key}_structure_logits" for key in HORIZON_KEYS
)
ONNX_OUTPUT_NAMES_V8: Tuple[str, ...] = (
    ONNX_OUTPUT_NAMES_V8_DIR
    + ONNX_OUTPUT_NAMES_V8_STRUCTURE
    + ("continuous_pred", "volume_state_logits")
)
ONNX_OUTPUT_NAMES_V9: Tuple[str, ...] = ONNX_OUTPUT_NAMES_V8 + ("chart_pattern_logits",)

# v11 fused multi-TF model: 2-class BEAR/BULL heads, no MFE/MAE.
# 5m/10m stay encoder inputs. h10m is not a trading head. HOLD is not a class.
FUSION_HORIZON_SPECS: Tuple[Tuple[str, int], ...] = (
    ("h30m", 6),
    ("h1h", 12),
    ("h2h", 24),
)
FUSION_HORIZON_KEYS: Tuple[str, ...] = tuple(key for key, _ in FUSION_HORIZON_SPECS)
FUSION_HORIZON_BARS_5M: Tuple[int, ...] = tuple(bars for _, bars in FUSION_HORIZON_SPECS)
N_FUSION_HORIZONS: int = len(FUSION_HORIZON_SPECS)
MAX_FUSION_HORIZON_BARS: int = max(FUSION_HORIZON_BARS_5M)
FUSION_DIR_COLS: Tuple[str, ...] = tuple(f"{key}_dir" for key in FUSION_HORIZON_KEYS)
FUSION_RETIRED_DIR_COLS: Tuple[str, ...] = ("h10m_dir",)
FUSION_IGNORE_INDEX = -1
FUSION_DIRECTION_CARDINALITY = 2
FUSION_DIRECTION_NAMES: Dict[int, str] = {0: "BEAR", 1: "BULL"}
FUSION_POSITION_LONG = "LONG"
FUSION_POSITION_SHORT = "SHORT"
FUSION_POSITION_HOLD = "HOLD"
FUSION_GRADE_HIGH = "HIGH"
FUSION_GRADE_MEDIUM = "MEDIUM"
FUSION_GRADE_LOW = "LOW"
# 2-class chance is 0.50; floors sit above noise (old 0.40/0.45 would auto-pass).
FUSION_HIGH_BALANCED_ACC = 0.58
FUSION_HIGH_MAX_ECE = 0.08
FUSION_MEDIUM_BALANCED_ACC = 0.55
FUSION_MIN_PROBABILITY = 0.60
# Frozen v10 4-head / 3-class names — live fusion must not load this layout.
ONNX_OUTPUT_NAMES_V10: Tuple[str, ...] = (
    "h10m_dir_logits",
    "h30m_dir_logits",
    "h1h_dir_logits",
    "h2h_dir_logits",
    "tf_fusion_logits",
)
ONNX_OUTPUT_NAMES_V11: Tuple[str, ...] = tuple(
    f"{key}_dir_logits" for key in FUSION_HORIZON_KEYS
) + ("tf_fusion_logits",)
FUSION_DURATION_ATR_MULT: Dict[str, Tuple[float, float]] = {
    "h30m": (1.0, 1.5),
    "h1h": (1.5, 2.25),
    "h2h": (2.0, 3.0),
}
FUSION_HORIZON_MINUTES: Dict[str, int] = {
    "h30m": 30,
    "h1h": 60,
    "h2h": 120,
}

# Next-bar candle families for 5m timing (not model classes).
CANDLE_FAMILY_DOJI = frozenset({0, 1, 2, 3, 8})
CANDLE_FAMILY_BULL = frozenset({4, 6, 9, 11})
CANDLE_FAMILY_BEAR = frozenset({5, 7, 10, 12})

# Minimum test-set correlation for future_volatility before ONNX export.
MIN_EXPORT_VOL_CORR: Dict[str, float] = {
    "5m": 0.10,
    "15m": 0.10,
    "30m": 0.10,
    "1h": 0.08,
    "2h": 0.08,
}

PROMOTION_VOL_CORR: float = 0.15
PROMOTION_REGIME_ACCURACY: float = 0.35

EXPORT_QUALITY_DISCLAIMER = (
    "Sanity gates detect broken exports, not trading edge. "
    "Promotion tier targets are informational."
)

TRANSFORMER_METADATA_FILENAME = "metadata_transformer.json"
TRANSFORMER_FEATURE_CONFIG_FILENAME = "feature_config.json"

def candle_family_from_class(class_id: int) -> str:
    """Map a candle class id to bull / bear / doji for timing modifiers."""
    cid = int(class_id)
    if cid in CANDLE_FAMILY_BULL:
        return "bull"
    if cid in CANDLE_FAMILY_BEAR:
        return "bear"
    return "doji"

def compute_path_edge(mfe: float, mae: float) -> float:
    """Directional edge from predicted path asymmetry (MFE minus MAE).

    Alias of :func:`compute_long_edge` kept for train/serve telemetry compatibility.
    """
    return compute_long_edge(mfe, mae)

def compute_long_edge(mfe: float, mae: float) -> float:
    """Long-side path edge: upside (MFE) minus downside (MAE)."""
    return float(mfe) - float(mae)

def compute_short_edge(mfe: float, mae: float) -> float:
    """Short-side path edge: downside (MAE) minus upside (MFE)."""
    return float(mae) - float(mfe)

def path_favorable_adverse(
    mfe: float,
    mae: float,
    *,
    side: str,
) -> Tuple[float, float]:
    """Return (favorable_pct, adverse_pct) for bracket sizing.

    Labels are long-centric (MFE = upside, MAE = downside). For shorts, favorable
    excursion is downside (MAE) and adverse is upside (MFE).
    """
    s = str(side or "BUY").strip().upper()
    if s in ("SELL", "SHORT", "STRONG_SELL"):
        return float(mae), float(mfe)
    return float(mfe), float(mae)

def model_family_for_resolution(resolution: str) -> str:
    """Canonical model_family string for a TF bundle."""
    res = resolution.strip().lower()
    if res == "mtf_fusion":
        return FUSION_MODEL_FAMILY
    if res not in RESOLUTION_MINUTES:
        raise ValueError(f"Unsupported resolution: {resolution!r}")
    return f"jacksparrow_transformer_btcusd_{res}"

def onnx_filename_for_resolution(resolution: str) -> str:
    res = resolution.strip().lower()
    if res == "mtf_fusion":
        return FUSION_ONNX_FILENAME
    if res not in RESOLUTION_MINUTES:
        raise ValueError(f"Unsupported resolution: {resolution!r}")
    return f"btcusd_{res}_transformer.onnx"

def bundle_dir_name(resolution: str) -> str:
    res = resolution.strip().lower()
    if res == "mtf_fusion":
        return FUSION_BUNDLE_DIR_NAME
    return f"JackSparrow_Transformer_BTCUSD_{res}"

def horizon_bars_for_wall_minutes(wall_minutes: int, resolution_minutes: int) -> int:
    """Convert wall-clock minutes to native-TF bar count."""
    return max(1, int(round(int(wall_minutes) / resolution_minutes)))

def label_horizon_bars_for_resolution(resolution_minutes: int) -> int:
    """Legacy helper: 8h wall-clock in bars for a TF grid."""
    return horizon_bars_for_wall_minutes(REFERENCE_LABEL_HORIZON_MINUTES, resolution_minutes)

def path_label_horizon_bars_for_resolution(resolution: str) -> int:
    """Path-label forward window in bars for a TF."""
    res = resolution.strip().lower()
    if res not in RESOLUTION_MINUTES:
        raise ValueError(f"Unsupported resolution: {resolution!r}")
    minutes = RESOLUTION_MINUTES[res]
    wall = DEFAULT_PATH_LABEL_HORIZON_MINUTES.get(res, REFERENCE_LABEL_HORIZON_MINUTES)
    return horizon_bars_for_wall_minutes(wall, minutes)

def continuous_loss_weights_for_resolution(resolution: str) -> Tuple[float, ...]:
    """Per-head loss weights aligned with CONTINUOUS_LABEL_COLS (0 = no gradient)."""
    weights = dict(DEFAULT_CONTINUOUS_LOSS_WEIGHTS)
    return tuple(float(weights.get(col, 1.0)) for col in CONTINUOUS_LABEL_COLS)

def default_training_config(resolution: str) -> Dict[str, Any]:
    """Default Colab training config for a single TF model."""
    res = resolution.strip().lower()
    if res not in RESOLUTION_MINUTES:
        raise ValueError(f"Unsupported resolution: {resolution!r}")
    minutes = RESOLUTION_MINUTES[res]
    path_horizon = path_label_horizon_bars_for_resolution(res)
    return {
        "symbol": "BTCUSD",
        "resolution": res,
        "resolution_minutes": minutes,
        "history_days": 900,
        "base_url": "https://api.india.delta.exchange",
        "atr_period": 14,
        "path_label_horizon_bars": path_horizon,
        "label_horizon_minutes_path": DEFAULT_PATH_LABEL_HORIZON_MINUTES.get(
            res, REFERENCE_LABEL_HORIZON_MINUTES
        ),
        "continuous_loss_weights": list(continuous_loss_weights_for_resolution(res)),
        "mae_floor_atr_mult": 0.25,
        "vol_regime_quantiles": [0.25, 0.5, 0.75],
        "window_len": 128,
        "stride": 8,
        "train_frac": 0.65,
        "val_frac": 0.15,
        "embargo_bars": path_horizon,
        "batch_size": 128,
        "epochs": 120,
        "lr": 1e-4,
        "d_model": 64,
        "nhead": 4,
        "num_layers": 2,
        "dropout": 0.25,
        "weight_decay": 1e-2,
        "early_stop_patience": 12,
        "early_stopping_enabled": True,
        "min_derivatives_coverage": 0.5,
        "derivatives_coverage_warn": 0.9,
        "min_export_vol_corr": MIN_EXPORT_VOL_CORR.get(res, 0.08),
        "default_threshold": 0.005,
        "seed": 42,
    }

def max_label_horizon_bars(
    path_label_horizon_bars: int = PATH_LABEL_HORIZON_BARS,
) -> int:
    """Maximum forward bars across all training labels."""
    return int(path_label_horizon_bars)

def scale_period(period: int, resolution_minutes: int, *, base_minutes: int = 5) -> int:
    """Scale indicator lookback to preserve wall-clock semantics across TFs."""
    return max(1, int(round(period * resolution_minutes / base_minutes)))

def feature_cols_for_resolution(resolution: str) -> Tuple[str, ...]:
    """Continuous feature columns for a TF bundle (5m appends closed HTF context)."""
    res = resolution.strip().lower()
    if res not in RESOLUTION_MINUTES:
        raise ValueError(f"Unsupported resolution: {resolution!r}")
    if res == "5m":
        return FEATURE_COLS + HTF_STRUCTURE_COLS
    return FEATURE_COLS

def fusion_native_feature_cols() -> Tuple[str, ...]:
    """Native-TF columns for the fused model (no resampled HTF context)."""
    return FEATURE_COLS

def v7_feature_cols_for_resolution(resolution: str) -> Tuple[str, ...]:
    """Input columns: native features plus causal chart_pattern_id."""
    return feature_cols_for_resolution(resolution) + (CHART_PATTERN_COL,)

def v8_feature_cols_for_resolution(resolution: str) -> Tuple[str, ...]:
    """v8 5m input columns (native features plus causal chart_pattern_id)."""
    return v7_feature_cols_for_resolution(resolution)

def v9_feature_cols_for_resolution(resolution: str) -> Tuple[str, ...]:
    """v9 5m input columns: geometry only; chart_pattern_id is a target head."""
    return feature_cols_for_resolution(resolution)

def onnx_output_names_for_contract(
    contract_version: str,
    *,
    resolution: str = "5m",
) -> Tuple[str, ...]:
    """ONNX head names for a bundle contract."""
    ver = str(contract_version or "").strip()
    res = resolution.strip().lower()
    if ver == FEATURE_CONTRACT_VERSION_V11 or res == "mtf_fusion":
        return ONNX_OUTPUT_NAMES_V11
    if ver == FEATURE_CONTRACT_VERSION_V10:
        return ONNX_OUTPUT_NAMES_V10
    if ver in (FEATURE_CONTRACT_VERSION, FEATURE_CONTRACT_VERSION_V9):
        if res == "5m":
            return ONNX_OUTPUT_NAMES_V9
        return ONNX_OUTPUT_NAMES_V6
    if ver == FEATURE_CONTRACT_VERSION_V8:
        if res == "5m":
            return ONNX_OUTPUT_NAMES_V8
        return ONNX_OUTPUT_NAMES_V6
    if ver == FEATURE_CONTRACT_VERSION_V7:
        if res == "5m":
            return ONNX_OUTPUT_NAMES_V7
        return ONNX_OUTPUT_NAMES_V7_BASE
    return ONNX_OUTPUT_NAMES_V6

def v8_future_leak_cols() -> frozenset:
    """Label/target columns that must never appear in 5m v8 model inputs."""
    leaked = set(V8_CONTINUOUS_LABEL_COLS)
    leaked.update(HORIZON_DIR_COLS)
    leaked.update(HORIZON_STRUCTURE_COLS)
    leaked.update(
        {
            VOLUME_STATE_COL,
            NEXT_DIRECTION_COL,
            NEXT_BODY_COL,
            NEXT_WICK_COL,
            NEXT_RANGE_COL,
            FUTURE_CANDLE_COL,
            "mfe",
            "mae",
            "future_volatility",
            "pattern_validates",
            *HORIZON_DIR_COLS_V7,
        }
    )
    return frozenset(leaked)

def expected_direction_from_chart_pattern(pattern_id: int) -> int:
    """Map chart pattern to expected next-candle direction (0/1/2). Neutral if none."""
    pid = int(pattern_id)
    if pid in (1, 5, 7):  # FLAG_BULL, DOUBLE_BOTTOM, BREAKOUT (unsigned handled elsewhere)
        return 2
    if pid in (2, 4, 8):  # FLAG_BEAR, DOUBLE_TOP, FAILED_BREAK
        return 0
    return 1

def default_research_config() -> Dict[str, Any]:
    """Colab research-pipeline defaults (5m multi-horizon path)."""
    return {
        "symbol": "BTCUSD",
        "resolution": "5m",
        "resolution_minutes": 5,
        "base_timeframe": "5m",
        "context_timeframes": ["15m", "30m", "1h", "2h"],
        "sequence_length": 64,
        "prediction_horizon": 1,
        "horizon_bars": list(HORIZON_BARS_5M),
        "train_ratio": 0.70,
        "validation_ratio": 0.15,
        "test_ratio": 0.15,
        "batch_size": 256,
        "epochs": 50,
        "learning_rate": 1e-4,
        "weight_decay": 1e-4,
        "dropout": 0.15,
        "early_stopping_patience": 8,
        "seed": 42,
        "d_model": 64,
        "nhead": 4,
        "num_layers": 2,
        "stride": 4,
        "scaler_mode": "train_fit",
        "gate_chart_volume": False,
        "path_label_horizon_bars": MAX_V8_HORIZON_BARS,
        "embargo_bars": MAX_V8_HORIZON_BARS,
        "mae_floor_atr_mult": 0.25,
        "atr_period": 14,
        "history_days": 900,
        "base_url": "https://api.india.delta.exchange",
        "loss_weights": dict(V9_STRUCTURE_LOSS_WEIGHTS),
        "run_optuna": False,
        "optuna_trials": 0,
        "run_shap": False,
        "run_ablations": False,
        "run_walk_forward": False,
        "walk_forward_folds": 3,
    }

def default_fusion_training_config() -> Dict[str, Any]:
    """Defaults for the single multi-TF fusion trainer."""
    return {
        "symbol": "BTCUSD",
        "resolutions": list(FUSION_INPUT_RESOLUTIONS),
        "horizon_keys": list(FUSION_HORIZON_KEYS),
        "horizon_bars": list(FUSION_HORIZON_BARS_5M),
        "window_len": FUSION_WINDOW_LEN,
        "stride": 4,
        "train_frac": 0.70,
        "val_frac": 0.15,
        "embargo_bars": FUSION_EMBARGO_BARS,
        "batch_size": 64,
        "epochs": 40,
        "lr": 1e-4,
        "weight_decay": 1e-3,
        "dropout": 0.30,
        "d_model": 64,
        "nhead": 4,
        "num_layers": 2,
        "early_stop_patience": 5,
        "label_smoothing": 0.05,
        "horizon_loss_weights": [1.0, 0.8, 0.4],
        "lr_schedule": "cosine",
        "seed": 42,
        "history_days": 900,
        "base_url": "https://api.india.delta.exchange",
        "atr_period": 14,
        "run_optuna": False,
        "optuna_trials": 8,
        "optuna_trial_epochs": 12,
        "run_walk_forward": True,
        "walk_forward_folds": 3,
        "walk_forward_embargo": FUSION_EMBARGO_BARS,
        "min_probability": FUSION_MIN_PROBABILITY,
        "high_balanced_acc": FUSION_HIGH_BALANCED_ACC,
        "high_max_ece": FUSION_HIGH_MAX_ECE,
        "medium_balanced_acc": FUSION_MEDIUM_BALANCED_ACC,
        "n_classes": FUSION_DIRECTION_CARDINALITY,
        "run_shap": False,
        "shap_background": 32,
        "shap_explain_n": 64,
    }

def ablation_feature_groups(resolution: str = "5m") -> Dict[str, Tuple[str, ...]]:
    """Nested feature sets A-F for out-of-sample ablation."""
    all_cols = v9_feature_cols_for_resolution(resolution)
    ohlcv = ("ret_1", "rv_16", "rv_96", "hour_sin", "hour_cos", "dow_sin", "dow_cos")
    geometry = ohlcv + (
        "body_ratio",
        "upper_wick_ratio",
        "lower_wick_ratio",
        "close_loc",
        "range_atr",
        "body_atr",
        "gap_atr",
        "inside_bar",
        "outside_bar",
        "engulf_score",
    )
    trend = geometry + (
        "ema50_dist_pct",
        "macd_hist",
        "rsi_14",
        "adx_14",
        "price_vs_ema9_atr",
        "price_vs_ema21_atr",
        "price_vs_ema50_atr",
        "price_vs_ema200_atr",
        "ema9_vs_21_atr",
        "ema21_vs_50_atr",
        "ema50_vs_200_atr",
        "ema21_slope_atr",
        "ema50_slope_atr",
        "trend_efficiency",
        "displacement_atr",
        "pct_with_trend",
    )
    structure = trend + (
        "hh_count",
        "hl_count",
        "lh_count",
        "ll_count",
        "structure_bias",
        "last_swing_dir",
        "bars_since_swing",
        "swing_amp_atr",
        "dist_to_support_atr",
        "dist_to_resistance_atr",
        "support_touch_count",
        "resistance_touch_count",
        "range_width_atr",
        "dist_to_resistance_pct",
        "dist_to_support_pct",
    )
    chart = structure + (
        "peak_diff_atr",
        "trough_diff_atr",
        "peak_sep_bars",
        "trough_sep_bars",
        "dist_neck_atr",
        "high_slope_atr",
        "low_slope_atr",
        "convergence",
        "width_now_atr",
        "pole_disp_atr",
        "flag_width_atr",
        "flag_slope_atr",
        "breakout_size_atr",
        "breakout_vol_ratio",
        "pre_breakout_comp",
        "bars_since_breakout",
        "retest_dist_atr",
        "failed_break",
    )
    return {
        "A": tuple(c for c in ohlcv if c in all_cols),
        "B": tuple(c for c in geometry if c in all_cols),
        "C": tuple(c for c in trend if c in all_cols),
        "D": tuple(c for c in structure if c in all_cols),
        "E": tuple(c for c in chart if c in all_cols),
        "F": all_cols,
    }


## Derivatives features

In [ ]:
"""Funding and OI derivative features scaled per resolution."""

from __future__ import annotations

import numpy as np
import pandas as pd

_EPS = 1e-9

def compute_funding_derivatives(
    fund_rate: pd.Series,
    ret_2: pd.Series,
    *,
    resolution_minutes: int,
) -> pd.DataFrame:
    """Funding z-score, momentum, and rate-of-change on native TF bars."""
    z_window = scale_period(16, resolution_minutes)
    roc_diff = max(1, scale_period(1, resolution_minutes))
    fz_mean = fund_rate.rolling(z_window, min_periods=5).mean()
    fz_std = fund_rate.rolling(z_window, min_periods=5).std().replace(0, _EPS)
    funding_zscore = ((fund_rate - fz_mean) / fz_std).fillna(0.0).clip(-4.0, 4.0)
    funding_mom = funding_zscore * ret_2
    diff = fund_rate.diff(roc_diff)
    roc_mu = diff.rolling(z_window, min_periods=5).mean()
    roc_std = diff.rolling(z_window, min_periods=5).std().replace(0, _EPS)
    funding_rate_roc = ((diff - roc_mu) / roc_std).fillna(0.0).clip(-4.0, 4.0)
    return pd.DataFrame(
        {
            "funding_zscore": funding_zscore,
            "funding_mom": funding_mom.fillna(0.0),
            "funding_rate_roc": funding_rate_roc,
        }
    )

def compute_oi_derivatives(
    primary: pd.DataFrame,
    *,
    resolution_minutes: int,
) -> pd.DataFrame:
    """OI-derived features on native TF bars."""
    n = len(primary)
    zero = pd.DataFrame(
        {
            "oi_change_2": np.zeros(n),
            "oi_delta_z": np.zeros(n),
            "oi_price_divergence": np.zeros(n),
            "oi_acceleration": np.zeros(n),
            "oi_zscore": np.zeros(n),
        },
        index=primary.index,
    )
    if "open_interest" not in primary.columns:
        return zero

    z_window = scale_period(16, resolution_minutes)
    change_window = max(1, scale_period(2, resolution_minutes))

    oi_s = primary["open_interest"].astype(float)
    if oi_s.notna().sum() == 0 or float(oi_s.max()) < _EPS:
        return zero

    oi_mu = oi_s.rolling(z_window, min_periods=max(2, z_window // 4)).mean()
    oi_std = oi_s.rolling(z_window, min_periods=max(2, z_window // 4)).std().clip(
        lower=_EPS
    )
    oi_zscore = ((oi_s - oi_mu) / oi_std).fillna(0.0).clip(-4.0, 4.0)

    oi_lagged = oi_s.shift(change_window)
    oi_change_2 = (
        ((oi_s - oi_lagged) / (oi_lagged.abs() + _EPS)).fillna(0.0).clip(-0.05, 0.05)
    )

    close_s = primary["close"].astype(float)
    close_lagged = close_s.shift(change_window)
    ret_2 = ((close_s - close_lagged) / (close_lagged.abs() + _EPS)).fillna(0.0)
    oi_price_divergence = (
        np.sign(oi_change_2.values) * -np.sign(ret_2.values)
    ).astype(np.float32)
    oi_acceleration = oi_change_2.diff().fillna(0.0).clip(-0.02, 0.02)

    oi_delta_1 = oi_s.diff(1).fillna(0.0)
    oi_delta_mu = oi_delta_1.rolling(z_window, min_periods=max(2, z_window // 4)).mean()
    oi_delta_std = oi_delta_1.rolling(z_window, min_periods=max(2, z_window // 4)).std().clip(
        lower=_EPS
    )
    oi_delta_z = ((oi_delta_1 - oi_delta_mu) / oi_delta_std).fillna(0.0).clip(-4.0, 4.0)

    return pd.DataFrame(
        {
            "oi_zscore": oi_zscore,
            "oi_change_2": oi_change_2,
            "oi_price_divergence": pd.Series(oi_price_divergence).fillna(0.0),
            "oi_acceleration": oi_acceleration,
            "oi_delta_z": oi_delta_z,
        },
        index=primary.index,
    ).replace([np.inf, -np.inf], 0.0).fillna(0.0)


## Market structure

In [ ]:
"""Causal OHLCV market-structure and chart-geometry features.

All values at bar t use information available at or before t. Swings are ATR
ZigZag confirmations (not future-looking fractals). Donchian/breakout ranges
exclude bar t. Higher-TF context is merged from fully closed resampled bars.
"""

from __future__ import annotations

from collections import deque
from typing import Deque, List, Tuple

import numpy as np
import pandas as pd

_EPS = 1e-9
_ATR_CLIP = 8.0
_SLOPE_CLIP = 2.0
_ZZ_TAU = 1.5
_ZZ_EVENT_M = 6
_ZZ_EPS_ATR = 0.15
_SWING_K = 20
_ER_N = 32
_SLOPE_LAG = 8
_DONCHIAN_N = 32
_RANGE_PCTILE_W = 256
_TOUCH_W = 64
_SR_DELTA_ATR = 0.5
_POLE_BARS = 12
_FLAG_BARS = 16
_BREAKOUT_THRESH = 0.25
_FAILED_BREAK_BARS = 8
_CHANNEL_K = 3
_ATR_SHORT = 14
_ATR_LONG = 56
_HTF_NATIVE_MINUTES = 5

_HTF_NATIVE_MAP = {
    "structure_bias": "structure_bias",
    "trend_efficiency": "trend_efficiency",
    "ema21_slope_atr": "ema21_slope_atr",
    "dist_support_atr": "dist_to_support_atr",
    "dist_resistance_atr": "dist_to_resistance_atr",
    "range_width_atr": "range_width_atr",
}

def _structure_safe_atr(atr: np.ndarray, close: np.ndarray, i: int) -> float:
    val = float(atr[i]) if np.isfinite(atr[i]) else float("nan")
    if not np.isfinite(val) or val <= 0:
        c = float(close[i]) if np.isfinite(close[i]) else 0.0
        return max(abs(c) * 0.01, _EPS)
    return val

def _clip_atr(val: float) -> float:
    if not np.isfinite(val):
        return 0.0
    return float(np.clip(val, -_ATR_CLIP, _ATR_CLIP))

def _ols_slope(x: np.ndarray, y: np.ndarray) -> float:
    n = float(len(x))
    if n < 2:
        return 0.0
    sx = float(x.sum())
    sy = float(y.sum())
    sxy = float((x * y).sum())
    sx2 = float((x * x).sum())
    den = n * sx2 - sx * sx
    if abs(den) < 1e-12:
        return 0.0
    return float((n * sxy - sx * sy) / den)

def _ensure_atr(df: pd.DataFrame, period: int = 14) -> pd.DataFrame:
    out = df.copy()
    if "atr" in out.columns and out["atr"].notna().any():
        return out
    prev_close = out["close"].shift(1)
    tr = pd.concat(
        [
            out["high"] - out["low"],
            (out["high"] - prev_close).abs(),
            (out["low"] - prev_close).abs(),
        ],
        axis=1,
    ).max(axis=1)
    out["atr"] = tr.rolling(period, min_periods=1).mean()
    return out

def _vectorized_trend_and_ma(out: pd.DataFrame) -> pd.DataFrame:
    """ER, displacement, MA distances/slopes, Donchian compression, flag geometry."""
    close = out["close"].astype(float)
    high = out["high"].astype(float)
    low = out["low"].astype(float)
    volume = out["volume"].astype(float) if "volume" in out.columns else pd.Series(
        0.0, index=out.index
    )
    atr = out["atr"].astype(float) if "atr" in out.columns else pd.Series(
        close.abs() * 0.01, index=out.index
    )
    atr_safe = atr.replace(0, np.nan).fillna(close.abs() * 0.01 + _EPS)

    abs_move = (close - close.shift(_ER_N)).abs()
    path = close.diff().abs().rolling(_ER_N, min_periods=2).sum()
    out["trend_efficiency"] = (abs_move / (path + _EPS)).clip(0.0, 1.0).fillna(0.0)
    out["displacement_atr"] = (abs_move / (atr_safe + _EPS)).clip(
        0.0, _ATR_CLIP
    ).fillna(0.0)

    window_dir = np.sign(close - close.shift(_ER_N))
    up_frac = (close.diff() > 0).astype(float).rolling(_ER_N, min_periods=1).mean()
    pct = np.where(window_dir >= 0, up_frac, 1.0 - up_frac)
    out["pct_with_trend"] = pd.Series(pct, index=out.index).fillna(0.0)

    ema9 = close.ewm(span=9, adjust=False).mean()
    ema21 = close.ewm(span=21, adjust=False).mean()
    ema50 = close.ewm(span=50, adjust=False).mean()
    ema200 = close.ewm(span=200, adjust=False).mean()
    out["price_vs_ema9_atr"] = ((close - ema9) / (atr_safe + _EPS)).clip(
        -_ATR_CLIP, _ATR_CLIP
    )
    out["price_vs_ema21_atr"] = ((close - ema21) / (atr_safe + _EPS)).clip(
        -_ATR_CLIP, _ATR_CLIP
    )
    out["price_vs_ema50_atr"] = ((close - ema50) / (atr_safe + _EPS)).clip(
        -_ATR_CLIP, _ATR_CLIP
    )
    out["price_vs_ema200_atr"] = ((close - ema200) / (atr_safe + _EPS)).clip(
        -_ATR_CLIP, _ATR_CLIP
    )
    out["ema9_vs_21_atr"] = ((ema9 - ema21) / (atr_safe + _EPS)).clip(
        -_ATR_CLIP, _ATR_CLIP
    )
    out["ema21_vs_50_atr"] = ((ema21 - ema50) / (atr_safe + _EPS)).clip(
        -_ATR_CLIP, _ATR_CLIP
    )
    out["ema50_vs_200_atr"] = ((ema50 - ema200) / (atr_safe + _EPS)).clip(
        -_ATR_CLIP, _ATR_CLIP
    )
    lag = float(_SLOPE_LAG)
    out["ema21_slope_atr"] = (
        (ema21 - ema21.shift(_SLOPE_LAG)) / (atr_safe * lag + _EPS)
    ).clip(-_SLOPE_CLIP, _SLOPE_CLIP).fillna(0.0)
    out["ema50_slope_atr"] = (
        (ema50 - ema50.shift(_SLOPE_LAG)) / (atr_safe * lag + _EPS)
    ).clip(-_SLOPE_CLIP, _SLOPE_CLIP).fillna(0.0)

    prior_high = high.shift(1)
    prior_low = low.shift(1)
    donch_high = prior_high.rolling(_DONCHIAN_N, min_periods=1).max()
    donch_low = prior_low.rolling(_DONCHIAN_N, min_periods=1).min()
    range_width_atr = ((donch_high - donch_low) / (atr_safe + _EPS)).clip(
        0.0, _ATR_CLIP
    )
    out["range_width_atr"] = range_width_atr.fillna(0.0)
    out["range_width_pctile"] = range_width_atr.rolling(
        _RANGE_PCTILE_W, min_periods=8
    ).rank(pct=True).fillna(0.0)
    atr_short = atr_safe.rolling(_ATR_SHORT, min_periods=1).mean()
    atr_long = atr_safe.rolling(_ATR_LONG, min_periods=1).mean()
    out["atr_contraction"] = (atr_short / (atr_long + _EPS)).clip(0.0, _ATR_CLIP).fillna(
        1.0
    )

    out["breakout_size_atr"] = ((close - donch_high) / (atr_safe + _EPS)).clip(
        -_ATR_CLIP, _ATR_CLIP
    ).fillna(0.0)
    vol_mean = volume.shift(1).rolling(_DONCHIAN_N, min_periods=1).mean()
    out["breakout_vol_ratio"] = (volume / (vol_mean + _EPS)).clip(0.0, _ATR_CLIP).fillna(
        0.0
    )
    out["pre_breakout_comp"] = out["range_width_pctile"].shift(1).fillna(0.0)

    pole_end = close.shift(_FLAG_BARS)
    pole_start = close.shift(_POLE_BARS + _FLAG_BARS)
    out["pole_disp_atr"] = ((pole_end - pole_start) / (atr_safe + _EPS)).clip(
        -_ATR_CLIP, _ATR_CLIP
    ).fillna(0.0)
    flag_high = high.rolling(_FLAG_BARS, min_periods=1).max()
    flag_low = low.rolling(_FLAG_BARS, min_periods=1).min()
    out["flag_width_atr"] = ((flag_high - flag_low) / (atr_safe + _EPS)).clip(
        0.0, _ATR_CLIP
    ).fillna(0.0)
    flag_lag = float(_FLAG_BARS)
    out["flag_slope_atr"] = (
        (close - close.shift(_FLAG_BARS)) / (atr_safe * flag_lag + _EPS)
    ).clip(-_SLOPE_CLIP, _SLOPE_CLIP).fillna(0.0)
    return out

def _zigzag_structure_loop(out: pd.DataFrame) -> pd.DataFrame:
    """Single-pass ATR ZigZag: HH/HL counts, S/R, breakout events, pattern geometry."""
    n = len(out)
    high = out["high"].to_numpy(dtype=np.float64)
    low = out["low"].to_numpy(dtype=np.float64)
    close = out["close"].to_numpy(dtype=np.float64)
    atr = out["atr"].to_numpy(dtype=np.float64) if "atr" in out.columns else np.full(
        n, np.nan
    )
    prior_high = np.roll(high, 1)
    prior_high[0] = high[0]
    prior_low = np.roll(low, 1)
    prior_low[0] = low[0]
    donch_h = np.empty(n, dtype=np.float64)
    donch_l = np.empty(n, dtype=np.float64)
    max_q: Deque[int] = deque()
    min_q: Deque[int] = deque()
    for t in range(n):
        left = t - _DONCHIAN_N
        while max_q and max_q[0] <= left:
            max_q.popleft()
        while min_q and min_q[0] <= left:
            min_q.popleft()
        src_h = prior_high[t]
        src_l = prior_low[t]
        while max_q and prior_high[max_q[-1]] <= src_h:
            max_q.pop()
        while min_q and prior_low[min_q[-1]] >= src_l:
            min_q.pop()
        max_q.append(t)
        min_q.append(t)
        donch_h[t] = prior_high[max_q[0]]
        donch_l[t] = prior_low[min_q[0]]

    hh_count = np.zeros(n, dtype=np.float64)
    hl_count = np.zeros(n, dtype=np.float64)
    lh_count = np.zeros(n, dtype=np.float64)
    ll_count = np.zeros(n, dtype=np.float64)
    structure_bias = np.zeros(n, dtype=np.float64)
    last_swing_dir = np.zeros(n, dtype=np.float64)
    bars_since_swing = np.zeros(n, dtype=np.float64)
    swing_amp_atr = np.zeros(n, dtype=np.float64)
    unconfirmed_ext_atr = np.zeros(n, dtype=np.float64)
    dist_to_support_atr = np.zeros(n, dtype=np.float64)
    dist_to_resistance_atr = np.zeros(n, dtype=np.float64)
    support_touch = np.zeros(n, dtype=np.float64)
    resistance_touch = np.zeros(n, dtype=np.float64)
    bars_since_breakout = np.zeros(n, dtype=np.float64)
    retest_dist_atr = np.zeros(n, dtype=np.float64)
    failed_break = np.zeros(n, dtype=np.float64)
    peak_diff_atr = np.zeros(n, dtype=np.float64)
    trough_diff_atr = np.zeros(n, dtype=np.float64)
    peak_sep_bars = np.zeros(n, dtype=np.float64)
    trough_sep_bars = np.zeros(n, dtype=np.float64)
    dist_neck_atr = np.zeros(n, dtype=np.float64)
    high_slope_atr = np.zeros(n, dtype=np.float64)
    low_slope_atr = np.zeros(n, dtype=np.float64)
    convergence = np.zeros(n, dtype=np.float64)
    width_now_atr = np.zeros(n, dtype=np.float64)

    direction = 1
    e_idx = 0
    e_price = high[0] if n else 0.0
    last_high_price = float("nan")
    last_low_price = float("nan")
    last_high_idx = -1
    last_low_idx = -1
    last_swing_idx = -1
    last_dir = 0.0
    highs: List[Tuple[int, float]] = []
    lows: List[Tuple[int, float]] = []
    events: Deque[str] = deque(maxlen=_ZZ_EVENT_M)
    last_bo_idx = -1
    last_bo_level = float("nan")
    last_bo_side = 0

    for t in range(n):
        atr_t = _structure_safe_atr(atr, close, t)
        thresh = _ZZ_TAU * atr_t
        eps = _ZZ_EPS_ATR * atr_t

        if direction == 1:
            if high[t] >= e_price:
                e_price = high[t]
                e_idx = t
            elif (e_price - low[t]) >= thresh:
                highs.append((e_idx, e_price))
                if len(highs) > _SWING_K:
                    highs = highs[-_SWING_K:]
                if np.isfinite(last_high_price):
                    if e_price > last_high_price + eps:
                        events.append("HH")
                    elif e_price < last_high_price - eps:
                        events.append("LH")
                last_high_price = e_price
                last_high_idx = e_idx
                last_swing_idx = e_idx
                last_dir = 1.0
                direction = -1
                e_price = low[t]
                e_idx = t
        else:
            if low[t] <= e_price:
                e_price = low[t]
                e_idx = t
            elif (high[t] - e_price) >= thresh:
                lows.append((e_idx, e_price))
                if len(lows) > _SWING_K:
                    lows = lows[-_SWING_K:]
                if np.isfinite(last_low_price):
                    if e_price > last_low_price + eps:
                        events.append("HL")
                    elif e_price < last_low_price - eps:
                        events.append("LL")
                last_low_price = e_price
                last_low_idx = e_idx
                last_swing_idx = e_idx
                last_dir = -1.0
                direction = 1
                e_price = high[t]
                e_idx = t

        n_hh = sum(1 for e in events if e == "HH")
        n_hl = sum(1 for e in events if e == "HL")
        n_lh = sum(1 for e in events if e == "LH")
        n_ll = sum(1 for e in events if e == "LL")
        n_ev = max(len(events), 1)
        hh_count[t] = float(n_hh)
        hl_count[t] = float(n_hl)
        lh_count[t] = float(n_lh)
        ll_count[t] = float(n_ll)
        structure_bias[t] = (n_hh + n_hl - n_lh - n_ll) / float(n_ev)
        last_swing_dir[t] = last_dir
        bars_since_swing[t] = float(t - last_swing_idx) if last_swing_idx >= 0 else float(t)
        if np.isfinite(last_high_price) and np.isfinite(last_low_price):
            swing_amp_atr[t] = _clip_atr(abs(last_high_price - last_low_price) / atr_t)
        unconfirmed_ext_atr[t] = _clip_atr((e_price - close[t]) / atr_t)

        swing_lows = [p for _, p in lows if p <= close[t]]
        swing_highs = [p for _, p in highs if p >= close[t]]
        support = max(swing_lows) if swing_lows else float(donch_l[t])
        resistance = min(swing_highs) if swing_highs else float(donch_h[t])
        dist_to_support_atr[t] = _clip_atr((close[t] - support) / atr_t)
        dist_to_resistance_atr[t] = _clip_atr((resistance - close[t]) / atr_t)
        delta = _SR_DELTA_ATR * atr_t
        if low[t] <= support + delta and close[t] >= support - delta:
            support_touch[t] = 1.0
        if high[t] >= resistance - delta and close[t] <= resistance + delta:
            resistance_touch[t] = 1.0

        up_ext = (close[t] - donch_h[t]) / atr_t
        dn_ext = (donch_l[t] - close[t]) / atr_t
        if up_ext >= _BREAKOUT_THRESH:
            last_bo_idx = t
            last_bo_level = float(donch_h[t])
            last_bo_side = 1
        elif dn_ext >= _BREAKOUT_THRESH:
            last_bo_idx = t
            last_bo_level = float(donch_l[t])
            last_bo_side = -1
        if last_bo_idx >= 0:
            bars_since_breakout[t] = float(t - last_bo_idx)
            retest_dist_atr[t] = _clip_atr((close[t] - last_bo_level) / atr_t)
            if last_bo_side > 0 and close[t] < last_bo_level:
                failed_break[t] = 1.0
            elif last_bo_side < 0 and close[t] > last_bo_level:
                failed_break[t] = 1.0
        else:
            bars_since_breakout[t] = float(t)

        if len(highs) >= 2:
            (i1, p1), (i2, p2) = highs[-2], highs[-1]
            peak_diff_atr[t] = _clip_atr((p2 - p1) / atr_t)
            peak_sep_bars[t] = float(i2 - i1)
            between = [lp for li, lp in lows if i1 < li < i2]
            if between:
                neck = min(between)
                dist_neck_atr[t] = _clip_atr((close[t] - neck) / atr_t)
        if len(lows) >= 2:
            (j1, q1), (j2, q2) = lows[-2], lows[-1]
            trough_diff_atr[t] = _clip_atr((q2 - q1) / atr_t)
            trough_sep_bars[t] = float(j2 - j1)

        if len(highs) >= _CHANNEL_K:
            hx = np.array([idx for idx, _ in highs[-_CHANNEL_K:]], dtype=np.float64)
            hy = np.array([px for _, px in highs[-_CHANNEL_K:]], dtype=np.float64)
            hs = _ols_slope(hx, hy)
            high_slope_atr[t] = float(np.clip(hs / atr_t, -_SLOPE_CLIP, _SLOPE_CLIP))
        if len(lows) >= _CHANNEL_K:
            lx = np.array([idx for idx, _ in lows[-_CHANNEL_K:]], dtype=np.float64)
            ly = np.array([px for _, px in lows[-_CHANNEL_K:]], dtype=np.float64)
            ls = _ols_slope(lx, ly)
            low_slope_atr[t] = float(np.clip(ls / atr_t, -_SLOPE_CLIP, _SLOPE_CLIP))
        convergence[t] = float(
            np.clip(high_slope_atr[t] - low_slope_atr[t], -_SLOPE_CLIP, _SLOPE_CLIP)
        )
        if len(highs) >= 1 and len(lows) >= 1:
            h_line = highs[-1][1] + high_slope_atr[t] * atr_t * (t - highs[-1][0])
            l_line = lows[-1][1] + low_slope_atr[t] * atr_t * (t - lows[-1][0])
            width_now_atr[t] = _clip_atr((h_line - l_line) / atr_t)

    touch_sup = pd.Series(support_touch).rolling(_TOUCH_W, min_periods=1).sum()
    touch_res = pd.Series(resistance_touch).rolling(_TOUCH_W, min_periods=1).sum()

    out["hh_count"] = hh_count
    out["hl_count"] = hl_count
    out["lh_count"] = lh_count
    out["ll_count"] = ll_count
    out["structure_bias"] = structure_bias
    out["last_swing_dir"] = last_swing_dir
    out["bars_since_swing"] = bars_since_swing
    out["swing_amp_atr"] = swing_amp_atr
    out["unconfirmed_ext_atr"] = unconfirmed_ext_atr
    out["dist_to_support_atr"] = dist_to_support_atr
    out["dist_to_resistance_atr"] = dist_to_resistance_atr
    out["support_touch_count"] = touch_sup.to_numpy(dtype=np.float64)
    out["resistance_touch_count"] = touch_res.to_numpy(dtype=np.float64)
    out["bars_since_breakout"] = bars_since_breakout
    out["retest_dist_atr"] = retest_dist_atr
    out["failed_break"] = failed_break
    out["peak_diff_atr"] = peak_diff_atr
    out["trough_diff_atr"] = trough_diff_atr
    out["peak_sep_bars"] = peak_sep_bars
    out["trough_sep_bars"] = trough_sep_bars
    out["dist_neck_atr"] = dist_neck_atr
    out["high_slope_atr"] = high_slope_atr
    out["low_slope_atr"] = low_slope_atr
    out["convergence"] = convergence
    out["width_now_atr"] = width_now_atr
    out[CHART_PATTERN_COL] = _chart_pattern_ids(
        failed_break=failed_break,
        bars_since_breakout=bars_since_breakout,
        breakout_size_atr=(
            out["breakout_size_atr"].to_numpy(dtype=np.float64)
            if "breakout_size_atr" in out.columns
            else np.zeros(n)
        ),
        peak_diff_atr=peak_diff_atr,
        trough_diff_atr=trough_diff_atr,
        peak_sep_bars=peak_sep_bars,
        trough_sep_bars=trough_sep_bars,
        dist_neck_atr=dist_neck_atr,
        high_slope_atr=high_slope_atr,
        low_slope_atr=low_slope_atr,
        convergence=convergence,
        width_now_atr=width_now_atr,
        pole_disp_atr=(
            out["pole_disp_atr"].to_numpy(dtype=np.float64)
            if "pole_disp_atr" in out.columns
            else np.zeros(n)
        ),
        flag_width_atr=(
            out["flag_width_atr"].to_numpy(dtype=np.float64)
            if "flag_width_atr" in out.columns
            else np.zeros(n)
        ),
        flag_slope_atr=(
            out["flag_slope_atr"].to_numpy(dtype=np.float64)
            if "flag_slope_atr" in out.columns
            else np.zeros(n)
        ),
        structure_bias=structure_bias,
    )
    return out

def _chart_pattern_ids(
    *,
    failed_break: np.ndarray,
    bars_since_breakout: np.ndarray,
    breakout_size_atr: np.ndarray,
    peak_diff_atr: np.ndarray,
    trough_diff_atr: np.ndarray,
    peak_sep_bars: np.ndarray,
    trough_sep_bars: np.ndarray,
    dist_neck_atr: np.ndarray,
    high_slope_atr: np.ndarray,
    low_slope_atr: np.ndarray,
    convergence: np.ndarray,
    width_now_atr: np.ndarray,
    pole_disp_atr: np.ndarray,
    flag_width_atr: np.ndarray,
    flag_slope_atr: np.ndarray,
    structure_bias: np.ndarray,
) -> np.ndarray:
    """Causal discrete chart-pattern id at bar t (0=NONE ... 8=FAILED_BREAK).

    FAILED_BREAK is a rising-edge event inside ``_FAILED_BREAK_BARS`` of the
    last Donchian breakout. It does not overwrite FLAG/TRIANGLE/DOUBLE/CHANNEL
    and does not stay on for the whole post-breakout regime. BREAKOUT is the
    event bar only and also does not overwrite those named geometries.
    """
    _ = breakout_size_atr
    n = int(failed_break.shape[0])
    ids = np.zeros(n, dtype=np.int64)
    channel = (np.abs(high_slope_atr - low_slope_atr) < 0.15) & (width_now_atr > 0.6)
    triangle = (np.abs(convergence) >= 0.08) & (width_now_atr < 2.5)
    double_top = (
        (np.abs(peak_diff_atr) < 0.45)
        & (peak_sep_bars >= 4)
        & (dist_neck_atr > 0.15)
    )
    double_bottom = (
        (np.abs(trough_diff_atr) < 0.45)
        & (trough_sep_bars >= 4)
        & (dist_neck_atr < -0.15)
    )
    flag_bull = (
        (pole_disp_atr > 1.0)
        & (flag_width_atr < 2.0)
        & (flag_slope_atr < 0.0)
        & (structure_bias > 0.15)
    )
    flag_bear = (
        (pole_disp_atr < -1.0)
        & (flag_width_atr < 2.0)
        & (flag_slope_atr > 0.0)
        & (structure_bias < -0.15)
    )
    # Event bar only. abs(size vs Donchian high) is not a two-sided breakout
    # and was tagging most 5m bars as BREAKOUT.
    fresh_break = bars_since_breakout <= 0.5
    failed_now = (failed_break >= 0.5) & (
        bars_since_breakout <= float(_FAILED_BREAK_BARS)
    )
    failed_prev = np.empty_like(failed_now)
    failed_prev[0] = False
    if n > 1:
        failed_prev[1:] = failed_now[:-1]
    failed_pulse = failed_now & ~failed_prev
    ids = np.where(channel, 6, ids)
    ids = np.where(triangle, 3, ids)
    ids = np.where(flag_bull, 1, ids)
    ids = np.where(flag_bear, 2, ids)
    ids = np.where(double_bottom, 5, ids)
    ids = np.where(double_top, 4, ids)
    named = np.isin(ids, [1, 2, 3, 4, 5, 6])
    ids = np.where(fresh_break & ~named, 7, ids)
    ids = np.where(failed_pulse & ~named, 8, ids)
    return ids

def add_market_structure_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add causal native-TF structure, trend, S/R, breakout, and geometry columns.

    Args:
        df: OHLCV frame with atr (computed if missing).

    Returns:
        Same frame with ``NATIVE_STRUCTURE_COLS`` assigned and finite-filled.
    """
    out = _ensure_atr(df)
    out = _vectorized_trend_and_ma(out)
    out = _zigzag_structure_loop(out)
    for col in NATIVE_STRUCTURE_COLS:
        if col not in out.columns:
            out[col] = 0.0
        out[col] = out[col].astype(float).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    if CHART_PATTERN_COL not in out.columns:
        out[CHART_PATTERN_COL] = 0
    out[CHART_PATTERN_COL] = (
        pd.to_numeric(out[CHART_PATTERN_COL], errors="coerce").fillna(0).astype(np.int64)
    )
    return out

def _resample_closed_ohlcv(
    df: pd.DataFrame,
    htf_minutes: int,
    native_minutes: int = _HTF_NATIVE_MINUTES,
) -> pd.DataFrame:
    """Resample native bars to a higher TF, keeping only complete buckets."""
    expected = max(1, int(round(htf_minutes / native_minutes)))
    src = df[["time", "open", "high", "low", "close"]].copy()
    if "volume" in df.columns:
        src["volume"] = df["volume"]
    else:
        src["volume"] = 0.0
    src["time"] = pd.to_datetime(src["time"], utc=True)
    grouped = src.set_index("time")
    rule = f"{int(htf_minutes)}min"
    agg = grouped.resample(rule, label="right", closed="right").agg(
        {
            "open": "first",
            "high": "max",
            "low": "min",
            "close": "last",
            "volume": "sum",
        }
    )
    counts = grouped["close"].resample(rule, label="right", closed="right").count()
    complete = agg.loc[counts >= expected].dropna(subset=["open", "high", "low", "close"])
    if complete.empty:
        return complete
    complete = complete.reset_index()
    return complete

def add_htf_structure_features(df: pd.DataFrame) -> pd.DataFrame:
    """Merge closed 15m/30m/1h/2h structure onto a 5m frame (backward asof)."""
    out = df.copy()
    out["time"] = pd.to_datetime(out["time"], utc=True)
    out = out.sort_values("time").reset_index(drop=True)

    for tf_name in HTF_SOURCE_TFS:
        minutes = RESOLUTION_MINUTES[tf_name]
        htf_raw = _resample_closed_ohlcv(out, minutes, _HTF_NATIVE_MINUTES)
        prefixed = {f"htf_{tf_name}_{field}": 0.0 for field in HTF_FEATURE_FIELDS}
        if htf_raw.empty:
            for col, val in prefixed.items():
                out[col] = val
            continue
        htf_feat = add_market_structure_features(_ensure_atr(htf_raw))
        merge_cols = {"time": htf_feat["time"]}
        for field in HTF_FEATURE_FIELDS:
            src = _HTF_NATIVE_MAP[field]
            merge_cols[f"htf_{tf_name}_{field}"] = htf_feat[src].to_numpy(dtype=np.float64)
        htf_df = pd.DataFrame(merge_cols).sort_values("time")
        out = pd.merge_asof(
            out.sort_values("time"),
            htf_df,
            on="time",
            direction="backward",
        )

    for col in HTF_STRUCTURE_COLS:
        if col not in out.columns:
            out[col] = 0.0
        out[col] = out[col].astype(float).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return out


## Feature engineering

In [ ]:
"""Causal per-TF feature engineering for BTCUSD transformers."""

from __future__ import annotations

from typing import Optional, Sequence

import numpy as np
import pandas as pd

WICK_NEGLIGIBLE = 0.05
WICK_BALANCE_MAX = 0.25
FLAT_ATR_MULT = 1e-4
_RATIO_COLS = ("body_ratio", "upper_wick_ratio", "lower_wick_ratio")
_STRUCTURE_EPS = 1e-9
_ATR_NORM_CLIP = 8.0

def assemble_raw_frame(
    df: pd.DataFrame,
    *,
    funding_df: Optional[pd.DataFrame] = None,
    oi_df: Optional[pd.DataFrame] = None,
) -> pd.DataFrame:
    """Merge OHLCV with funding/OI using the same path as live inference."""
    out = df.copy()
    if "timestamp" in out.columns and "time" not in out.columns:
        out["time"] = pd.to_datetime(out["timestamp"], utc=True)
    elif "time" in out.columns:
        out["time"] = pd.to_datetime(out["time"], utc=True)
    else:
        raise ValueError("OHLCV frame requires timestamp or time column")

    required = ("open", "high", "low", "close", "volume")
    missing = [c for c in required if c not in out.columns]
    if missing:
        raise ValueError(f"OHLCV frame missing columns: {missing}")

    out = out.sort_values("time").reset_index(drop=True)

    if funding_df is not None and not funding_df.empty:
        fund = funding_df.copy()
        if "timestamp" in fund.columns:
            fund["time"] = pd.to_datetime(fund["timestamp"], utc=True)
        rate_col = "funding_rate" if "funding_rate" in fund.columns else "close"
        fund = fund[["time", rate_col]].rename(columns={rate_col: "funding_rate"})
        out = pd.merge_asof(
            out.sort_values("time"),
            fund.sort_values("time"),
            on="time",
            direction="backward",
        )
    elif "funding_rate" not in out.columns:
        out["funding_rate"] = np.nan

    if oi_df is not None and not oi_df.empty:
        oi = oi_df.copy()
        if "timestamp" in oi.columns:
            oi["time"] = pd.to_datetime(oi["timestamp"], utc=True)
        oi_col = None
        for candidate in ("open_interest", "oi_contracts", "close"):
            if candidate in oi.columns:
                oi_col = candidate
                break
        if oi_col is not None:
            oi = oi[["time", oi_col]].rename(columns={oi_col: "open_interest"})
            out = pd.merge_asof(
                out.sort_values("time"),
                oi.sort_values("time"),
                on="time",
                direction="backward",
            )
    elif "open_interest" not in out.columns:
        out["open_interest"] = np.nan

    if "funding_rate" in out.columns:
        out["funding_rate"] = out["funding_rate"].ffill()
    if "open_interest" in out.columns:
        out["open_interest"] = out["open_interest"].ffill()

    return out

def prepare_raw_frame(
    df: pd.DataFrame,
    *,
    funding_df: Optional[pd.DataFrame] = None,
    oi_df: Optional[pd.DataFrame] = None,
) -> pd.DataFrame:
    """Alias for assemble_raw_frame (agent inference entry point)."""
    return assemble_raw_frame(df, funding_df=funding_df, oi_df=oi_df)

def classify_candle_shape(out: pd.DataFrame, *, sr_window: int) -> pd.Series:
    """Assign mutually exclusive single-bar candle class ids 0..12.

    Uses bar-i OHLC geometry plus rolling body-size quantiles up to bar i.
    First matching ``np.select`` condition wins.

    Args:
        out: Frame with open/high/low/close, atr, and wick/body ratio columns.
        sr_window: Rolling lookback for doji/marubozu quantiles.

    Returns:
        int64 Series of class ids aligned with ``out.index``.
    """
    rng_raw = (out["high"] - out["low"]).astype(float)
    atr = out["atr"] if "atr" in out.columns else pd.Series(0.0, index=out.index)
    is_flat = (rng_raw.fillna(0) <= 0) | (
        rng_raw.fillna(0) < atr.fillna(0) * FLAT_ATR_MULT
    )

    body_abs = out["body_ratio"].abs()
    upper = out["upper_wick_ratio"]
    lower = out["lower_wick_ratio"]
    doji_thresh = body_abs.rolling(sr_window, min_periods=20).quantile(0.15)
    marubozu_thresh = body_abs.rolling(sr_window, min_periods=20).quantile(0.85)
    body_median = body_abs.rolling(sr_window, min_periods=20).median()

    wick_sum = upper + lower
    wick_imbalance = (upper - lower).abs() / (wick_sum + 1e-9)
    cond_wicks_balanced = (wick_sum > 2 * WICK_NEGLIGIBLE) & (
        wick_imbalance < WICK_BALANCE_MAX
    )

    cond_doji = ~is_flat & (body_abs < doji_thresh)
    cond_dragonfly = (
        cond_doji
        & (upper < WICK_NEGLIGIBLE)
        & (lower > 2 * WICK_NEGLIGIBLE)
    )
    cond_gravestone = (
        cond_doji
        & (lower < WICK_NEGLIGIBLE)
        & (upper > 2 * WICK_NEGLIGIBLE)
    )
    cond_doji_standard = cond_doji
    cond_marubozu_bull = (
        ~is_flat
        & ~cond_doji
        & (out["body_ratio"] > marubozu_thresh)
        & (upper < WICK_NEGLIGIBLE)
        & (lower < WICK_NEGLIGIBLE)
    )
    cond_marubozu_bear = (
        ~is_flat
        & ~cond_doji
        & (-out["body_ratio"] > marubozu_thresh)
        & (upper < WICK_NEGLIGIBLE)
        & (lower < WICK_NEGLIGIBLE)
    )
    cond_hammer = (
        ~is_flat
        & ~cond_doji
        & (lower > 2 * body_abs)
        & (upper < body_abs)
    )
    cond_inv_hammer = (
        ~is_flat
        & ~cond_doji
        & (upper > 2 * body_abs)
        & (lower < body_abs)
    )
    cond_spinning = (
        ~is_flat
        & ~cond_doji
        & (body_abs < body_median)
        & cond_wicks_balanced
    )
    cond_belt_bull = (
        ~is_flat
        & ~cond_doji
        & (out["close"] > out["open"])
        & (lower < WICK_NEGLIGIBLE)
        & (upper > WICK_NEGLIGIBLE)
    )
    cond_belt_bear = (
        ~is_flat
        & ~cond_doji
        & (out["close"] < out["open"])
        & (upper < WICK_NEGLIGIBLE)
        & (lower > WICK_NEGLIGIBLE)
    )
    cond_standard_bull = ~is_flat & (out["close"] > out["open"])
    cond_standard_bear = ~is_flat & (out["close"] < out["open"])

    conditions = [
        is_flat,
        cond_dragonfly,
        cond_gravestone,
        cond_doji_standard,
        cond_marubozu_bull,
        cond_marubozu_bear,
        cond_hammer,
        cond_inv_hammer,
        cond_spinning,
        cond_belt_bull,
        cond_belt_bear,
        cond_standard_bull,
        cond_standard_bear,
    ]
    choices = list(range(CANDLE_CLASS_CARDINALITY))
    ids = np.select(conditions, choices, default=3).astype(np.int64)
    return pd.Series(ids, index=out.index, dtype="int64")

def summarize_candle_class_distribution(
    feat_df: pd.DataFrame,
    *,
    high_frac: float = 0.40,
    low_frac: float = 0.001,
) -> pd.Series:
    """Return class fractions and print warnings for extreme imbalance.

    Args:
        feat_df: Feature frame containing ``candle_class_id``.
        high_frac: Warn if any class exceeds this share of bars.
        low_frac: Warn if any present class is below this share.

    Returns:
        Normalized value counts indexed by class id.
    """
    counts = feat_df[CANDLE_CLASS_COL].value_counts(normalize=True).sort_index()
    for cid, frac in counts.items():
        if float(frac) > high_frac or float(frac) < low_frac:
            print(
                f"WARNING: candle class {int(cid)} fraction {float(frac):.4f} "
                f"(thresholds {low_frac:.4f} / {high_frac:.2f})"
            )
    return counts

def add_candle_structure_features(out: pd.DataFrame) -> pd.DataFrame:
    """Add causal bar-geometry and bar-to-bar relation features.

    All columns use OHLCV and ATR at or before bar t. Shift-based values on
    the first bar fill to 0.

    Args:
        out: Frame with open/high/low/close and atr.

    Returns:
        The same frame with seven structure columns assigned.
    """
    rng_raw = (out["high"] - out["low"]).astype(float)
    atr = out["atr"] if "atr" in out.columns else pd.Series(0.0, index=out.index)
    is_flat = rng_raw.fillna(0) <= 0

    close_loc = (out["close"] - out["low"]) / (rng_raw + _STRUCTURE_EPS)
    out["close_loc"] = close_loc.where(~is_flat, 0.5).fillna(0.5)

    out["range_atr"] = (rng_raw / (atr + _STRUCTURE_EPS)).clip(
        -_ATR_NORM_CLIP, _ATR_NORM_CLIP
    )
    out["body_atr"] = (
        (out["close"] - out["open"]).abs() / (atr + _STRUCTURE_EPS)
    ).clip(-_ATR_NORM_CLIP, _ATR_NORM_CLIP)
    out["gap_atr"] = (
        (out["open"] - out["close"].shift(1)) / (atr + _STRUCTURE_EPS)
    ).clip(-_ATR_NORM_CLIP, _ATR_NORM_CLIP)

    prev_high = out["high"].shift(1)
    prev_low = out["low"].shift(1)
    out["inside_bar"] = (
        (out["high"] <= prev_high) & (out["low"] >= prev_low)
    ).astype(np.float32)
    out["outside_bar"] = (
        (out["high"] >= prev_high) & (out["low"] <= prev_low)
    ).astype(np.float32)

    prior_body = out["close"].shift(1) - out["open"].shift(1)
    cur_body = out["close"] - out["open"]
    opposite = np.sign(cur_body) != np.sign(prior_body)
    prior_nonzero = prior_body.abs() > _STRUCTURE_EPS
    penetration = (out["close"] - out["open"].shift(1)) / (
        prior_body.abs() + _STRUCTURE_EPS
    )
    out["engulf_score"] = (
        penetration.clip(-2.0, 2.0)
        * np.sign(cur_body)
        * opposite.astype(np.float64)
        * prior_nonzero.astype(np.float64)
    )

    shift_fill_cols = ("gap_atr", "inside_bar", "outside_bar", "engulf_score")
    for col in shift_fill_cols:
        out[col] = out[col].fillna(0.0)
    return out

def add_features(
    df: pd.DataFrame,
    *,
    resolution_minutes: int = 15,
    atr_period: int = 14,
    rsi_period: int = 14,
    adx_period: int = 14,
    ema_period: int = 50,
    macd_fast: int = 12,
    macd_slow: int = 26,
    macd_signal: int = 9,
    include_htf: bool = True,
) -> pd.DataFrame:
    """Compute causal features on a native TF grid.

    Args:
        include_htf: When True (default), 5m frames also merge resampled HTF
            structure. The fused multi-TF model passes False and encodes each
            timeframe from independently sampled OHLCV instead.
    """
    out = df.copy()
    rv_short = scale_period(16, resolution_minutes)
    rv_long = scale_period(96, resolution_minutes)
    sr_window = scale_period(96, resolution_minutes)
    obv_window = scale_period(96, resolution_minutes)
    vol_window = scale_period(96, resolution_minutes)
    ret2_bars = max(1, scale_period(2, resolution_minutes))

    atr_period = scale_period(atr_period, resolution_minutes)
    rsi_period = scale_period(rsi_period, resolution_minutes)
    adx_period = scale_period(adx_period, resolution_minutes)
    ema_period = scale_period(ema_period, resolution_minutes)
    macd_fast = scale_period(macd_fast, resolution_minutes)
    macd_slow = scale_period(macd_slow, resolution_minutes)
    macd_signal = scale_period(macd_signal, resolution_minutes)

    out["ret_1"] = np.log(out["close"] / out["close"].shift(1))

    prev_close = out["close"].shift(1)
    tr = pd.concat(
        [
            out["high"] - out["low"],
            (out["high"] - prev_close).abs(),
            (out["low"] - prev_close).abs(),
        ],
        axis=1,
    ).max(axis=1)
    out["atr"] = tr.rolling(atr_period).mean()

    out["rv_16"] = out["ret_1"].rolling(rv_short).std()
    out["rv_96"] = out["ret_1"].rolling(rv_long).std()

    out["ema50"] = out["close"].ewm(span=ema_period, adjust=False).mean()
    out["ema50_dist_pct"] = (out["close"] - out["ema50"]) / out["ema50"]

    ema_fast_s = out["close"].ewm(span=macd_fast, adjust=False).mean()
    ema_slow_s = out["close"].ewm(span=macd_slow, adjust=False).mean()
    macd_line = ema_fast_s - ema_slow_s
    macd_signal_line = macd_line.ewm(span=macd_signal, adjust=False).mean()
    out["macd_hist"] = macd_line - macd_signal_line

    delta = out["close"].diff()
    gain = delta.clip(lower=0).rolling(rsi_period).mean()
    loss = (-delta.clip(upper=0)).rolling(rsi_period).mean()
    rs = gain / (loss + 1e-9)
    out["rsi_14"] = 100 - (100 / (1 + rs))

    up_move = out["high"].diff()
    down_move = -out["low"].diff()
    plus_dm = np.where((up_move > down_move) & (up_move > 0), up_move, 0.0)
    minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0.0)
    atr_for_di = tr.rolling(adx_period).mean()
    plus_di = 100 * pd.Series(plus_dm, index=out.index).rolling(adx_period).mean() / (
        atr_for_di + 1e-9
    )
    minus_di = 100 * pd.Series(minus_dm, index=out.index).rolling(adx_period).mean() / (
        atr_for_di + 1e-9
    )
    dx = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di + 1e-9)
    out["adx_14"] = dx.rolling(adx_period).mean()

    obv_raw = (np.sign(out["close"].diff()) * out["volume"]).fillna(0).cumsum()
    out["obv_z"] = (obv_raw - obv_raw.rolling(obv_window).mean()) / (
        obv_raw.rolling(obv_window).std() + 1e-9
    )

    out["vol_z"] = (out["volume"] - out["volume"].rolling(vol_window).mean()) / (
        out["volume"].rolling(vol_window).std() + 1e-9
    )

    rng_raw = (out["high"] - out["low"]).astype(float)
    is_flat = (rng_raw.fillna(0) <= 0) | (
        rng_raw.fillna(0) < out["atr"].fillna(0) * FLAT_ATR_MULT
    )
    rng = rng_raw.replace(0, np.nan)
    out["body_ratio"] = (out["close"] - out["open"]) / rng
    out["upper_wick_ratio"] = (
        out["high"] - out[["open", "close"]].max(axis=1)
    ) / rng
    out["lower_wick_ratio"] = (
        out[["open", "close"]].min(axis=1) - out["low"]
    ) / rng
    out.loc[is_flat, list(_RATIO_COLS)] = 0.0
    out[list(_RATIO_COLS)] = out[list(_RATIO_COLS)].fillna(0.0)

    out["hour"] = out["time"].dt.hour
    out["hour_sin"] = np.sin(2 * np.pi * out["hour"] / 24)
    out["hour_cos"] = np.cos(2 * np.pi * out["hour"] / 24)
    out["dow"] = out["time"].dt.dayofweek
    out["dow_sin"] = np.sin(2 * np.pi * out["dow"] / 7)
    out["dow_cos"] = np.cos(2 * np.pi * out["dow"] / 7)

    rolling_high = out["high"].rolling(sr_window).max()
    rolling_low = out["low"].rolling(sr_window).min()
    out["dist_to_resistance_pct"] = (rolling_high - out["close"]) / out["close"]
    out["dist_to_support_pct"] = (out["close"] - rolling_low) / out["close"]

    if "funding_rate" not in out.columns:
        out["funding_rate"] = np.nan

    if "open_interest" in out.columns:
        oi_z_window = scale_period(96, resolution_minutes)
        out["oi_z"] = (out["open_interest"] - out["open_interest"].rolling(oi_z_window).mean()) / (
            out["open_interest"].rolling(oi_z_window).std() + 1e-9
        )
    else:
        out["oi_z"] = np.nan

    ret_2 = out["close"].pct_change(ret2_bars)
    fund_rate = out["funding_rate"].fillna(0.0)
    funding_deriv = compute_funding_derivatives(
        fund_rate, ret_2, resolution_minutes=resolution_minutes
    )
    for col in funding_deriv.columns:
        out[col] = funding_deriv[col].values

    oi_deriv = compute_oi_derivatives(out, resolution_minutes=resolution_minutes)
    out["oi_change_2"] = oi_deriv["oi_change_2"]
    out["oi_delta_z"] = oi_deriv["oi_delta_z"]
    out["oi_price_divergence"] = oi_deriv["oi_price_divergence"]
    out["oi_acceleration"] = oi_deriv["oi_acceleration"]
    out["funding_x_oi"] = out["funding_zscore"] * oi_deriv["oi_zscore"]

    out = add_candle_structure_features(out)
    out[CANDLE_CLASS_COL] = classify_candle_shape(out, sr_window=sr_window)
    out = add_market_structure_features(out)
    if include_htf and int(resolution_minutes) == 5:
        out = add_htf_structure_features(out)

    return out

def build_feature_matrix(
    df: pd.DataFrame,
    *,
    resolution_minutes: int = 15,
    atr_period: int = 14,
    dropna: bool = True,
) -> pd.DataFrame:
    """Return feature columns ready for windowing."""
    feat = add_features(df, resolution_minutes=resolution_minutes, atr_period=atr_period)
    if dropna:
        feat = feat.dropna().reset_index(drop=True)
    return feat

def latest_closed_feature_row(feat_df: pd.DataFrame) -> pd.Series:
    """Closed-bar row used for diagnostics (second-to-last after dropna)."""
    if len(feat_df) < 2:
        raise ValueError("Need at least 2 feature rows for closed-bar semantics")
    return feat_df.iloc[-2]

def validate_feature_columns(
    feat_df: pd.DataFrame,
    *,
    require_finite_closed_bar: bool = False,
    feature_cols: Optional[Sequence[str]] = None,
) -> None:
    cols = tuple(feature_cols) if feature_cols is not None else FEATURE_COLS
    missing = [c for c in cols if c not in feat_df.columns]
    if missing:
        raise ValueError(f"Feature matrix missing columns: {missing}")
    if CANDLE_CLASS_COL not in feat_df.columns:
        raise ValueError(f"Feature matrix missing {CANDLE_CLASS_COL}")
    ids = feat_df[CANDLE_CLASS_COL]
    if ((ids < 0) | (ids > CANDLE_CLASS_CARDINALITY - 1)).any():
        raise ValueError(
            f"{CANDLE_CLASS_COL} out of range [0, {CANDLE_CLASS_CARDINALITY - 1}]"
        )
    if require_finite_closed_bar and len(feat_df) >= 2:
        closed = latest_closed_feature_row(feat_df)
        for col in cols:
            val = closed[col]
            if not np.isfinite(float(val)):
                raise ValueError(f"Non-finite closed-bar value for {col}: {val}")
        cid = int(closed[CANDLE_CLASS_COL])
        if cid < 0 or cid > CANDLE_CLASS_CARDINALITY - 1:
            raise ValueError(f"Closed-bar {CANDLE_CLASS_COL} out of range: {cid}")


## Inference helpers

In [ ]:
"""Window building and label un-standardization for per-TF transformer inference."""

from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Dict, Mapping, Sequence, Tuple

import numpy as np

def load_feature_config(path: Path) -> Dict[str, Any]:
    raw = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(raw, dict):
        raise ValueError(f"{path} must contain a JSON object")
    return raw

def resolve_feature_config(bundle_dir: Path) -> Dict[str, Any]:
    cfg_path = bundle_dir / TRANSFORMER_FEATURE_CONFIG_FILENAME
    if not cfg_path.is_file():
        raise FileNotFoundError(
            f"Missing {TRANSFORMER_FEATURE_CONFIG_FILENAME} in {bundle_dir}"
        )
    return load_feature_config(cfg_path)

def zscore_window(window: np.ndarray) -> np.ndarray:
    """Per-window z-score (matches Colab WindowDataset)."""
    mu = window.mean(axis=0, keepdims=True)
    sd = window.std(axis=0, keepdims=True) + 1e-6
    return ((window - mu) / sd).astype(np.float32)

def build_continuous_window(
    feat_values: np.ndarray,
    *,
    window_len: int,
    feature_cols: Sequence[str] = FEATURE_COLS,
) -> np.ndarray:
    """Build a z-scored (1, window_len, n_features) tensor from continuous cols."""
    if feat_values.shape[0] < window_len:
        raise ValueError(
            f"Need at least {window_len} feature rows, got {feat_values.shape[0]}"
        )
    window = feat_values[-window_len:, :].astype(np.float32)
    if not np.isfinite(window).all():
        raise ValueError("Feature window contains non-finite values")
    normed = zscore_window(window)
    return normed[np.newaxis, :, :]

def build_candle_class_window(
    class_ids: np.ndarray,
    *,
    window_len: int,
    max_class_id: int = CANDLE_CLASS_CARDINALITY - 1,
) -> np.ndarray:
    """Build a raw (1, window_len) int64 tensor of candle class ids."""
    ids = np.asarray(class_ids).reshape(-1)
    if ids.shape[0] < window_len:
        raise ValueError(
            f"Need at least {window_len} candle class rows, got {ids.shape[0]}"
        )
    window = ids[-window_len:].astype(np.int64)
    if np.any((window < 0) | (window > max_class_id)):
        raise ValueError(
            f"{CANDLE_CLASS_COL} out of range [0, {max_class_id}]"
        )
    return window[np.newaxis, :]

def build_inference_window(
    feat_values: np.ndarray,
    *,
    window_len: int,
    feature_cols: Sequence[str] = FEATURE_COLS,
) -> np.ndarray:
    """Build a single (1, window_len, n_features) tensor from feature matrix values."""
    return build_continuous_window(
        feat_values, window_len=window_len, feature_cols=feature_cols
    )

def unstandardize_continuous(
    pred_z: np.ndarray,
    label_mean: Sequence[float],
    label_std: Sequence[float],
    *,
    label_cols: Sequence[str] = CONTINUOUS_LABEL_COLS,
) -> Dict[str, float]:
    """Map standardized ONNX output back to real units."""
    mean = np.asarray(label_mean, dtype=np.float64)
    std = np.asarray(label_std, dtype=np.float64)
    flat = np.asarray(pred_z, dtype=np.float64).reshape(-1)
    if flat.shape[0] != len(label_cols):
        raise ValueError(
            f"Expected {len(label_cols)} continuous outputs, got {flat.shape[0]}"
        )
    real = flat * std + mean
    return {str(col): float(val) for col, val in zip(label_cols, real)}

def softmax(logits: np.ndarray) -> np.ndarray:
    x = np.asarray(logits, dtype=np.float64).reshape(-1)
    x = x - x.max()
    exp = np.exp(x)
    return exp / (exp.sum() + 1e-12)

def parse_regime_prediction(
    regime_logits: np.ndarray,
    regime_names: Mapping[str, str],
) -> Tuple[int, str, Dict[str, float]]:
    probs = softmax(regime_logits)
    idx = int(np.argmax(probs))
    name = regime_names.get(str(idx), regime_names.get(idx, f"CLASS_{idx}"))
    prob_map = {
        regime_names.get(str(i), f"CLASS_{i}"): float(probs[i])
        for i in range(len(probs))
    }
    return idx, str(name), prob_map

def require_onnx_output_names(
    output_names: Sequence[str],
    *,
    contract_version: str | None = None,
    resolution: str = "15m",
) -> None:
    """Reject bundles that lack the heads required by the given contract."""
    have = {str(name) for name in output_names}
    ver = str(contract_version or FEATURE_CONTRACT_VERSION_V6)
    required = onnx_output_names_for_contract(ver, resolution=resolution)
    known = (
        FEATURE_CONTRACT_VERSION,
        FEATURE_CONTRACT_VERSION_V11,
        FEATURE_CONTRACT_VERSION_V10,
        FEATURE_CONTRACT_VERSION_V8,
        FEATURE_CONTRACT_VERSION_V7,
        FEATURE_CONTRACT_VERSION_V6,
    )
    if ver not in known and not required:
        required = ONNX_OUTPUT_NAMES_V6
    missing = [name for name in required if name not in have]
    if missing:
        raise RuntimeError(
            f"ONNX bundle is not {ver}: missing outputs {missing}. Retrain all TFs."
        )

def feature_config_from_training_export(
    *,
    feature_cols: Sequence[str],
    window_len: int,
    label_mean: Sequence[float],
    label_std: Sequence[float],
    q_edges: Sequence[float],
    config: Mapping[str, Any],
    contract_version: str | None = None,
    onnx_output_names: Sequence[str] | None = None,
    continuous_label_cols: Sequence[str] | None = None,
) -> Dict[str, Any]:
    """Build feature_config.json payload written by Colab export cell."""
    version = str(contract_version or FEATURE_CONTRACT_VERSION_V6)
    if onnx_output_names is not None:
        names = list(onnx_output_names)
    elif version == FEATURE_CONTRACT_VERSION_V11:
        names = list(ONNX_OUTPUT_NAMES_V11)
    elif version == FEATURE_CONTRACT_VERSION_V10:
        names = list(ONNX_OUTPUT_NAMES_V10)
    elif version == FEATURE_CONTRACT_VERSION:
        names = list(ONNX_OUTPUT_NAMES_V9)
    elif version == FEATURE_CONTRACT_VERSION_V8:
        names = list(ONNX_OUTPUT_NAMES_V8)
    elif version == FEATURE_CONTRACT_VERSION_V7:
        names = list(ONNX_OUTPUT_NAMES_V7)
    else:
        names = list(ONNX_OUTPUT_NAMES)
    if continuous_label_cols is not None:
        label_cols = list(continuous_label_cols)
    elif version in (FEATURE_CONTRACT_VERSION, FEATURE_CONTRACT_VERSION_V8):
        label_cols = list(V8_CONTINUOUS_LABEL_COLS)
    else:
        label_cols = list(CONTINUOUS_LABEL_COLS)
    return {
        "feature_contract_version": version,
        "feature_cols": list(feature_cols),
        "categorical_cols": [CANDLE_CLASS_COL],
        "categorical_cardinality": {CANDLE_CLASS_COL: CANDLE_CLASS_CARDINALITY},
        "candle_class_names": {str(k): v for k, v in CANDLE_CLASS_NAMES.items()},
        "window_len": int(window_len),
        "continuous_label_cols": label_cols,
        "label_mean": [float(x) for x in label_mean],
        "label_std": [float(x) for x in label_std],
        "regime_names": {"0": "LOW", "1": "NORMAL", "2": "HIGH", "3": "EXTREME"},
        "structure_outcome_names": {
            str(k): v for k, v in STRUCTURE_OUTCOME_NAMES.items()
        },
        "onnx_output_names": names,
        "vol_regime_quantile_edges": [float(x) for x in q_edges],
        "config": dict(config),
    }

def metadata_from_training_export(
    *,
    resolution: str,
    label_mean: Sequence[float],
    label_std: Sequence[float],
    config: Mapping[str, Any],
    test_metrics: Mapping[str, Any] | None = None,
    export_quality: Mapping[str, Any] | None = None,
    onnx_output_names: Sequence[str] | None = None,
) -> Dict[str, Any]:
    """Build metadata_transformer.json for a per-TF bundle."""
    res = resolution.strip().lower()
    cfg = dict(config)
    names = list(onnx_output_names or ONNX_OUTPUT_NAMES)
    meta: Dict[str, Any] = {
        "version": "transformer_per_tf_v1",
        "model_name": f"jacksparrow_transformer_BTCUSD_{res}",
        "model_family": model_family_for_resolution(res),
        "symbol": str(cfg.get("symbol") or "BTCUSD"),
        "resolution": res,
        "resolution_minutes": int(cfg.get("resolution_minutes") or 15),
        "onnx_filename": onnx_filename_for_resolution(res),
        "feature_config_filename": TRANSFORMER_FEATURE_CONFIG_FILENAME,
        "path_label_horizon_bars": int(cfg.get("path_label_horizon_bars") or 8),
        "atr_period": int(cfg.get("atr_period") or 14),
        "default_threshold": float(cfg.get("default_threshold") or 0.005),
        "primary_signal_mode": "path_edge",
        "onnx_output_names": names,
        "label_mean": [float(x) for x in label_mean],
        "label_std": [float(x) for x in label_std],
        "test_metrics": dict(test_metrics or {}),
        "training_config": cfg,
    }
    if export_quality:
        meta["export_quality"] = dict(export_quality)
    return meta

def training_config_for_resolution(resolution: str) -> Dict[str, Any]:
    return default_training_config(resolution)


## Delta Exchange India data

In [ ]:
"""Public Delta India history fetch for Colab transformer training."""

from __future__ import annotations

import time
from datetime import datetime, timedelta, timezone
from typing import Any, Dict, List, Optional

import pandas as pd
import requests

_RESOLUTION_MINUTES = {
    "1m": 1,
    "3m": 3,
    "5m": 5,
    "10m": 10,
    "15m": 15,
    "30m": 30,
    "1h": 60,
    "2h": 120,
    "4h": 240,
    "1d": 1440,
}

# Documented candles cap is 2000; live India API has returned 4000 (newest-in-range
# when the window is larger). Request windows stay at the documented 2000. The loop
# walks backward from the oldest timestamp actually returned so a lower cap cannot
# open gaps. Production loaders use 500 for the same reason.
DELTA_CANDLE_PAGE_BARS = 2000
REQUEST_DELAY_SECONDS = 0.2
EMPTY_PAGE_RETRIES = 3
MIN_OHLCV_COMPLETENESS = 0.95

def _bar_seconds(resolution: str) -> int:
    if resolution not in _RESOLUTION_MINUTES:
        raise ValueError(
            f"Unsupported resolution {resolution!r}; "
            f"use one of {sorted(_RESOLUTION_MINUTES)}"
        )
    return int(_RESOLUTION_MINUTES[resolution]) * 60

def _parse_candle_rows(payload: Any) -> List[Dict[str, Any]]:
    """Normalize Delta history/candles payloads to a list of candle dicts."""
    if not isinstance(payload, dict):
        return []
    result = payload.get("result", [])
    if isinstance(result, dict):
        result = result.get("candles", []) or []
    if not isinstance(result, list):
        return []
    rows: List[Dict[str, Any]] = []
    for row in result:
        if isinstance(row, dict) and "time" in row:
            rows.append(row)
    return rows

def _candle_epoch(row: Dict[str, Any]) -> Optional[int]:
    try:
        return int(row["time"])
    except (KeyError, TypeError, ValueError):
        return None

def _request_candle_page(
    url: str,
    *,
    symbol: str,
    resolution: str,
    start_ts: int,
    end_ts: int,
) -> List[Dict[str, Any]]:
    last_error: Optional[Exception] = None
    for attempt in range(EMPTY_PAGE_RETRIES):
        try:
            response = requests.get(
                url,
                params={
                    "symbol": symbol,
                    "resolution": resolution,
                    "start": int(start_ts),
                    "end": int(end_ts),
                },
                timeout=30,
            )
            response.raise_for_status()
            payload = response.json()
        except (requests.RequestException, ValueError) as exc:
            last_error = exc
            time.sleep(REQUEST_DELAY_SECONDS * (attempt + 1))
            continue
        if payload.get("success") is False:
            last_error = RuntimeError(f"Delta candles error for {symbol}: {payload}")
            time.sleep(REQUEST_DELAY_SECONDS * (attempt + 1))
            continue
        rows = _parse_candle_rows(payload)
        if rows:
            return rows
        last_error = None
        time.sleep(REQUEST_DELAY_SECONDS * (attempt + 1))
    if last_error is not None:
        raise last_error
    return []

def _rows_to_frame(all_rows: List[Dict[str, Any]]) -> pd.DataFrame:
    df = pd.DataFrame(all_rows)
    expected_cols = ["time", "open", "high", "low", "close", "volume"]
    df = df[[c for c in expected_cols if c in df.columns]]
    df["time"] = pd.to_datetime(df["time"], unit="s", utc=True)
    df = df.drop_duplicates(subset="time").sort_values("time").reset_index(drop=True)
    for col in ["open", "high", "low", "close", "volume"]:
        if col in df.columns:
            df[col] = df[col].astype(float)
    return df

def fetch_candles(
    symbol: str,
    resolution: str,
    start_ts: int,
    end_ts: int,
    base_url: str,
    *,
    page_bars: int = DELTA_CANDLE_PAGE_BARS,
) -> pd.DataFrame:
    """Pull OHLC-style candles with response-driven backward pagination.

    Each request covers at most ``page_bars`` of wall-clock. The next ``end`` is the
    oldest timestamp actually returned minus one second, so a server cap below the
    requested window cannot skip bars.

    Args:
        symbol: Delta symbol (e.g. ``BTCUSD``, ``FUNDING:BTCUSD``).
        resolution: Candle resolution key in ``_RESOLUTION_MINUTES``.
        start_ts: Inclusive Unix seconds.
        end_ts: Exclusive-ish Unix seconds (API treats the range as a window).
        base_url: Exchange origin, without a trailing path.
        page_bars: Max bars per request window (default documented 2000).

    Returns:
        Deduplicated OHLCV frame sorted by ``time`` ascending.

    Raises:
        ValueError: Unknown resolution.
        RuntimeError: No candles in the requested range.
    """
    url = f"{base_url.rstrip('/')}/v2/history/candles"
    bar_seconds = _bar_seconds(resolution)
    page_bars = max(1, int(page_bars))
    start_ts, end_ts = int(start_ts), int(end_ts)
    if end_ts <= start_ts:
        raise ValueError("end_ts must be greater than start_ts")

    all_rows: List[Dict[str, Any]] = []
    cursor_end = end_ts
    seen_oldest: Optional[int] = None
    max_pages = max(2, (end_ts - start_ts) // bar_seconds + 5)

    for _ in range(max_pages):
        if cursor_end <= start_ts:
            break
        page_start = max(start_ts, cursor_end - page_bars * bar_seconds)
        if page_start >= cursor_end:
            break
        rows = _request_candle_page(
            url,
            symbol=symbol,
            resolution=resolution,
            start_ts=page_start,
            end_ts=cursor_end,
        )
        if not rows:
            break
        epochs = [e for e in (_candle_epoch(r) for r in rows) if e is not None]
        if not epochs:
            break
        oldest = min(epochs)
        if seen_oldest is not None and oldest >= seen_oldest:
            break
        seen_oldest = oldest
        all_rows.extend(rows)
        cursor_end = oldest - 1
        time.sleep(REQUEST_DELAY_SECONDS)

    if not all_rows:
        raise RuntimeError(
            f"No candle data returned for symbol={symbol}; "
            "check symbol/resolution/date range."
        )
    return _rows_to_frame(all_rows)

def validate_ohlcv_completeness(
    df: pd.DataFrame,
    resolution: str,
    *,
    min_completeness: float = MIN_OHLCV_COMPLETENESS,
    symbol: str = "",
) -> Dict[str, Any]:
    """Measure gap rate on a native-TF OHLCV frame.

    Completeness is ``1 - gaps / (n - 1)`` where a gap is a bar-to-bar delta greater
    than ``1.5 * bar_seconds``. This catches sawtooth holes from truncated pages.

    Args:
        df: Frame with a ``time`` column (datetime or Unix seconds).
        resolution: Candle resolution key.
        min_completeness: Raise if the gap-free fraction is below this.
        symbol: Optional label for the error message.

    Returns:
        Report with ``rows``, ``gaps``, ``completeness``, and ``span_bars``.

    Raises:
        ValueError: Empty frame or completeness below ``min_completeness``.
    """
    if df is None or df.empty or "time" not in df.columns:
        raise ValueError("Cannot validate completeness on an empty OHLCV frame")
    bar_seconds = _bar_seconds(resolution)
    time_col = df["time"]
    if pd.api.types.is_datetime64_any_dtype(time_col):
        epochs = (pd.to_datetime(time_col, utc=True).astype("int64") // 10**9).tolist()
    else:
        epochs = pd.to_numeric(time_col, errors="coerce").dropna().astype(int).tolist()
    if len(epochs) < 2:
        raise ValueError("Need at least 2 OHLCV bars to validate completeness")

    gaps = 0
    for prev, cur in zip(epochs, epochs[1:]):
        if int(cur) - int(prev) > bar_seconds * 1.5:
            gaps += 1
    completeness = 1.0 - (gaps / max(1, len(epochs) - 1))
    span_bars = int((epochs[-1] - epochs[0]) // bar_seconds) + 1
    report: Dict[str, Any] = {
        "rows": len(epochs),
        "gaps": int(gaps),
        "completeness": round(float(completeness), 4),
        "span_bars": span_bars,
        "expected_bars": span_bars,
    }
    if completeness < min_completeness:
        label = f"{symbol} {resolution}".strip()
        raise ValueError(
            f"OHLCV completeness {completeness:.1%} below {min_completeness:.0%} "
            f"for {label or 'candles'} ({gaps} gaps, {len(epochs)} rows, "
            f"span {span_bars} bars). Refetch with response-driven pagination."
        )
    return report

def validate_derivatives_coverage(
    df: pd.DataFrame,
    *,
    min_coverage: float = 0.5,
    warn_coverage: float = 0.9,
) -> Dict[str, Any]:
    """Check funding/OI non-null coverage over the OHLCV timeline."""
    n = len(df)
    if n == 0:
        raise ValueError("Cannot validate derivatives coverage on empty frame")

    report: Dict[str, Any] = {"rows": n, "columns": {}}
    for col in ("funding_rate", "open_interest"):
        if col not in df.columns:
            report["columns"][col] = {"coverage": 0.0, "status": "missing"}
            continue
        coverage = float(df[col].notna().mean())
        status = "ok"
        if coverage < min_coverage:
            status = "error"
        elif coverage < warn_coverage:
            status = "warn"
        report["columns"][col] = {
            "coverage": round(coverage, 4),
            "status": status,
        }

    worst = min(v["coverage"] for v in report["columns"].values())
    report["worst_coverage"] = worst
    if worst < min_coverage:
        raise ValueError(
            "Derivatives coverage below minimum "
            f"({worst:.1%} < {min_coverage:.0%}): {report['columns']}"
        )
    return report

def fetch_history_bundle(
    *,
    symbol: str,
    resolution: str,
    history_days: int,
    base_url: str,
    min_derivatives_coverage: float = 0.5,
    derivatives_coverage_warn: float = 0.9,
    min_ohlcv_completeness: float = MIN_OHLCV_COMPLETENESS,
) -> pd.DataFrame:
    """Fetch OHLCV + funding + OI merged frame (notebook-compatible)."""
    end_dt = datetime.now(timezone.utc)
    start_dt = end_dt - timedelta(days=history_days)
    start_ts, end_ts = int(start_dt.timestamp()), int(end_dt.timestamp())

    raw_df = fetch_candles(symbol, resolution, start_ts, end_ts, base_url)
    ohlcv_report = validate_ohlcv_completeness(
        raw_df,
        resolution,
        min_completeness=min_ohlcv_completeness,
        symbol=symbol,
    )
    print(
        f"OHLCV {symbol} {resolution}: {ohlcv_report['rows']} bars, "
        f"completeness {ohlcv_report['completeness']:.1%}, "
        f"gaps={ohlcv_report['gaps']}"
    )

    try:
        funding_df = fetch_candles(
            f"FUNDING:{symbol}", resolution, start_ts, end_ts, base_url
        )
        funding_df = funding_df[["time", "close"]].rename(columns={"close": "funding_rate"})
    except Exception:
        funding_df = pd.DataFrame({"time": [], "funding_rate": []})

    try:
        oi_df = fetch_candles(f"OI:{symbol}", resolution, start_ts, end_ts, base_url)
        oi_df = oi_df[["time", "close"]].rename(columns={"close": "open_interest"})
    except Exception:
        oi_df = pd.DataFrame({"time": [], "open_interest": []})

    raw_df = assemble_raw_frame(raw_df, funding_df=funding_df, oi_df=oi_df)

    report = validate_derivatives_coverage(
        raw_df,
        min_coverage=min_derivatives_coverage,
        warn_coverage=derivatives_coverage_warn,
    )
    for col, info in report["columns"].items():
        if info["status"] == "warn":
            print(
                f"WARNING: {col} coverage {info['coverage']:.1%} "
                f"below recommended {derivatives_coverage_warn:.0%}"
            )
        elif info["status"] == "ok":
            print(f"{col} coverage: {info['coverage']:.1%}")

    return raw_df


## Pattern geometry utils

In [ ]:
"""Shared utilities for pattern feature computation."""

from dataclasses import dataclass

import pandas as pd

@dataclass
class CandleGeometry:
    """Pre-computed geometry for a single candle — avoids redundant calculations."""

    body: float
    upper_wick: float
    lower_wick: float
    total_range: float
    body_ratio: float
    upper_ratio: float
    lower_ratio: float
    is_bullish: bool

    @classmethod
    def from_row(cls, row: pd.Series) -> "CandleGeometry":
        body = abs(row["close"] - row["open"])
        upper_wick = row["high"] - max(row["open"], row["close"])
        lower_wick = min(row["open"], row["close"]) - row["low"]
        total_range = row["high"] - row["low"]
        safe_range = total_range if total_range > 1e-10 else 1e-10
        return cls(
            body=body,
            upper_wick=upper_wick,
            lower_wick=lower_wick,
            total_range=total_range,
            body_ratio=body / safe_range,
            upper_ratio=upper_wick / safe_range,
            lower_ratio=lower_wick / safe_range,
            is_bullish=row["close"] >= row["open"],
        )

def compute_atr(df: pd.DataFrame, period: int = 14) -> pd.Series:
    """Compute Average True Range."""
    high_low = df["high"] - df["low"]
    high_close = (df["high"] - df["close"].shift()).abs()
    low_close = (df["low"] - df["close"].shift()).abs()
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    return tr.rolling(period).mean()


## Native candlestick encodings

In [ ]:
"""
Candlestick pattern detection for ML features.

Detects single-candle, two-candle, and three-candle patterns.
Produces ML-ready feature columns aligned with input DataFrame index.
"""

import pandas as pd

# Feature names for registration
CANDLESTICK_FEATURES = [
    "cdl_doji",
    "cdl_long_legged_doji",
    "cdl_dragonfly_doji",
    "cdl_gravestone_doji",
    "cdl_hammer",
    "cdl_inv_hammer",
    "cdl_hanging_man",
    "cdl_shooting_star",
    "cdl_bull_marubozu",
    "cdl_bear_marubozu",
    "cdl_spinning_top",
    "cdl_bull_engulfing",
    "cdl_bear_engulfing",
    "cdl_bull_harami",
    "cdl_bear_harami",
    "cdl_piercing",
    "cdl_dark_cloud",
    "cdl_tweezer_top",
    "cdl_tweezer_bottom",
    "cdl_bull_kicker",
    "cdl_bear_kicker",
    "cdl_morning_star",
    "cdl_evening_star",
    "cdl_three_white_soldiers",
    "cdl_three_black_crows",
    "cdl_three_inside_up",
    "cdl_three_inside_down",
    "cdl_abandoned_baby_bull",
    "cdl_abandoned_baby_bear",
    "cdl_bull_score",
    "cdl_bear_score",
    "cdl_net_score",
    "cdl_reversal_signal",
    "cdl_indecision_score",
    "cdl_body_ratio",
    "cdl_upper_wick_ratio",
    "cdl_lower_wick_ratio",
    "cdl_consecutive_bull",
    "cdl_consecutive_bear",
]

class CandlestickPatternEngine:
    """
    Detects candlestick patterns and produces ML-ready feature columns.
    All methods operate on a full OHLCV DataFrame and return a feature DataFrame
    aligned by index — safe for both training (batch) and live (last-row) use.
    """

    BULL_PATTERN_WEIGHTS = {
        "cdl_hammer": 0.8,
        "cdl_dragonfly_doji": 0.6,
        "cdl_bull_engulfing": 0.95,
        "cdl_bull_harami": 0.6,
        "cdl_piercing": 0.75,
        "cdl_morning_star": 0.9,
        "cdl_three_white_soldiers": 0.85,
        "cdl_bull_marubozu": 0.7,
        "cdl_tweezer_bottom": 0.65,
        "cdl_three_inside_up": 0.7,
        "cdl_abandoned_baby_bull": 0.95,
        "cdl_bull_kicker": 0.9,
        "cdl_inv_hammer": 0.5,
    }

    BEAR_PATTERN_WEIGHTS = {
        "cdl_shooting_star": 0.8,
        "cdl_gravestone_doji": 0.6,
        "cdl_bear_engulfing": 0.95,
        "cdl_bear_harami": 0.6,
        "cdl_dark_cloud": 0.75,
        "cdl_evening_star": 0.9,
        "cdl_three_black_crows": 0.85,
        "cdl_bear_marubozu": 0.7,
        "cdl_tweezer_top": 0.65,
        "cdl_three_inside_down": 0.7,
        "cdl_abandoned_baby_bear": 0.95,
        "cdl_bear_kicker": 0.9,
        "cdl_hanging_man": 0.7,
    }

    def compute_all(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Main entry point. Returns DataFrame of all candlestick pattern features.
        Aligned with input df index.
        """
        required = ["open", "high", "low", "close"]
        for col in required:
            if col not in df.columns:
                raise ValueError(f"Missing required column: {col}")

        geo = list(df.apply(CandleGeometry.from_row, axis=1))
        out = pd.DataFrame(index=df.index)

        # Single-candle patterns
        out["cdl_doji"] = self._doji(df, geo)
        out["cdl_long_legged_doji"] = self._long_legged_doji(df, geo)
        out["cdl_dragonfly_doji"] = self._dragonfly_doji(df, geo)
        out["cdl_gravestone_doji"] = self._gravestone_doji(df, geo)
        out["cdl_hammer"] = self._hammer(df, geo)
        out["cdl_inv_hammer"] = self._inverted_hammer(df, geo)
        out["cdl_hanging_man"] = self._hanging_man(df, geo)
        out["cdl_shooting_star"] = self._shooting_star(df, geo)
        out["cdl_bull_marubozu"] = self._bull_marubozu(df, geo)
        out["cdl_bear_marubozu"] = self._bear_marubozu(df, geo)
        out["cdl_spinning_top"] = self._spinning_top(df, geo)

        # Two-candle patterns
        out["cdl_bull_engulfing"] = self._bull_engulfing(df, geo)
        out["cdl_bear_engulfing"] = self._bear_engulfing(df, geo)
        out["cdl_bull_harami"] = self._bull_harami(df, geo)
        out["cdl_bear_harami"] = self._bear_harami(df, geo)
        out["cdl_piercing"] = self._piercing(df, geo)
        out["cdl_dark_cloud"] = self._dark_cloud(df, geo)
        out["cdl_tweezer_top"] = self._tweezer_top(df, geo)
        out["cdl_tweezer_bottom"] = self._tweezer_bottom(df, geo)
        out["cdl_bull_kicker"] = self._bull_kicker(df, geo)
        out["cdl_bear_kicker"] = self._bear_kicker(df, geo)

        # Three-candle patterns
        out["cdl_morning_star"] = self._morning_star(df, geo)
        out["cdl_evening_star"] = self._evening_star(df, geo)
        out["cdl_three_white_soldiers"] = self._three_white_soldiers(df, geo)
        out["cdl_three_black_crows"] = self._three_black_crows(df, geo)
        out["cdl_three_inside_up"] = self._three_inside_up(df, geo)
        out["cdl_three_inside_down"] = self._three_inside_down(df, geo)
        out["cdl_abandoned_baby_bull"] = self._abandoned_baby(df, geo, direction="bull")
        out["cdl_abandoned_baby_bear"] = self._abandoned_baby(df, geo, direction="bear")

        # Composite scores
        out["cdl_bull_score"] = self._bull_composite(out)
        out["cdl_bear_score"] = self._bear_composite(out)
        out["cdl_net_score"] = out["cdl_bull_score"] - out["cdl_bear_score"]
        out["cdl_reversal_signal"] = out["cdl_net_score"].clip(-1, 1)
        out["cdl_indecision_score"] = (
            out["cdl_doji"] * 0.6
            + out["cdl_long_legged_doji"] * 0.8
            + out["cdl_spinning_top"] * 0.4
        ).clip(0, 1)

        # Raw geometry features
        out["cdl_body_ratio"] = [g.body_ratio for g in geo]
        out["cdl_upper_wick_ratio"] = [g.upper_ratio for g in geo]
        out["cdl_lower_wick_ratio"] = [g.lower_ratio for g in geo]
        out["cdl_consecutive_bull"] = self._consecutive_direction(df, direction="bull")
        out["cdl_consecutive_bear"] = self._consecutive_direction(df, direction="bear")

        return out.fillna(0)

    def _doji(self, df: pd.DataFrame, geo: list) -> pd.Series:
        return pd.Series(
            [1 if g.body_ratio < 0.10 and g.total_range > 0 else 0 for g in geo],
            index=df.index,
        )

    def _long_legged_doji(self, df: pd.DataFrame, geo: list) -> pd.Series:
        return pd.Series(
            [
                1
                if g.body_ratio < 0.10 and g.upper_ratio > 0.30 and g.lower_ratio > 0.30
                else 0
                for g in geo
            ],
            index=df.index,
        )

    def _dragonfly_doji(self, df: pd.DataFrame, geo: list) -> pd.Series:
        return pd.Series(
            [
                1
                if g.body_ratio < 0.10 and g.upper_ratio < 0.05 and g.lower_ratio > 0.60
                else 0
                for g in geo
            ],
            index=df.index,
        )

    def _gravestone_doji(self, df: pd.DataFrame, geo: list) -> pd.Series:
        return pd.Series(
            [
                1
                if g.body_ratio < 0.10 and g.lower_ratio < 0.05 and g.upper_ratio > 0.60
                else 0
                for g in geo
            ],
            index=df.index,
        )

    def _hammer(self, df: pd.DataFrame, geo: list) -> pd.Series:
        atr = df["close"].diff().abs().rolling(14).mean()
        result = []
        for i, g in enumerate(geo):
            atr_val = atr.iloc[i] if i < len(atr) else 0.0
            if pd.isna(atr_val) or atr_val <= 0:
                atr_val = df["close"].iloc[i] * 0.01
            cond = (
                g.lower_wick >= 2.0 * g.body
                and g.upper_ratio < 0.20
                and g.body_ratio > 0.05
                and g.total_range > float(atr_val) * 0.5
            )
            result.append(1 if cond else 0)
        return pd.Series(result, index=df.index)

    def _inverted_hammer(self, df: pd.DataFrame, geo: list) -> pd.Series:
        return pd.Series(
            [
                1
                if g.upper_wick >= 2.0 * g.body
                and g.lower_ratio < 0.20
                and g.body_ratio > 0.05
                else 0
                for g in geo
            ],
            index=df.index,
        )

    def _hanging_man(self, df: pd.DataFrame, geo: list) -> pd.Series:
        return pd.Series(
            [
                1
                if g.lower_wick >= 2.0 * g.body
                and g.upper_ratio < 0.20
                and g.body_ratio > 0.05
                else 0
                for g in geo
            ],
            index=df.index,
        )

    def _shooting_star(self, df: pd.DataFrame, geo: list) -> pd.Series:
        return pd.Series(
            [
                1
                if g.upper_wick >= 2.0 * g.body
                and g.lower_ratio < 0.20
                and g.body_ratio > 0.05
                and not g.is_bullish
                else 0
                for g in geo
            ],
            index=df.index,
        )

    def _bull_marubozu(self, df: pd.DataFrame, geo: list) -> pd.Series:
        return pd.Series(
            [1 if g.is_bullish and g.body_ratio > 0.95 else 0 for g in geo],
            index=df.index,
        )

    def _bear_marubozu(self, df: pd.DataFrame, geo: list) -> pd.Series:
        return pd.Series(
            [1 if not g.is_bullish and g.body_ratio > 0.95 else 0 for g in geo],
            index=df.index,
        )

    def _spinning_top(self, df: pd.DataFrame, geo: list) -> pd.Series:
        return pd.Series(
            [
                1
                if 0.10 < g.body_ratio < 0.40
                and g.upper_ratio > 0.20
                and g.lower_ratio > 0.20
                else 0
                for g in geo
            ],
            index=df.index,
        )

    def _bull_engulfing(self, df: pd.DataFrame, geo: list) -> pd.Series:
        result = [0] * len(df)
        for i in range(1, len(df)):
            prev, curr = geo[i - 1], geo[i]
            if (
                not prev.is_bullish
                and curr.is_bullish
                and df["open"].iloc[i] <= df["close"].iloc[i - 1]
                and df["close"].iloc[i] >= df["open"].iloc[i - 1]
            ):
                result[i] = 1
        return pd.Series(result, index=df.index)

    def _bear_engulfing(self, df: pd.DataFrame, geo: list) -> pd.Series:
        result = [0] * len(df)
        for i in range(1, len(df)):
            prev, curr = geo[i - 1], geo[i]
            if (
                prev.is_bullish
                and not curr.is_bullish
                and df["open"].iloc[i] >= df["close"].iloc[i - 1]
                and df["close"].iloc[i] <= df["open"].iloc[i - 1]
            ):
                result[i] = 1
        return pd.Series(result, index=df.index)

    def _bull_harami(self, df: pd.DataFrame, geo: list) -> pd.Series:
        result = [0] * len(df)
        for i in range(1, len(df)):
            prev, curr = geo[i - 1], geo[i]
            if (
                not prev.is_bullish
                and curr.is_bullish
                and curr.body < prev.body * 0.5
                and df["open"].iloc[i] > df["close"].iloc[i - 1]
                and df["close"].iloc[i] < df["open"].iloc[i - 1]
            ):
                result[i] = 1
        return pd.Series(result, index=df.index)

    def _bear_harami(self, df: pd.DataFrame, geo: list) -> pd.Series:
        result = [0] * len(df)
        for i in range(1, len(df)):
            prev, curr = geo[i - 1], geo[i]
            if (
                prev.is_bullish
                and not curr.is_bullish
                and curr.body < prev.body * 0.5
                and df["open"].iloc[i] < df["close"].iloc[i - 1]
                and df["close"].iloc[i] > df["open"].iloc[i - 1]
            ):
                result[i] = 1
        return pd.Series(result, index=df.index)

    def _piercing(self, df: pd.DataFrame, geo: list) -> pd.Series:
        result = [0] * len(df)
        for i in range(1, len(df)):
            prev, curr = geo[i - 1], geo[i]
            midpoint = (df["open"].iloc[i - 1] + df["close"].iloc[i - 1]) / 2
            if (
                not prev.is_bullish
                and curr.is_bullish
                and df["open"].iloc[i] < df["close"].iloc[i - 1]
                and df["close"].iloc[i] > midpoint
            ):
                result[i] = 1
        return pd.Series(result, index=df.index)

    def _dark_cloud(self, df: pd.DataFrame, geo: list) -> pd.Series:
        result = [0] * len(df)
        for i in range(1, len(df)):
            prev, curr = geo[i - 1], geo[i]
            midpoint = (df["open"].iloc[i - 1] + df["close"].iloc[i - 1]) / 2
            if (
                prev.is_bullish
                and not curr.is_bullish
                and df["open"].iloc[i] > df["close"].iloc[i - 1]
                and df["close"].iloc[i] < midpoint
            ):
                result[i] = 1
        return pd.Series(result, index=df.index)

    def _tweezer_top(self, df: pd.DataFrame, geo: list) -> pd.Series:
        result = [0] * len(df)
        for i in range(1, len(df)):
            high_diff = abs(df["high"].iloc[i] - df["high"].iloc[i - 1])
            atr = df["close"].iloc[max(0, i - 14) : i].diff().abs().mean()
            atr_val = atr if not pd.isna(atr) and atr > 0 else df["close"].iloc[i] * 0.01
            if not geo[i].is_bullish and high_diff < atr_val * 0.1:
                result[i] = 1
        return pd.Series(result, index=df.index)

    def _tweezer_bottom(self, df: pd.DataFrame, geo: list) -> pd.Series:
        result = [0] * len(df)
        for i in range(1, len(df)):
            low_diff = abs(df["low"].iloc[i] - df["low"].iloc[i - 1])
            atr = df["close"].iloc[max(0, i - 14) : i].diff().abs().mean()
            atr_val = atr if not pd.isna(atr) and atr > 0 else df["close"].iloc[i] * 0.01
            if geo[i].is_bullish and low_diff < atr_val * 0.1:
                result[i] = 1
        return pd.Series(result, index=df.index)

    def _bull_kicker(self, df: pd.DataFrame, geo: list) -> pd.Series:
        result = [0] * len(df)
        for i in range(1, len(df)):
            if (
                not geo[i - 1].is_bullish
                and geo[i].is_bullish
                and df["open"].iloc[i] > df["open"].iloc[i - 1]
                and geo[i].body_ratio > 0.5
            ):
                result[i] = 1
        return pd.Series(result, index=df.index)

    def _bear_kicker(self, df: pd.DataFrame, geo: list) -> pd.Series:
        result = [0] * len(df)
        for i in range(1, len(df)):
            if (
                geo[i - 1].is_bullish
                and not geo[i].is_bullish
                and df["open"].iloc[i] < df["open"].iloc[i - 1]
                and geo[i].body_ratio > 0.5
            ):
                result[i] = 1
        return pd.Series(result, index=df.index)

    def _morning_star(self, df: pd.DataFrame, geo: list) -> pd.Series:
        result = [0] * len(df)
        for i in range(2, len(df)):
            a, b, c = geo[i - 2], geo[i - 1], geo[i]
            mid_a = (df["open"].iloc[i - 2] + df["close"].iloc[i - 2]) / 2
            if (
                not a.is_bullish
                and a.body_ratio > 0.4
                and b.body_ratio < 0.3
                and c.is_bullish
                and df["close"].iloc[i] > mid_a
            ):
                result[i] = 1
        return pd.Series(result, index=df.index)

    def _evening_star(self, df: pd.DataFrame, geo: list) -> pd.Series:
        result = [0] * len(df)
        for i in range(2, len(df)):
            a, b, c = geo[i - 2], geo[i - 1], geo[i]
            mid_a = (df["open"].iloc[i - 2] + df["close"].iloc[i - 2]) / 2
            if (
                a.is_bullish
                and a.body_ratio > 0.4
                and b.body_ratio < 0.3
                and not c.is_bullish
                and df["close"].iloc[i] < mid_a
            ):
                result[i] = 1
        return pd.Series(result, index=df.index)

    def _three_white_soldiers(self, df: pd.DataFrame, geo: list) -> pd.Series:
        result = [0] * len(df)
        for i in range(2, len(df)):
            a, b, c = geo[i - 2], geo[i - 1], geo[i]
            if (
                a.is_bullish
                and b.is_bullish
                and c.is_bullish
                and df["close"].iloc[i] > df["close"].iloc[i - 1] > df["close"].iloc[i - 2]
                and a.body_ratio > 0.5
                and b.body_ratio > 0.5
                and c.body_ratio > 0.5
            ):
                result[i] = 1
        return pd.Series(result, index=df.index)

    def _three_black_crows(self, df: pd.DataFrame, geo: list) -> pd.Series:
        result = [0] * len(df)
        for i in range(2, len(df)):
            a, b, c = geo[i - 2], geo[i - 1], geo[i]
            if (
                not a.is_bullish
                and not b.is_bullish
                and not c.is_bullish
                and df["close"].iloc[i] < df["close"].iloc[i - 1] < df["close"].iloc[i - 2]
                and a.body_ratio > 0.5
                and b.body_ratio > 0.5
                and c.body_ratio > 0.5
            ):
                result[i] = 1
        return pd.Series(result, index=df.index)

    def _three_inside_up(self, df: pd.DataFrame, geo: list) -> pd.Series:
        result = [0] * len(df)
        for i in range(2, len(df)):
            a, b, c = geo[i - 2], geo[i - 1], geo[i]
            if (
                not a.is_bullish
                and b.is_bullish
                and c.is_bullish
                and b.body < a.body
                and df["close"].iloc[i] > df["close"].iloc[i - 1]
            ):
                result[i] = 1
        return pd.Series(result, index=df.index)

    def _three_inside_down(self, df: pd.DataFrame, geo: list) -> pd.Series:
        result = [0] * len(df)
        for i in range(2, len(df)):
            a, b, c = geo[i - 2], geo[i - 1], geo[i]
            if (
                a.is_bullish
                and not b.is_bullish
                and not c.is_bullish
                and b.body < a.body
                and df["close"].iloc[i] < df["close"].iloc[i - 1]
            ):
                result[i] = 1
        return pd.Series(result, index=df.index)

    def _abandoned_baby(
        self, df: pd.DataFrame, geo: list, direction: str
    ) -> pd.Series:
        result = [0] * len(df)
        for i in range(2, len(df)):
            a, b, c = geo[i - 2], geo[i - 1], geo[i]
            is_doji_b = b.body_ratio < 0.10
            if direction == "bull":
                gap1 = df["low"].iloc[i - 1] < df["low"].iloc[i - 2]
                gap2 = df["low"].iloc[i] > df["high"].iloc[i - 1]
                if not a.is_bullish and is_doji_b and c.is_bullish and gap1 and gap2:
                    result[i] = 1
            else:
                gap1 = df["high"].iloc[i - 1] > df["high"].iloc[i - 2]
                gap2 = df["high"].iloc[i] < df["low"].iloc[i - 1]
                if a.is_bullish and is_doji_b and not c.is_bullish and gap1 and gap2:
                    result[i] = 1
        return pd.Series(result, index=df.index)

    def _bull_composite(self, out: pd.DataFrame) -> pd.Series:
        score = pd.Series(0.0, index=out.index)
        for col, weight in self.BULL_PATTERN_WEIGHTS.items():
            if col in out.columns:
                score += out[col] * weight
        total_weight = sum(self.BULL_PATTERN_WEIGHTS.values())
        return (score / total_weight).clip(0, 1)

    def _bear_composite(self, out: pd.DataFrame) -> pd.Series:
        score = pd.Series(0.0, index=out.index)
        for col, weight in self.BEAR_PATTERN_WEIGHTS.items():
            if col in out.columns:
                score += out[col] * weight
        total_weight = sum(self.BEAR_PATTERN_WEIGHTS.values())
        return (score / total_weight).clip(0, 1)

    def _consecutive_direction(
        self, df: pd.DataFrame, direction: str
    ) -> pd.Series:
        is_bull = (df["close"] >= df["open"]).astype(int)
        values = is_bull if direction == "bull" else (1 - is_bull)
        result = []
        count = 0
        for v in values:
            count = count + 1 if v == 1 else 0
            result.append(count)
        return pd.Series(result, index=df.index)


## Native chart encodings

In [ ]:
"""
Chart pattern detection for ML features.

Detects support/resistance, trendlines, flags, triangles, reversals, and breakouts.
Produces ML-ready feature columns aligned with input DataFrame index.
"""

import numpy as np
import pandas as pd
from scipy.signal import argrelextrema
from scipy.stats import linregress

# Feature names for registration
CHART_PATTERN_FEATURES = [
    "sr_support_dist_pct",
    "sr_resistance_dist_pct",
    "sr_at_support",
    "sr_at_resistance",
    "sr_support_strength",
    "sr_resistance_strength",
    "sr_range_position",
    "tl_uptrend_detected",
    "tl_downtrend_detected",
    "tl_trend_slope",
    "tl_dist_to_trendline",
    "tl_near_trendline",
    "tl_breakout_up",
    "tl_breakout_down",
    "chp_bull_flag",
    "chp_bear_flag",
    "chp_bull_flag_strength",
    "chp_bear_flag_strength",
    "chp_asc_triangle",
    "chp_desc_triangle",
    "chp_sym_triangle",
    "chp_triangle_apex_dist",
    "chp_double_top",
    "chp_double_bottom",
    "chp_double_top_dist",
    "chp_double_bottom_dist",
    "chp_hs_detected",
    "chp_ihs_detected",
    "bo_at_high",
    "bo_at_low",
    "bo_volume_confirmation",
    "bo_breakout_score",
]

def _chart_safe_atr(atr_value: object, price: float) -> float:
    """Positive ATR for ratios. Zero ATR appears on flat / warmup bars."""
    try:
        a = float(atr_value)
    except (TypeError, ValueError):
        a = float("nan")
    if not np.isfinite(a) or a <= 0.0:
        return max(abs(float(price)) * 0.01, 1e-9)
    return a

class ChartPatternEngine:
    """
    Detects chart patterns and computes structural market features.
    All methods return full-length DataFrames aligned with input index.
    """

    def compute_all(self, df: pd.DataFrame, atr_period: int = 14) -> pd.DataFrame:
        """Main entry point — computes all chart pattern features."""
        atr = self._compute_atr(df, atr_period)
        out = pd.DataFrame(index=df.index)

        sr = self._compute_support_resistance(df, atr)
        out = pd.concat([out, sr], axis=1)

        tl = self._compute_trendlines(df, atr)
        out = pd.concat([out, tl], axis=1)

        flags = self._compute_flags(df, atr)
        out = pd.concat([out, flags], axis=1)

        tri = self._compute_triangles(df, atr)
        out = pd.concat([out, tri], axis=1)

        rev = self._compute_reversal_patterns(df, atr)
        out = pd.concat([out, rev], axis=1)

        bo = self._compute_breakouts(df, atr)
        out = pd.concat([out, bo], axis=1)

        return out.replace([np.inf, -np.inf], 0).fillna(0)

    def _compute_atr(self, df: pd.DataFrame, period: int = 14) -> pd.Series:
        high_low = df["high"] - df["low"]
        high_close = (df["high"] - df["close"].shift()).abs()
        low_close = (df["low"] - df["close"].shift()).abs()
        tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
        return tr.rolling(period).mean()

    def _compute_support_resistance(
        self,
        df: pd.DataFrame,
        atr: pd.Series,
        window: int = 5,
        lookback: int = 60,
    ) -> pd.DataFrame:
        out = pd.DataFrame(index=df.index)
        highs = df["high"].values
        lows = df["low"].values
        closes = df["close"].values

        high_idx = argrelextrema(highs, np.greater, order=window)[0]
        low_idx = argrelextrema(lows, np.less, order=window)[0]

        sr_support_dist = np.full(len(df), np.nan)
        sr_resistance_dist = np.full(len(df), np.nan)
        sr_at_support = np.zeros(len(df))
        sr_at_resistance = np.zeros(len(df))
        sr_support_strength = np.zeros(len(df))
        sr_resistance_strength = np.zeros(len(df))
        sr_range_position = np.full(len(df), 0.5)

        for i in range(lookback, len(df)):
            current_price = closes[i]
            current_atr = _chart_safe_atr(atr.iloc[i], float(current_price))

            recent_highs_idx = high_idx[(high_idx >= i - lookback) & (high_idx < i)]
            recent_lows_idx = low_idx[(low_idx >= i - lookback) & (low_idx < i)]

            if len(recent_highs_idx) > 0:
                resistance_levels = highs[recent_highs_idx]
                above = resistance_levels[resistance_levels > current_price]
                if len(above) > 0:
                    nearest_res = above.min()
                    sr_resistance_dist[i] = (
                        (nearest_res - current_price) / current_price * 100
                    )
                    sr_at_resistance[i] = (
                        1 if abs(nearest_res - current_price) < current_atr else 0
                    )
                    sr_resistance_strength[i] = np.sum(
                        np.abs(resistance_levels - nearest_res) < current_atr * 0.5
                    )

            if len(recent_lows_idx) > 0:
                support_levels = lows[recent_lows_idx]
                below = support_levels[support_levels < current_price]
                if len(below) > 0:
                    nearest_sup = below.max()
                    sr_support_dist[i] = (
                        (current_price - nearest_sup) / current_price * 100
                    )
                    sr_at_support[i] = (
                        1 if abs(current_price - nearest_sup) < current_atr else 0
                    )
                    sr_support_strength[i] = np.sum(
                        np.abs(support_levels - nearest_sup) < current_atr * 0.5
                    )

            sup_d = sr_support_dist[i] if not np.isnan(sr_support_dist[i]) else 0
            res_d = sr_resistance_dist[i] if not np.isnan(sr_resistance_dist[i]) else 0
            total = sup_d + res_d
            if total > 0:
                sr_range_position[i] = res_d / total

        out["sr_support_dist_pct"] = sr_support_dist
        out["sr_resistance_dist_pct"] = sr_resistance_dist
        out["sr_at_support"] = sr_at_support
        out["sr_at_resistance"] = sr_at_resistance
        out["sr_support_strength"] = sr_support_strength
        out["sr_resistance_strength"] = sr_resistance_strength
        out["sr_range_position"] = sr_range_position

        return out

    def _compute_trendlines(
        self,
        df: pd.DataFrame,
        atr: pd.Series,
        min_touches: int = 3,
        lookback: int = 50,
    ) -> pd.DataFrame:
        out = pd.DataFrame(index=df.index)
        highs = df["high"].values
        lows = df["low"].values
        closes = df["close"].values

        high_idx = argrelextrema(highs, np.greater, order=5)[0]
        low_idx = argrelextrema(lows, np.less, order=5)[0]

        tl_uptrend = np.zeros(len(df))
        tl_downtrend = np.zeros(len(df))
        tl_slope = np.zeros(len(df))
        tl_dist = np.zeros(len(df))
        tl_near = np.zeros(len(df))
        tl_break_up = np.zeros(len(df))
        tl_break_down = np.zeros(len(df))

        for i in range(lookback, len(df)):
            current_atr = _chart_safe_atr(atr.iloc[i], float(closes[i]))

            ul_idx = low_idx[(low_idx >= i - lookback) & (low_idx < i)]
            if len(ul_idx) >= min_touches:
                x = ul_idx.astype(float)
                y = lows[ul_idx]
                slope, intercept, r, _, _ = linregress(x, y)
                if slope > 0 and r > 0.7:
                    tl_uptrend[i] = 1
                    trendline_val = slope * i + intercept
                    dist = (closes[i] - trendline_val) / current_atr
                    tl_slope[i] = slope / (closes[i] * 0.0001)
                    tl_dist[i] = max(0, dist)
                    tl_near[i] = 1 if abs(dist) < 1.0 else 0
                    if closes[i] < trendline_val and closes[i - 1] >= (
                        slope * (i - 1) + intercept
                    ):
                        tl_break_down[i] = 1

            dl_idx = high_idx[(high_idx >= i - lookback) & (high_idx < i)]
            if len(dl_idx) >= min_touches:
                x = dl_idx.astype(float)
                y = highs[dl_idx]
                slope, intercept, r, _, _ = linregress(x, y)
                if slope < 0 and r > 0.7:
                    tl_downtrend[i] = 1
                    trendline_val = slope * i + intercept
                    if closes[i] > trendline_val and closes[i - 1] <= (
                        slope * (i - 1) + intercept
                    ):
                        tl_break_up[i] = 1

        out["tl_uptrend_detected"] = tl_uptrend
        out["tl_downtrend_detected"] = tl_downtrend
        out["tl_trend_slope"] = tl_slope
        out["tl_dist_to_trendline"] = tl_dist
        out["tl_near_trendline"] = tl_near
        out["tl_breakout_up"] = tl_break_up
        out["tl_breakout_down"] = tl_break_down

        return out

    def _compute_flags(
        self,
        df: pd.DataFrame,
        atr: pd.Series,
        pole_bars: int = 10,
        flag_bars: int = 20,
    ) -> pd.DataFrame:
        out = pd.DataFrame(index=df.index)
        closes = df["close"].values
        n = len(df)

        bull_flag = np.zeros(n)
        bear_flag = np.zeros(n)
        bull_flag_strength = np.zeros(n)
        bear_flag_strength = np.zeros(n)

        for i in range(pole_bars + flag_bars, n):
            current_atr = _chart_safe_atr(atr.iloc[i], float(closes[i]))

            pole_start = i - pole_bars - flag_bars
            pole_end = i - flag_bars
            pole_move = closes[pole_end] - closes[pole_start]
            pole_magnitude = abs(pole_move) / (current_atr * pole_bars)

            if pole_magnitude > 0.5:
                flag_closes = closes[pole_end:i]
                flag_range = np.max(flag_closes) - np.min(flag_closes)
                flag_drift = closes[i - 1] - closes[pole_end]

                if pole_move > 0:
                    if flag_range < abs(pole_move) * 0.5 and flag_drift < 0:
                        bull_flag[i] = 1
                        bull_flag_strength[i] = pole_magnitude
                else:
                    if flag_range < abs(pole_move) * 0.5 and flag_drift > 0:
                        bear_flag[i] = 1
                        bear_flag_strength[i] = pole_magnitude

        out["chp_bull_flag"] = bull_flag
        out["chp_bear_flag"] = bear_flag
        out["chp_bull_flag_strength"] = bull_flag_strength
        out["chp_bear_flag_strength"] = bear_flag_strength

        return out

    def _compute_triangles(
        self,
        df: pd.DataFrame,
        atr: pd.Series,
        lookback: int = 40,
    ) -> pd.DataFrame:
        out = pd.DataFrame(index=df.index)
        highs = df["high"].values
        lows = df["low"].values
        n = len(df)

        asc_tri = np.zeros(n)
        desc_tri = np.zeros(n)
        sym_tri = np.zeros(n)
        apex_dist = np.zeros(n)

        for i in range(lookback, n):
            window_highs = highs[i - lookback : i]
            window_lows = lows[i - lookback : i]
            x = np.arange(lookback, dtype=float)

            slope_h, _, _, _, _ = linregress(x, window_highs)
            slope_l, _, _, _, _ = linregress(x, window_lows)

            atr_val = _chart_safe_atr(atr.iloc[i], float(highs[i]))

            if abs(slope_h) < 0.1 * atr_val and slope_l > 0:
                asc_tri[i] = 1
            elif slope_h < 0 and abs(slope_l) < 0.1 * atr_val:
                desc_tri[i] = 1
            elif slope_h < 0 and slope_l > 0:
                sym_tri[i] = 1
                price_range = window_highs[-1] - window_lows[-1]
                if abs(slope_h - slope_l) > 1e-10:
                    bars_to_apex = price_range / abs(slope_h - slope_l)
                    apex_dist[i] = bars_to_apex / lookback

        out["chp_asc_triangle"] = asc_tri
        out["chp_desc_triangle"] = desc_tri
        out["chp_sym_triangle"] = sym_tri
        out["chp_triangle_apex_dist"] = apex_dist

        return out

    def _compute_reversal_patterns(
        self,
        df: pd.DataFrame,
        atr: pd.Series,
        lookback: int = 60,
    ) -> pd.DataFrame:
        out = pd.DataFrame(index=df.index)
        highs = df["high"].values
        lows = df["low"].values
        closes = df["close"].values
        n = len(df)

        double_top = np.zeros(n)
        double_bottom = np.zeros(n)
        double_top_dist = np.zeros(n)
        double_bottom_dist = np.zeros(n)
        hs_detected = np.zeros(n)
        ihs_detected = np.zeros(n)

        high_idx = argrelextrema(highs, np.greater, order=5)[0]
        low_idx = argrelextrema(lows, np.less, order=5)[0]

        for i in range(lookback, n):
            current_atr = _chart_safe_atr(atr.iloc[i], float(closes[i]))

            recent_hi = high_idx[(high_idx >= i - lookback) & (high_idx < i)]
            if len(recent_hi) >= 2:
                h1, h2 = highs[recent_hi[-2]], highs[recent_hi[-1]]
                separation = recent_hi[-1] - recent_hi[-2]
                if abs(h1 - h2) < current_atr * 0.5 and 5 <= separation <= lookback // 2:
                    double_top[i] = 1
                    double_top_dist[i] = (closes[i] - min(h1, h2)) / current_atr

            recent_lo = low_idx[(low_idx >= i - lookback) & (low_idx < i)]
            if len(recent_lo) >= 2:
                l1, l2 = lows[recent_lo[-2]], lows[recent_lo[-1]]
                separation = recent_lo[-1] - recent_lo[-2]
                if abs(l1 - l2) < current_atr * 0.5 and 5 <= separation <= lookback // 2:
                    double_bottom[i] = 1
                    double_bottom_dist[i] = (max(l1, l2) - closes[i]) / current_atr

            if len(recent_hi) >= 3:
                p1, p2, p3 = (
                    highs[recent_hi[-3]],
                    highs[recent_hi[-2]],
                    highs[recent_hi[-1]],
                )
                if p2 > p1 and p2 > p3 and abs(p1 - p3) < current_atr:
                    hs_detected[i] = 1

            if len(recent_lo) >= 3:
                t1, t2, t3 = (
                    lows[recent_lo[-3]],
                    lows[recent_lo[-2]],
                    lows[recent_lo[-1]],
                )
                if t2 < t1 and t2 < t3 and abs(t1 - t3) < current_atr:
                    ihs_detected[i] = 1

        out["chp_double_top"] = double_top
        out["chp_double_bottom"] = double_bottom
        out["chp_double_top_dist"] = double_top_dist
        out["chp_double_bottom_dist"] = double_bottom_dist
        out["chp_hs_detected"] = hs_detected
        out["chp_ihs_detected"] = ihs_detected

        return out

    def _compute_breakouts(
        self,
        df: pd.DataFrame,
        atr: pd.Series,
        lookback: int = 20,
    ) -> pd.DataFrame:
        out = pd.DataFrame(index=df.index)
        closes = df["close"].values
        n = len(df)

        bo_at_high = np.zeros(n)
        bo_at_low = np.zeros(n)
        bo_volume_conf = np.zeros(n)
        bo_score = np.zeros(n)

        for i in range(lookback, n):
            window_closes = closes[i - lookback : i]
            range_high = np.max(window_closes)
            range_low = np.min(window_closes)
            current = closes[i]
            current_atr = _chart_safe_atr(atr.iloc[i], float(current))

            if current >= range_high - current_atr * 0.3:
                bo_at_high[i] = 1

            if current <= range_low + current_atr * 0.3:
                bo_at_low[i] = 1

            if "volume" in df.columns:
                vol = df["volume"].values
                avg_vol = np.mean(vol[i - lookback : i])
                if vol[i] > avg_vol * 1.5:
                    bo_volume_conf[i] = 1

            bo_score[i] = float(bo_at_high[i]) * 0.5 + float(bo_volume_conf[i]) * 0.5

        out["bo_at_high"] = bo_at_high
        out["bo_at_low"] = bo_at_low
        out["bo_volume_confirmation"] = bo_volume_conf
        out["bo_breakout_score"] = bo_score

        return out


## Independent MTF frames

In [ ]:
"""Independent multi-timeframe OHLCV construction and causal as-of alignment.

10m is built from two consecutive closed 5m bars outside the model. Higher TFs
are never resampled inside the encoder; they are joined with merge_asof on
fully closed bar close-time.
"""

from __future__ import annotations

from typing import Any, Dict, Mapping, Optional, Sequence

import numpy as np
import pandas as pd

def _ensure_time_column(df: pd.DataFrame) -> pd.DataFrame:
    """Return a copy with UTC ``time`` plus original columns."""
    out = df.copy()
    if "time" in out.columns:
        out["time"] = pd.to_datetime(out["time"], utc=True)
    elif "timestamp" in out.columns:
        ts = out["timestamp"]
        if pd.api.types.is_numeric_dtype(ts):
            out["time"] = pd.to_datetime(ts, unit="s", utc=True)
        else:
            out["time"] = pd.to_datetime(ts, utc=True)
    else:
        raise ValueError("OHLCV frame requires time or timestamp")
    return out.sort_values("time").reset_index(drop=True)

def bar_close_time(open_time: pd.Series, resolution_minutes: int) -> pd.Series:
    """Candle close time from open time (Delta bars are labeled at open)."""
    return pd.to_datetime(open_time, utc=True) + pd.Timedelta(
        minutes=int(resolution_minutes)
    )

def build_10m_ohlcv_from_5m(df5m: pd.DataFrame) -> pd.DataFrame:
    """Construct an independent 10m OHLCV series from two closed 5m bars.

    Aggregation is exchange-aligned (left-closed, left-labeled): each 10m bar
    covers ``[T, T+10m)`` and uses the two 5m opens at T and T+5m. High/low
    are the true extrema of those source bars. This is done *outside* the
    model — the encoder never resamples 5m on the forward pass.
    """
    src = _ensure_time_column(df5m)
    required = ("open", "high", "low", "close", "volume")
    missing = [c for c in required if c not in src.columns]
    if missing:
        raise ValueError(f"5m OHLCV missing columns: {missing}")
    if src.empty:
        return src.iloc[0:0].copy()

    grouped = src.set_index("time")
    rule = "10min"
    agg = grouped.resample(rule, label="left", closed="left").agg(
        {
            "open": "first",
            "high": "max",
            "low": "min",
            "close": "last",
            "volume": "sum",
        }
    )
    counts = grouped["close"].resample(rule, label="left", closed="left").count()
    # Keep complete 2-bar buckets only so a partial last 10m is not a feature.
    complete = counts >= 2
    agg = agg.loc[complete].dropna(subset=["open", "high", "low", "close"])
    out = agg.reset_index()
    out = out.rename(columns={"time": "time"})
    out["timestamp"] = out["time"]
    return out.sort_values("time").reset_index(drop=True)

def closed_bars_asof(
    tf_df: pd.DataFrame,
    decision_time: pd.Timestamp,
    *,
    resolution_minutes: int,
) -> pd.DataFrame:
    """Rows of ``tf_df`` whose candle close time is ``<= decision_time``.

    A 1h bar that opened at 10:00 is still forming at 10:17 and is excluded.
    """
    frame = _ensure_time_column(tf_df)
    if frame.empty:
        return frame
    close_t = bar_close_time(frame["time"], resolution_minutes)
    t = pd.Timestamp(decision_time)
    if t.tzinfo is None:
        t = t.tz_localize("UTC")
    else:
        t = t.tz_convert("UTC")
    return frame.loc[close_t <= t].reset_index(drop=True)

def asof_join_last_closed(
    decision_times: Sequence[pd.Timestamp],
    tf_df: pd.DataFrame,
    *,
    resolution_minutes: int,
) -> pd.DataFrame:
    """For each decision time, attach the last fully closed TF bar (as-of)."""
    left = pd.DataFrame(
        {"decision_time": pd.to_datetime(list(decision_times), utc=True)}
    ).sort_values("decision_time")
    right = _ensure_time_column(tf_df)
    if right.empty:
        empty = left.copy()
        empty["time"] = pd.NaT
        return empty
    right = right.copy()
    right["close_time"] = bar_close_time(right["time"], resolution_minutes)
    right = right.sort_values("close_time")
    merged = pd.merge_asof(
        left,
        right,
        left_on="decision_time",
        right_on="close_time",
        direction="backward",
        allow_exact_matches=True,
    )
    # Guard: never keep a bar that closed after the decision clock.
    late = merged["close_time"].notna() & (merged["close_time"] > merged["decision_time"])
    if late.any():
        merged.loc[late, right.columns] = np.nan
    return merged

def last_n_closed_bars(
    tf_df: pd.DataFrame,
    decision_time: pd.Timestamp,
    *,
    resolution_minutes: int,
    window_len: int,
) -> pd.DataFrame:
    """Last ``window_len`` fully closed native bars at ``decision_time``."""
    closed = closed_bars_asof(
        tf_df, decision_time, resolution_minutes=resolution_minutes
    )
    if len(closed) < int(window_len):
        return closed
    return closed.iloc[-int(window_len) :].reset_index(drop=True)

def normalize_mtf_frames(
    frames: Mapping[str, pd.DataFrame],
) -> Dict[str, pd.DataFrame]:
    """Ensure every fusion TF frame has UTC ``time`` (build 10m if missing)."""
    out: Dict[str, pd.DataFrame] = {}
    for res, df in frames.items():
        if df is None or (isinstance(df, pd.DataFrame) and df.empty):
            continue
        out[str(res)] = _ensure_time_column(df)
    if "10m" not in out and "5m" in out:
        out["10m"] = build_10m_ohlcv_from_5m(out["5m"])
    return out

def fusion_frames_from_fetch(
    df5m: pd.DataFrame,
    df30m: pd.DataFrame,
    df1h: pd.DataFrame,
    df2h: pd.DataFrame,
    *,
    df10m: Optional[pd.DataFrame] = None,
) -> Dict[str, pd.DataFrame]:
    """Assemble the five independent OHLCV datasets used by the fused model."""
    frames: Dict[str, pd.DataFrame] = {
        "5m": df5m,
        "30m": df30m,
        "1h": df1h,
        "2h": df2h,
    }
    if df10m is not None and not df10m.empty:
        frames["10m"] = df10m
    return normalize_mtf_frames(frames)

def assert_no_lookahead(
    tf_df: pd.DataFrame,
    decision_time: pd.Timestamp,
    *,
    resolution_minutes: int,
) -> None:
    """Raise if any bar close time is after the decision clock."""
    closed = closed_bars_asof(
        tf_df, decision_time, resolution_minutes=resolution_minutes
    )
    if closed.empty:
        return
    close_t = bar_close_time(closed["time"], resolution_minutes)
    t = pd.Timestamp(decision_time)
    if t.tzinfo is None:
        t = t.tz_localize("UTC")
    else:
        t = t.tz_convert("UTC")
    if bool((close_t > t).any()):
        raise AssertionError(
            f"Lookahead: {resolution_minutes}m bar close after {t.isoformat()}"
        )

def resolution_minutes(resolution: str) -> int:
    res = str(resolution).strip().lower()
    if res not in RESOLUTION_MINUTES:
        raise ValueError(f"Unsupported resolution {resolution!r}")
    return int(RESOLUTION_MINUTES[res])

def required_fusion_resolutions() -> Sequence[str]:
    return FUSION_INPUT_RESOLUTIONS

def frames_have_fusion_inputs(frames: Mapping[str, Any]) -> bool:
    """True when every fusion TF is present and non-empty."""
    for res in FUSION_INPUT_RESOLUTIONS:
        df = frames.get(res)
        if not isinstance(df, pd.DataFrame) or df.empty:
            return False
    return True


## Fusion 2-class labels

In [ ]:
"""Fusion labels: per-horizon ATR dead zone, 2-class BEAR/BULL for training.

Labels live on the 5m decision clock. Features at t are causal. Targets use
close[t+k] only. No MFE/MAE path heads. NEUTRAL is ignore_index (-1), not a class.
h30m/h1h use 0.5 ATR; h2h uses 0.75 ATR so weak chop is ignored.
"""

from __future__ import annotations

from typing import Dict, Tuple

import numpy as np
import pandas as pd

def _ensure_time(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "time" in out.columns:
        out["time"] = pd.to_datetime(out["time"], utc=True)
    elif "timestamp" in out.columns:
        ts = out["timestamp"]
        if pd.api.types.is_numeric_dtype(ts):
            out["time"] = pd.to_datetime(ts, unit="s", utc=True)
        else:
            out["time"] = pd.to_datetime(ts, utc=True)
    else:
        raise ValueError("Label frame requires time or timestamp")
    return out.sort_values("time").reset_index(drop=True)

def horizon_atr_weak(horizon_key: str) -> float:
    """Per-head ATR dead-zone width. Unknown keys fall back to 0.5 ATR."""
    return float(FUSION_HORIZON_ATR_WEAK.get(str(horizon_key), HORIZON_DIR_ATR_WEAK))

def three_class_direction(move: float, atr: float) -> int:
    """Map ATR-normalized close-to-close move to BEAR/NEUTRAL/BULL (0/1/2)."""
    ratio = float(move) / max(float(atr), 1e-9)
    if ratio >= float(HORIZON_DIR_ATR_WEAK):
        return 2
    if ratio <= -float(HORIZON_DIR_ATR_WEAK):
        return 0
    return 1

def three_class_to_train_label(class_id: float) -> int:
    """Map internal 3-class id to 2-class train id. NEUTRAL/NaN → ignore_index."""
    if not np.isfinite(class_id):
        return int(FUSION_IGNORE_INDEX)
    cid = int(class_id)
    if cid == 0:
        return 0
    if cid == 2:
        return 1
    return int(FUSION_IGNORE_INDEX)

def compute_fusion_horizon_labels(df5m: pd.DataFrame) -> pd.DataFrame:
    """Label +30m/+1h/+2h position on the 5m grid (internal 3-class 0/1/2).

    ``y_h`` at bar t uses ``close[t+k] - close[t]`` over ATR at t.
    The last ``MAX_FUSION_HORIZON_BARS`` rows are NaN (insufficient future).
    Training maps NEUTRAL to ignore_index via ``fusion_label_matrix``.
    """
    out = _ensure_time(df5m)
    n = len(out)
    close = out["close"].to_numpy(dtype=np.float64)
    if "atr" in out.columns:
        atr = out["atr"].to_numpy(dtype=np.float64)
    else:
        prev = np.concatenate([[close[0]], close[:-1]])
        tr = np.maximum(
            out["high"].to_numpy(dtype=np.float64) - out["low"].to_numpy(dtype=np.float64),
            np.maximum(np.abs(out["high"].to_numpy() - prev), np.abs(out["low"].to_numpy() - prev)),
        )
        atr = pd.Series(tr).rolling(14, min_periods=1).mean().to_numpy(dtype=np.float64)
    max_k = int(MAX_FUSION_HORIZON_BARS)
    n_valid = max(n - max_k, 0)
    atr_valid = atr[:n_valid].copy()
    atr_valid = np.where(np.isfinite(atr_valid), atr_valid, 0.0)
    denom = np.maximum(atr_valid, 1e-9)
    for key, col, k in zip(FUSION_HORIZON_KEYS, FUSION_DIR_COLS, FUSION_HORIZON_BARS_5M):
        weak = horizon_atr_weak(str(key))
        labels = np.full(n, np.nan, dtype=np.float64)
        if n_valid > 0:
            move = close[int(k) : int(k) + n_valid] - close[:n_valid]
            ratio = move / denom
            cls = np.ones(n_valid, dtype=np.float64)
            cls[ratio >= weak] = 2.0
            cls[ratio <= -weak] = 0.0
            labels[:n_valid] = cls
        out[col] = labels
    return out

def trim_fusion_label_tail(df: pd.DataFrame) -> pd.DataFrame:
    """Drop the last 24 five-minute bars that cannot form a +2h label."""
    if len(df) <= int(MAX_FUSION_HORIZON_BARS):
        return df.iloc[0:0].copy()
    return df.iloc[: -int(MAX_FUSION_HORIZON_BARS)].reset_index(drop=True)

def fusion_label_matrix(df: pd.DataFrame) -> np.ndarray:
    """(n, 3) int64 train labels: BEAR=0, BULL=1, NEUTRAL/NaN=-1."""
    cols = list(FUSION_DIR_COLS)
    arr = df.loc[:, cols].to_numpy(dtype=np.float64)
    out = np.full(arr.shape, int(FUSION_IGNORE_INDEX), dtype=np.int64)
    finite = np.isfinite(arr)
    mapped = np.full(arr.shape, int(FUSION_IGNORE_INDEX), dtype=np.int64)
    mapped[arr == 0] = 0
    mapped[arr == 2] = 1
    out[finite] = mapped[finite]
    return out

def fusion_future_leak_cols() -> frozenset:
    """Columns that must never appear as model inputs."""
    return frozenset(FUSION_DIR_COLS) | frozenset(FUSION_RETIRED_DIR_COLS)

def embargo_bars() -> int:
    return int(FUSION_EMBARGO_BARS)

def direction_name(class_id: int) -> str:
    cid = int(class_id)
    if cid < 0:
        return "IGNORED"
    return FUSION_DIRECTION_NAMES.get(cid, "IGNORED")

def horizon_key_to_col() -> Dict[str, str]:
    return {key: f"{key}_dir" for key in FUSION_HORIZON_KEYS}

def label_class_mix(y: np.ndarray) -> Dict[str, Dict[str, float]]:
    """Per-horizon 2-class counts plus ignore rate for reports."""
    report: Dict[str, Dict[str, float]] = {}
    for j, key in enumerate(FUSION_HORIZON_KEYS):
        col = y[:, j] if y.ndim == 2 else y
        n = int(len(col))
        ignored = int(np.sum(col < 0))
        counts: Dict[str, float] = {
            "BEAR": float(np.sum(col == 0)),
            "BULL": float(np.sum(col == 1)),
            "IGNORED": float(ignored),
            "ignore_rate": float(ignored) / float(max(n, 1)),
        }
        report[key] = counts
    return report

def valid_label_mask(y: np.ndarray) -> np.ndarray:
    """True where at least one horizon is a 2-class BEAR/BULL label."""
    if y.ndim == 1:
        return (y == 0) | (y == 1)
    return np.any((y == 0) | (y == 1), axis=1)

def split_hint() -> Tuple[int, int]:
    """(embargo_bars, max_horizon_bars) for purged splits."""
    return int(FUSION_EMBARGO_BARS), int(MAX_FUSION_HORIZON_BARS)


## Per-TF native features

In [ ]:
"""Per-timeframe pattern encodings for the fused multi-TF model.

Each TF is featured on its *native* grid. Resampled HTF structure is never
added; 10m/30m/1h/2h representations come from independently sampled OHLCV.
"""

from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Mapping, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

def fusion_feature_cols() -> Tuple[str, ...]:
    """Continuous columns in each TF window (native TA + pattern engines)."""
    return fusion_native_feature_cols() + tuple(CANDLESTICK_FEATURES) + tuple(
        CHART_PATTERN_FEATURES
    )

def add_native_tf_features(
    df: pd.DataFrame,
    *,
    resolution: str,
    funding_df: Optional[pd.DataFrame] = None,
    oi_df: Optional[pd.DataFrame] = None,
    atr_period: int = 14,
) -> pd.DataFrame:
    """Causal features on one independently sampled TF. No HTF resample."""
    res = str(resolution).strip().lower()
    minutes = int(RESOLUTION_MINUTES[res])
    raw = assemble_raw_frame(df, funding_df=funding_df, oi_df=oi_df)
    feat = add_features(
        raw,
        resolution_minutes=minutes,
        atr_period=atr_period,
        include_htf=False,
    )
    candle_engine = CandlestickPatternEngine()
    chart_engine = ChartPatternEngine()
    cdl = candle_engine.compute_all(feat)
    chart = chart_engine.compute_all(feat, atr_period=atr_period)
    extra = pd.concat([cdl, chart], axis=1)
    extra = extra.loc[:, ~extra.columns.duplicated()]
    overlap = [c for c in extra.columns if c in feat.columns]
    if overlap:
        extra = extra.drop(columns=overlap)
    if not extra.empty:
        feat = pd.concat([feat, extra], axis=1)
    missing = [c for c in fusion_feature_cols() if c not in feat.columns]
    if missing:
        zeros = pd.DataFrame(0.0, index=feat.index, columns=missing)
        feat = pd.concat([feat, zeros], axis=1)
    if CANDLE_CLASS_COL not in feat.columns:
        feat[CANDLE_CLASS_COL] = 0
    return feat

def _window_matrix(
    feat_df: pd.DataFrame,
    *,
    cols: Sequence[str],
    window_len: int,
) -> np.ndarray:
    values = feat_df.loc[:, list(cols)].to_numpy(dtype=np.float64)
    values = np.nan_to_num(values, nan=0.0, posinf=0.0, neginf=0.0)
    if len(values) < int(window_len):
        pad = np.zeros((int(window_len) - len(values), values.shape[1]), dtype=np.float64)
        values = np.vstack([pad, values]) if len(values) else pad
    else:
        values = values[-int(window_len) :]
    return values.astype(np.float32)

def encode_tf_window(
    tf_df: pd.DataFrame,
    decision_time: pd.Timestamp,
    *,
    resolution: str,
    window_len: int = FUSION_WINDOW_LEN,
    funding_df: Optional[pd.DataFrame] = None,
    oi_df: Optional[pd.DataFrame] = None,
    featured: Optional[pd.DataFrame] = None,
    zscore: bool = True,
) -> np.ndarray:
    """64-bar native window for one TF at decision time T (closed bars only)."""
    minutes = int(RESOLUTION_MINUTES[str(resolution).strip().lower()])
    if featured is None:
        closed = last_n_closed_bars(
            tf_df,
            decision_time,
            resolution_minutes=minutes,
            window_len=max(int(window_len) * 4, 256),
        )
        featured = add_native_tf_features(
            closed,
            resolution=resolution,
            funding_df=funding_df,
            oi_df=oi_df,
        )
        featured = last_n_closed_bars(
            featured,
            decision_time,
            resolution_minutes=minutes,
            window_len=int(window_len),
        )
    else:
        featured = last_n_closed_bars(
            featured,
            decision_time,
            resolution_minutes=minutes,
            window_len=int(window_len),
        )
    cols = fusion_feature_cols()
    window = _window_matrix(featured, cols=cols, window_len=int(window_len))
    if zscore:
        window = zscore_window(window)
    return window

def encode_all_tf_windows(
    frames: Mapping[str, pd.DataFrame],
    decision_time: pd.Timestamp,
    *,
    window_len: int = FUSION_WINDOW_LEN,
    funding_df: Optional[pd.DataFrame] = None,
    oi_df: Optional[pd.DataFrame] = None,
    featured_by_tf: Optional[Mapping[str, pd.DataFrame]] = None,
    zscore: bool = True,
) -> Dict[str, np.ndarray]:
    """Independent Z-ready windows for 5m/10m/30m/1h/2h at time T."""
    normalized = normalize_mtf_frames(frames)
    out: Dict[str, np.ndarray] = {}
    for res in FUSION_INPUT_RESOLUTIONS:
        df = normalized.get(res)
        if df is None or df.empty:
            raise ValueError(f"Missing independent OHLCV for {res}")
        feat = None if featured_by_tf is None else featured_by_tf.get(res)
        out[res] = encode_tf_window(
            df,
            decision_time,
            resolution=res,
            window_len=window_len,
            funding_df=funding_df,
            oi_df=oi_df,
            featured=feat,
            zscore=zscore,
        )
    return out

def precompute_featured_frames(
    frames: Mapping[str, pd.DataFrame],
    *,
    funding_df: Optional[pd.DataFrame] = None,
    oi_df: Optional[pd.DataFrame] = None,
) -> Dict[str, pd.DataFrame]:
    """Feature each TF once (training). Do not resample across TFs."""
    normalized = normalize_mtf_frames(frames)
    featured: Dict[str, pd.DataFrame] = {}
    for res in FUSION_INPUT_RESOLUTIONS:
        df = normalized.get(res)
        if df is None or df.empty:
            raise ValueError(f"Missing independent OHLCV for {res}")
        featured[res] = add_native_tf_features(
            df,
            resolution=res,
            funding_df=funding_df if res == "5m" else None,
            oi_df=oi_df if res == "5m" else None,
        )
    return featured

def stack_tf_windows(
    windows: Mapping[str, np.ndarray],
) -> np.ndarray:
    """Stack TF windows as (n_tf, window_len, n_features) in fusion order."""
    mats: List[np.ndarray] = []
    for res in FUSION_INPUT_RESOLUTIONS:
        if res not in windows:
            raise KeyError(f"Missing window for {res}")
        mats.append(np.asarray(windows[res], dtype=np.float32))
    return np.stack(mats, axis=0)

_FUSION_WINDOW_CHUNK = 4096

def _zscore_windows(mat: np.ndarray) -> np.ndarray:
    """In-place per-window z-score over time. Matches ``zscore_window``."""
    mu = mat.mean(axis=1, keepdims=True)
    sd = mat.std(axis=1, keepdims=True) + 1e-6
    np.subtract(mat, mu, out=mat)
    np.divide(mat, sd, out=mat)
    return mat

def _zscore_windows_chunked(
    mat: np.ndarray,
    *,
    chunk_size: int = _FUSION_WINDOW_CHUNK,
) -> np.ndarray:
    """Z-score in sample chunks so a memmap is not pulled fully into RAM."""
    n = int(mat.shape[0])
    cs = max(int(chunk_size), 1)
    for start in range(0, n, cs):
        stop = min(start + cs, n)
        sl = np.array(mat[start:stop], dtype=np.float32, copy=True)
        _zscore_windows(sl)
        mat[start:stop] = sl
    return mat

def _open_fusion_window_memmap(
    memmap_dir: Path,
    resolution: str,
    shape: Tuple[int, int, int],
) -> np.ndarray:
    """Create a float32 .npy memmap for one TF's training windows."""
    dest = Path(memmap_dir)
    dest.mkdir(parents=True, exist_ok=True)
    path = dest / f"windows_{resolution}.npy"
    if path.exists():
        path.unlink()
    return np.lib.format.open_memmap(path, mode="w+", dtype=np.float32, shape=shape)

def _gather_windows(
    values: np.ndarray,
    end_idx: np.ndarray,
    window_len: int,
    *,
    out: Optional[np.ndarray] = None,
    chunk_size: int = _FUSION_WINDOW_CHUNK,
) -> np.ndarray:
    """Slice ``window_len`` rows ending at each inclusive ``end_idx`` (pad left).

    ``end_idx == -1`` means no closed bar yet and the row stays zeros.
    Fancy-index gathers run in chunks so peak RAM stays near one chunk.
    """
    n_samples = int(end_idx.shape[0])
    n_feat = int(values.shape[1]) if values.size else 0
    width = int(window_len)
    if out is None:
        out = np.zeros((n_samples, width, n_feat), dtype=np.float32)
    elif tuple(out.shape) != (n_samples, width, n_feat):
        raise ValueError(f"out shape {out.shape} != {(n_samples, width, n_feat)}")
    if values.size == 0 or n_feat == 0:
        return out
    n_bars = int(values.shape[0])
    cs = max(int(chunk_size), 1)
    for start in range(0, n_samples, cs):
        stop = min(start + cs, n_samples)
        out[start:stop] = 0
        idx = end_idx[start:stop]
        valid = (idx >= 0) & (idx < n_bars)
        full = valid & (idx >= width - 1)
        if np.any(full):
            ends = idx[full].astype(np.int64, copy=False)
            starts = ends - width + 1
            offsets = starts[:, None] + np.arange(width, dtype=np.int64)[None, :]
            dest = np.flatnonzero(full) + start
            out[dest] = values[offsets]
        for i in np.flatnonzero(valid & ~full):
            end = int(idx[i]) + 1
            sl = values[:end]
            out[start + int(i), width - len(sl) :, :] = sl
    return out

def collect_training_windows(
    featured_by_tf: Mapping[str, pd.DataFrame],
    decision_times: Sequence[pd.Timestamp],
    *,
    window_len: int = FUSION_WINDOW_LEN,
    zscore: bool = True,
    memmap_dir: Optional[Path] = None,
) -> Dict[str, np.ndarray]:
    """Build (n, window, feat) arrays per TF. Decision times are 5m closes.

    Uses ``searchsorted`` on bar close times so each TF is scanned once.
    Semantics match ``encode_tf_window(..., featured=frame)``.

    When ``memmap_dir`` is set, each TF array is a float32 memmap on disk so
    Colab does not hold five full window tensors in RAM.
    """
    n = len(decision_times)
    cols = list(fusion_feature_cols())
    width = int(window_len)
    shape = (int(n), width, len(cols))
    decisions = pd.to_datetime(pd.Index(list(decision_times)), utc=True)
    dec_ns = decisions.asi8
    out: Dict[str, np.ndarray] = {}
    mmap_root = Path(memmap_dir) if memmap_dir is not None else None
    for res in FUSION_INPUT_RESOLUTIONS:
        if res not in featured_by_tf:
            raise KeyError(f"Missing featured frame for {res}")
        if mmap_root is not None:
            mat = _open_fusion_window_memmap(mmap_root, str(res), shape)
        else:
            mat = np.zeros(shape, dtype=np.float32)
        feat = featured_by_tf[res]
        minutes = int(RESOLUTION_MINUTES[str(res).strip().lower()])
        if feat is None or feat.empty or "time" not in feat.columns:
            out[res] = mat
            continue
        missing = [c for c in cols if c not in feat.columns]
        frame = feat
        if missing:
            zeros = pd.DataFrame(0.0, index=feat.index, columns=missing)
            frame = pd.concat([feat, zeros], axis=1)
        close_ns = pd.DatetimeIndex(
            pd.to_datetime(bar_close_time(frame["time"], minutes), utc=True)
        ).asi8
        values = frame.loc[:, cols].to_numpy(dtype=np.float64)
        values = np.nan_to_num(values, nan=0.0, posinf=0.0, neginf=0.0)
        values = values.astype(np.float32, copy=False)
        order = np.argsort(close_ns, kind="mergesort")
        close_sorted = close_ns[order]
        values = values[order]
        end_idx = np.searchsorted(close_sorted, dec_ns, side="right") - 1
        _gather_windows(values, end_idx, width, out=mat)
        del values
        if zscore:
            _zscore_windows_chunked(mat)
        if mmap_root is not None:
            mat.flush()
        out[res] = mat
    return out

# Re-export so tests can assert 15m is not in the fusion input contract.
NATIVE_ONLY_FEATURE_COLS = FEATURE_COLS


## Fused transformer

In [ ]:
"""Shared-encoder multi-TF fusion Transformer (v11).

One parameter set applied independently to 5m/10m/30m/1h/2h windows, softmax
fusion weights, three 2-class horizon heads. Cross-entropy only — no PnL loss.
NEUTRAL labels are ignore_index and never enter the loss.
"""

from __future__ import annotations

import math
from typing import Any, Dict, List, Mapping, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset

def inverse_frequency_class_weights(
    class_ids: np.ndarray,
    n_classes: int,
) -> np.ndarray:
    """Inverse-frequency weights (mean-normalized) for imbalanced CE heads.

    Ignored labels (< 0) are excluded. Empty class counts are clipped to 1.
    """
    ids = np.asarray(class_ids, dtype=np.int64).ravel()
    ids = ids[ids >= 0]
    if ids.size == 0:
        return np.ones(int(n_classes), dtype=np.float32)
    counts = np.bincount(ids, minlength=int(n_classes)).astype(np.float64)
    counts = np.maximum(counts, 1.0)
    weights = 1.0 / counts
    weights = weights * (float(n_classes) / weights.sum())
    return weights.astype(np.float32)

class MtfFusionDataset(Dataset):
    """Per-sample dict of TF windows plus (n_horizons,) 2-class labels."""

    def __init__(
        self,
        windows: Dict[str, np.ndarray],
        labels: np.ndarray,
        *,
        per_window_zscore: bool = False,
    ) -> None:
        self.resolutions = tuple(FUSION_INPUT_RESOLUTIONS)
        n = None
        self.windows: Dict[str, np.ndarray] = {}
        for res in self.resolutions:
            arr = windows[res]
            if not isinstance(arr, np.ndarray) or arr.dtype != np.float32:
                arr = np.asarray(arr, dtype=np.float32)
            self.windows[res] = arr
            if n is None:
                n = len(arr)
            elif len(arr) != n:
                raise ValueError(f"Window length mismatch for {res}")
        self.labels = np.asarray(labels, dtype=np.int64)
        if n is None or len(self.labels) != n:
            raise ValueError("Labels length does not match windows")
        self.per_window_zscore = bool(per_window_zscore)

    def __len__(self) -> int:
        return int(len(self.labels))

    def __getitem__(self, i: int) -> Tuple[torch.Tensor, ...]:
        tensors: List[torch.Tensor] = []
        for res in self.resolutions:
            window = np.ascontiguousarray(self.windows[res][i], dtype=np.float32)
            if not window.flags.writeable:
                window = np.array(window, dtype=np.float32, copy=True)
            if self.per_window_zscore:
                window = zscore_window(window)
            tensors.append(torch.from_numpy(window))
        tensors.append(torch.as_tensor(self.labels[i], dtype=torch.long))
        return tuple(tensors)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512) -> None:
        super().__init__()
        self.pe = nn.Parameter(torch.randn(1, max_len, d_model) * 0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, : x.size(1), :]

class MtfFusionTransformer(nn.Module):
    """Shared encoder E, softmax TF weights, three independent 2-class heads."""

    def __init__(
        self,
        n_features: int,
        d_model: int = 64,
        nhead: int = 4,
        num_layers: int = 2,
        dropout: float = 0.30,
        max_len: int = FUSION_WINDOW_LEN,
        n_tfs: int = 5,
        n_horizons: int = N_FUSION_HORIZONS,
        n_classes: int = FUSION_DIRECTION_CARDINALITY,
    ) -> None:
        super().__init__()
        self.n_tfs = int(n_tfs)
        self.n_horizons = int(n_horizons)
        self.n_classes = int(n_classes)
        self.tf_embed = nn.Embedding(self.n_tfs, d_model)
        self.input_proj = nn.Linear(n_features, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len=max_len)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.fusion_logits = nn.Parameter(torch.zeros(self.n_tfs))
        shared_dim = max(d_model // 2, 16)
        self.shared = nn.Sequential(
            nn.Linear(d_model, shared_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.dir_heads = nn.ModuleList(
            [nn.Linear(shared_dim, self.n_classes) for _ in range(self.n_horizons)]
        )

    def encode_tf(self, x: torch.Tensor, tf_index: int) -> torch.Tensor:
        """Encode one TF window (batch, time, feat) → (batch, d_model)."""
        h = self.input_proj(x)
        tf_ids = torch.full(
            (x.size(0), x.size(1)),
            int(tf_index),
            device=x.device,
            dtype=torch.long,
        )
        h = h + self.tf_embed(tf_ids)
        h = self.pos_enc(h)
        h = self.encoder(h)
        h = self.norm(h)
        return h[:, -1, :]

    def fusion_weights(self) -> torch.Tensor:
        return torch.softmax(self.fusion_logits, dim=0)

    def forward(self, *tf_windows: torch.Tensor) -> Tuple[torch.Tensor, ...]:
        if len(tf_windows) != self.n_tfs:
            raise ValueError(
                f"Expected {self.n_tfs} TF windows, got {len(tf_windows)}"
            )
        zs = [self.encode_tf(tf_windows[i], i) for i in range(self.n_tfs)]
        stacked = torch.stack(zs, dim=1)
        weights = self.fusion_weights().view(1, self.n_tfs, 1)
        combined = (stacked * weights).sum(dim=1)
        shared = self.shared(combined)
        dir_outs = tuple(head(shared) for head in self.dir_heads)
        fusion_logits = self.fusion_logits.unsqueeze(0).expand(shared.size(0), -1)
        return (*dir_outs, fusion_logits)

def fusion_model_from_config(
    n_features: int,
    config: Mapping[str, Any],
) -> MtfFusionTransformer:
    """Build the fused transformer from a training CONFIG mapping."""
    return MtfFusionTransformer(
        n_features=int(n_features),
        d_model=int(config.get("d_model") or 64),
        nhead=int(config.get("nhead") or 4),
        num_layers=int(config.get("num_layers") or 2),
        dropout=float(config.get("dropout") or 0.30),
        max_len=int(config.get("window_len") or FUSION_WINDOW_LEN),
    )

def compute_fusion_loss(
    outputs: Sequence[torch.Tensor],
    labels: torch.Tensor,
    *,
    class_weights: Optional[torch.Tensor] = None,
    label_smoothing: float = 0.0,
    horizon_weights: Optional[Sequence[float]] = None,
) -> torch.Tensor:
    """Weighted mean CE across 2-class horizon heads. Ignores labels < 0."""
    n_h = int(labels.size(1))
    loss = labels.new_zeros(())
    weight_sum = 0.0
    last_logits = outputs[0]
    smooth = float(label_smoothing)
    for j in range(n_h):
        logits = outputs[j]
        last_logits = logits
        target = labels[:, j]
        valid = target >= 0
        if not bool(valid.any()):
            continue
        head_w = 1.0
        if horizon_weights is not None and j < len(horizon_weights):
            head_w = float(horizon_weights[j])
        if head_w <= 0.0:
            continue
        loss = loss + head_w * nn.functional.cross_entropy(
            logits[valid],
            target[valid],
            weight=class_weights,
            reduction="mean",
            label_smoothing=smooth,
        )
        weight_sum += head_w
    if weight_sum <= 0.0:
        return last_logits.sum() * 0.0
    return loss / float(weight_sum)

def train_mtf_fusion(
    model: MtfFusionTransformer,
    train_loader: torch.utils.data.DataLoader,
    val_loader: torch.utils.data.DataLoader,
    *,
    device: torch.device,
    epochs: int,
    lr: float,
    weight_decay: float,
    patience: int,
    class_weights: Optional[torch.Tensor] = None,
    label_smoothing: float = 0.0,
    horizon_weights: Optional[Sequence[float]] = None,
    lr_schedule: str = "constant",
) -> Dict[str, Any]:
    """AdamW + early stopping on validation CE. Never sees the test loader."""
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = None
    if str(lr_schedule).lower() == "cosine":
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            opt, T_max=max(int(epochs), 1)
        )
    best_val = float("inf")
    best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
    stale = 0
    history: List[Tuple[int, float, float]] = []
    aborted = False
    cw = class_weights.to(device) if class_weights is not None else None
    hw: Optional[Sequence[float]] = None
    if horizon_weights is not None:
        hw = [float(x) for x in horizon_weights]

    def _batch_loss(batch: Sequence[torch.Tensor]) -> torch.Tensor:
        windows = batch[:-1]
        labels = batch[-1]
        outs = model(*windows)
        return compute_fusion_loss(
            outs,
            labels,
            class_weights=cw,
            label_smoothing=float(label_smoothing),
            horizon_weights=hw,
        )

    for epoch in range(int(epochs)):
        model.train()
        train_loss = 0.0
        n_train = 0
        for batch in train_loader:
            batch_d = tuple(t.to(device) for t in batch)
            opt.zero_grad(set_to_none=True)
            loss = _batch_loss(batch_d)
            loss_val = float(loss.detach().item())
            if not math.isfinite(loss_val):
                aborted = True
                break
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            train_loss += loss_val
            n_train += 1
        if aborted or n_train == 0:
            break
        model.eval()
        val_loss = 0.0
        n_val = 0
        with torch.no_grad():
            for batch in val_loader:
                batch_d = tuple(t.to(device) for t in batch)
                batch_val = float(_batch_loss(batch_d).item())
                if not math.isfinite(batch_val):
                    aborted = True
                    break
                val_loss += batch_val
                n_val += 1
        if aborted or n_val == 0:
            break
        train_loss /= max(n_train, 1)
        val_loss /= max(n_val, 1)
        history.append((epoch + 1, train_loss, val_loss))
        print(f"epoch {epoch + 1:03d}  train={train_loss:.4f}  val={val_loss:.4f}")
        if scheduler is not None:
            scheduler.step()
        if val_loss < best_val:
            best_val = val_loss
            stale = 0
            best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
        else:
            stale += 1
            if stale >= int(patience):
                print(f"Early stopping at epoch {epoch + 1}")
                break
    model.load_state_dict(best_state)
    ok = (not aborted) and math.isfinite(best_val) and bool(history)
    return {
        "ok": ok,
        "best_val_loss": float(best_val) if math.isfinite(best_val) else float("inf"),
        "epochs_ran": float(history[-1][0]) if history else 0.0,
        "history": history,
    }


## Fusion research pipeline

In [ ]:
"""Training, walk-forward validation, and ONNX export for the fused MTF model.

Walk-forward and Optuna see only development/validation folds. The held-out
test split is scored once after freeze.
"""

from __future__ import annotations

import gc
import json
from pathlib import Path
from typing import Any, Callable, Dict, List, Mapping, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

def set_research_seed(seed: int) -> None:
    """Seed numpy and torch for Colab reproducibility."""
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))

def walk_forward_slices(
    n: int,
    *,
    folds: int,
    embargo: int,
) -> List[Tuple[slice, slice]]:
    """Expanding train, trailing val; never includes a held-out final test."""
    folds = max(int(folds), 1)
    out: List[Tuple[slice, slice]] = []
    for i in range(folds):
        train_end = max(2, int(n * (0.50 + 0.10 * i)))
        val_end = min(n, train_end + max(int(n * 0.12), 8))
        gap = min(max(int(embargo), 0), max(0, val_end - train_end - 1))
        val_start = train_end + gap
        if val_start >= val_end:
            continue
        out.append((slice(0, train_end), slice(val_start, val_end)))
    return out

def split_purged_windows(
    arrays: Mapping[str, np.ndarray],
    *,
    train_frac: float,
    val_frac: float,
    embargo_bars: int,
) -> Dict[str, Dict[str, np.ndarray]]:
    """Time-ordered train/val/test split with embargo gaps between segments."""
    if not arrays:
        raise ValueError("split_purged_windows requires at least one array")
    lengths = {k: len(v) for k, v in arrays.items()}
    n = next(iter(lengths.values()))
    if any(length != n for length in lengths.values()):
        raise ValueError(f"Array length mismatch: {lengths}")
    train_end = int(n * train_frac)
    val_end = train_end + int(n * val_frac)
    embargo = int(embargo_bars)
    slices = {
        "train": slice(0, train_end),
        "val": slice(train_end + embargo, val_end),
        "test": slice(val_end + embargo, n),
    }
    splits = {
        split: {k: v[sl] for k, v in arrays.items()} for split, sl in slices.items()
    }
    first_key = next(iter(arrays))
    if len(splits["train"][first_key]) == 0 or len(splits["val"][first_key]) == 0:
        raise ValueError(
            f"Insufficient windows after split: train={len(splits['train'][first_key])}, "
            f"val={len(splits['val'][first_key])}, test={len(splits['test'][first_key])}"
        )
    return splits

def leakage_audit(feature_cols: Sequence[str]) -> None:
    """Raise if any target or future column leaked into the input feature list.

    Causal structure fields such as ``last_swing_dir`` are valid inputs. Only
    horizon targets, resampled HTF columns, and explicit future_* names are banned.
    """
    banned = set(fusion_future_leak_cols())
    leaked: List[str] = []
    for col in feature_cols:
        name = str(col)
        if (
            name in banned
            or name.startswith("htf_")
            or name.startswith("future_")
            or name.startswith("horizon_")
        ):
            leaked.append(name)
    if leaked:
        raise RuntimeError(f"Future/HTF/target columns in inputs: {leaked}")
    print("Leakage audit passed: no horizon dirs, no resampled HTF structure in X.")

def confusion_counts(pred: np.ndarray, true: np.ndarray, n_classes: int) -> np.ndarray:
    """Integer confusion matrix (rows=true, cols=pred)."""
    mat = np.zeros((n_classes, n_classes), dtype=np.int64)
    pred_i = np.asarray(pred, dtype=np.int64)
    true_i = np.asarray(true, dtype=np.int64)
    for t, p in zip(true_i, pred_i):
        if 0 <= t < n_classes and 0 <= p < n_classes:
            mat[t, p] += 1
    return mat

def feature_finite_report(
    values: np.ndarray,
    feature_cols: Optional[Sequence[str]] = None,
) -> Dict[str, Any]:
    """Print finite-rate before training. Does not mutate caller arrays."""
    arr = np.asarray(values, dtype=np.float64)
    finite_rate = float(np.isfinite(arr).mean()) if arr.size else 1.0
    n_inf = int(np.isinf(arr).sum())
    n_nan = int(np.isnan(arr).sum())
    print(f"feature finite_rate={finite_rate:.4f} inf={n_inf} nan={n_nan}")
    if feature_cols is not None:
        print(f"n_features={len(feature_cols)}")
    return {"finite_rate": finite_rate, "inf": n_inf, "nan": n_nan}

def fusion_feature_groups(feature_cols: Sequence[str]) -> Dict[str, List[str]]:
    """Partition fusion columns into candle / chart / trend / vol / other.

    ``htf_resample`` must stay empty — resampled HTF columns are leakage.
    """
    names = [str(c) for c in feature_cols]
    assigned: set[str] = set()
    groups: Dict[str, List[str]] = {
        "candle": [],
        "chart": [],
        "trend": [],
        "vol": [],
        "other": [],
        "htf_resample": [],
    }
    trend_exact = {"macd_hist", "adx_14", "rsi_14"}
    for col in names:
        if col.startswith("htf_"):
            groups["htf_resample"].append(col)
            assigned.add(col)
            continue
        if "cdl_" in col or "body" in col or "wick" in col:
            groups["candle"].append(col)
            assigned.add(col)
            continue
        if col.startswith(("sr_", "tl_", "chp_")):
            groups["chart"].append(col)
            assigned.add(col)
            continue
        if "ema" in col or col in trend_exact:
            groups["trend"].append(col)
            assigned.add(col)
            continue
        if "atr" in col or col.startswith("rv_"):
            groups["vol"].append(col)
            assigned.add(col)
            continue
    groups["other"] = [c for c in names if c not in assigned]
    if groups["htf_resample"]:
        raise RuntimeError(
            f"htf_ columns must not appear in fusion inputs: {groups['htf_resample']}"
        )
    return groups

class StackedHorizonScorer(nn.Module):
    """Map (B, n_tf, T, F) stacked windows to BULL−BEAR logit for one head."""

    def __init__(self, model: MtfFusionTransformer, horizon_index: int) -> None:
        super().__init__()
        self.inner = model
        self.horizon_index = int(horizon_index)

    def forward(self, stacked: torch.Tensor) -> torch.Tensor:
        n_tfs = int(stacked.size(1))
        windows = [stacked[:, i, :, :].contiguous() for i in range(n_tfs)]
        outputs = self.inner(*windows)
        logits = outputs[self.horizon_index]
        # (B, 1) so GradientExplainer can index outputs[:, idx].
        return (logits[:, 1] - logits[:, 0]).reshape(-1, 1)

def _windows_to_stacked(windows: Mapping[str, np.ndarray]) -> np.ndarray:
    mats: List[np.ndarray] = []
    n: Optional[int] = None
    for res in FUSION_INPUT_RESOLUTIONS:
        arr = np.asarray(windows[res], dtype=np.float32)
        if n is None:
            n = int(arr.shape[0])
        elif int(arr.shape[0]) != n:
            raise ValueError(f"val window length mismatch for {res}")
        mats.append(arr)
    return np.stack(mats, axis=1)

def _print_shap_tables(
    groups: Mapping[str, List[str]],
    per_feature: Mapping[str, float],
    group_totals: Mapping[str, float],
    *,
    horizon_key: str,
    top_k: int = 20,
) -> None:
    total = float(sum(group_totals.values())) or 1.0
    print(f"Gradient SHAP groups ({horizon_key}, mean |SHAP|):")
    for name in ("candle", "chart", "trend", "vol", "other"):
        val = float(group_totals.get(name) or 0.0)
        share = val / total
        n_cols = len(groups.get(name) or [])
        print(f"  {name}: {val:.6f}  share={share:.3f}  n={n_cols}")
    ranked = sorted(per_feature.items(), key=lambda kv: kv[1], reverse=True)
    print(f"Top {min(int(top_k), len(ranked))} features ({horizon_key}):")
    for name, val in ranked[: int(top_k)]:
        print(f"  {name}: {float(val):.6f}")

def _shap_values_to_array(raw: Any, expected_ndim: int = 4) -> np.ndarray:
    """Coerce GradientExplainer output to (B, n_tf, T, F).

    Some SHAP builds add a singleton output axis (rank 5) when the scorer
    returns (B, 1) instead of (B,). Squeeze size-1 axes until rank matches.
    """
    if isinstance(raw, (list, tuple)):
        raw = raw[0]
    if torch.is_tensor(raw):
        raw = raw.detach().cpu().tolist()
    arr = np.asarray(raw, dtype=np.float64)
    while arr.ndim > expected_ndim:
        squeezed = False
        for axis in range(arr.ndim):
            if int(arr.shape[axis]) == 1:
                arr = np.squeeze(arr, axis=axis)
                squeezed = True
                break
        if not squeezed:
            break
    if arr.ndim != expected_ndim:
        raise ValueError(f"SHAP values rank {arr.ndim} != {expected_ndim}")
    return arr

def run_gradient_shap(
    feature_cols: Sequence[str],
    *,
    model: torch.nn.Module,
    val_windows: Mapping[str, np.ndarray],
    device: torch.device,
    config: Optional[Mapping[str, Any]] = None,
) -> Dict[str, Any]:
    """Gradient SHAP on a validation subsample. Never reads the test split."""
    cfg = dict(config or {})
    cols = [str(c) for c in feature_cols]
    groups = fusion_feature_groups(cols)
    try:
        import shap  # type: ignore[import-not-found]
    except ImportError:
        print("SHAP skipped: package 'shap' is not installed.")
        return {
            "ok": False,
            "reason": "shap not installed",
            "groups": {k: list(v) for k, v in groups.items() if k != "htf_resample"},
            "horizons": {},
        }

    stacked = _windows_to_stacked(val_windows)
    n = int(stacked.shape[0])
    if n < 2:
        print("SHAP skipped: need at least 2 validation windows.")
        return {
            "ok": False,
            "reason": "insufficient val windows",
            "groups": {k: list(v) for k, v in groups.items() if k != "htf_resample"},
            "horizons": {},
        }

    seed = int(cfg.get("seed") or 42)
    rng = np.random.default_rng(seed)
    n_bg = max(2, min(int(cfg.get("shap_background") or 32), n))
    n_ex = max(2, min(int(cfg.get("shap_explain_n") or 64), n))
    bg_idx = rng.choice(n, size=n_bg, replace=False)
    ex_idx = rng.choice(n, size=n_ex, replace=False)

    model.eval()
    n_horizons = int(getattr(model, "n_horizons", len(FUSION_HORIZON_KEYS)))
    horizon_keys = list(FUSION_HORIZON_KEYS)[:n_horizons]
    device_t = torch.device(device)
    horizons_out: Dict[str, Any] = {}

    explain_sizes = [n_ex]
    halved = n_ex
    while halved > 8:
        halved = max(8, halved // 2)
        if halved not in explain_sizes:
            explain_sizes.append(halved)

    for h_idx, key in enumerate(horizon_keys):
        scorer = StackedHorizonScorer(model, h_idx).to(device_t)
        scorer.eval()
        shap_arr: Optional[np.ndarray] = None
        used_ex = n_ex
        used_bg = n_bg
        last_err = ""
        for try_ex in explain_sizes:
            try_bg = min(used_bg, try_ex, n_bg)
            try:
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                bg = torch.as_tensor(
                    stacked[bg_idx[:try_bg]], dtype=torch.float32, device=device_t
                )
                ex = torch.as_tensor(
                    stacked[ex_idx[:try_ex]], dtype=torch.float32, device=device_t
                )
                explainer = shap.GradientExplainer(scorer, bg)
                raw = explainer.shap_values(ex)
                shap_arr = _shap_values_to_array(raw)
                used_ex = try_ex
                used_bg = try_bg
                last_err = ""
                break
            except (RuntimeError, MemoryError, ValueError, IndexError) as exc:
                last_err = str(exc)
                print(
                    f"SHAP {key} failed at explain_n={try_ex} "
                    f"({type(exc).__name__}); retrying smaller subsample."
                )
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
        if shap_arr is None:
            print(f"SHAP skipped for {key}: {last_err}")
            horizons_out[key] = {"ok": False, "reason": last_err}
            continue
        per_feat_arr = np.mean(np.abs(shap_arr), axis=(0, 1, 2))
        if per_feat_arr.shape[0] != len(cols):
            print(
                f"SHAP {key}: feature dim {per_feat_arr.shape[0]} != {len(cols)}; skip."
            )
            horizons_out[key] = {"ok": False, "reason": "feature dim mismatch"}
            continue
        per_feature = {
            cols[i]: float(per_feat_arr[i]) for i in range(len(cols))
        }
        group_totals = {
            gname: float(sum(per_feature.get(c, 0.0) for c in gcols))
            for gname, gcols in groups.items()
            if gname != "htf_resample"
        }
        _print_shap_tables(
            groups, per_feature, group_totals, horizon_key=key
        )
        ranked = sorted(per_feature.items(), key=lambda kv: kv[1], reverse=True)
        horizons_out[key] = {
            "ok": True,
            "background_n": int(used_bg),
            "explain_n": int(used_ex),
            "features": per_feature,
            "groups": group_totals,
            "top_features": [
                {"name": n, "mean_abs_shap": float(v)} for n, v in ranked[:20]
            ],
        }

    any_ok = any(bool(row.get("ok")) for row in horizons_out.values())
    report = {
        "ok": bool(any_ok),
        "reason": "gradient shap on val subsample" if any_ok else "all heads failed",
        "groups": {k: list(v) for k, v in groups.items() if k != "htf_resample"},
        "horizons": horizons_out,
        "split": "val",
    }
    return report

def shap_grouped_stub(
    feature_cols: Sequence[str],
    *,
    enabled: bool,
    model: Optional[torch.nn.Module] = None,
    val_windows: Optional[Mapping[str, np.ndarray]] = None,
    device: Optional[torch.device] = None,
    config: Optional[Mapping[str, Any]] = None,
) -> Dict[str, Any]:
    """Grouped SHAP entry point. Computes Gradient SHAP only when enabled."""
    groups = fusion_feature_groups(feature_cols)
    group_lists = {k: list(v) for k, v in groups.items() if k != "htf_resample"}
    if not enabled:
        print("SHAP skipped (CONFIG run_shap=False). Fusion feature groups:")
        for name, cols in group_lists.items():
            print(f"  {name}: {len(cols)} cols")
        return {
            "ok": False,
            "enabled": False,
            "reason": "run_shap=False",
            "groups": group_lists,
            "horizons": {},
        }
    if model is None or val_windows is None:
        print("SHAP enabled but model/val_windows missing; skip compute.")
        return {
            "ok": False,
            "enabled": True,
            "reason": "model or val_windows missing",
            "groups": group_lists,
            "horizons": {},
        }
    dev = device if device is not None else torch.device("cpu")
    print("Gradient SHAP on validation subsample (test split unused).")
    try:
        report = run_gradient_shap(
            feature_cols,
            model=model,
            val_windows=val_windows,
            device=dev,
            config=config,
        )
    except Exception as exc:
        print(
            f"SHAP failed (export continues): {type(exc).__name__}: {exc}"
        )
        return {
            "ok": False,
            "enabled": True,
            "reason": f"{type(exc).__name__}: {exc}",
            "groups": group_lists,
            "horizons": {},
        }
    report["enabled"] = True
    return report

def _fusion_train_extra(config: Mapping[str, Any]) -> Dict[str, Any]:
    raw_w = config.get("horizon_loss_weights") or [1.0, 0.8, 0.4]
    return {
        "label_smoothing": float(config.get("label_smoothing") or 0.0),
        "horizon_weights": [float(x) for x in raw_w],
        "lr_schedule": str(config.get("lr_schedule") or "cosine"),
    }

def optuna_search(
    config: Mapping[str, Any],
    *,
    n_features: int,
    train_loader: DataLoader,
    val_loader: DataLoader,
    device: torch.device,
    class_weights: Optional[torch.Tensor] = None,
    train_fn: Optional[Callable[..., Dict[str, Any]]] = None,
    model_factory: Optional[Callable[..., MtfFusionTransformer]] = None,
) -> Dict[str, Any]:
    """Search dropout / weight_decay / lr on val CE. Never reads test."""
    cfg = dict(config)
    if not cfg.get("run_optuna"):
        print(
            "Optuna skipped (CONFIG run_optuna=False). "
            "Search space: lr, weight_decay, dropout."
        )
        return cfg
    try:
        import optuna
    except ImportError:
        print("Optuna not installed; leaving CONFIG unchanged.")
        return cfg

    trainer = train_fn or train_mtf_fusion
    factory = model_factory or fusion_model_from_config
    extra = _fusion_train_extra(cfg)
    n_trials = max(1, int(cfg.get("optuna_trials") or 8))
    trial_epochs = max(1, int(cfg.get("optuna_trial_epochs") or 12))
    patience = max(2, int(cfg.get("early_stop_patience") or 5))

    def objective(trial: Any) -> float:
        trial_cfg = dict(cfg)
        trial_cfg["lr"] = float(trial.suggest_float("lr", 3e-5, 3e-4, log=True))
        trial_cfg["weight_decay"] = float(
            trial.suggest_float("weight_decay", 1e-4, 3e-3, log=True)
        )
        trial_cfg["dropout"] = float(trial.suggest_float("dropout", 0.20, 0.40))
        model = factory(n_features, trial_cfg).to(device)
        hist = trainer(
            model,
            train_loader,
            val_loader,
            device=device,
            epochs=trial_epochs,
            lr=float(trial_cfg["lr"]),
            weight_decay=float(trial_cfg["weight_decay"]),
            patience=patience,
            class_weights=class_weights,
            **extra,
        )
        if not hist.get("ok"):
            return float("inf")
        return float(hist["best_val_loss"])

    print(f"Optuna: {n_trials} trials on val CE (test unused).")
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials)
    best = dict(study.best_params)
    cfg["lr"] = float(best["lr"])
    cfg["weight_decay"] = float(best["weight_decay"])
    cfg["dropout"] = float(best["dropout"])
    cfg["optuna_best"] = {
        "params": best,
        "best_val_loss": float(study.best_value),
        "n_trials": n_trials,
    }
    print("Optuna best", json.dumps(cfg["optuna_best"], indent=2, default=str))
    return cfg

def optuna_search_stub(
    config: Mapping[str, Any],
    **kwargs: Any,
) -> Dict[str, Any]:
    """Backward-compatible alias. Requires the same kwargs as optuna_search."""
    if not kwargs:
        print("optuna_search_stub needs loaders; leaving CONFIG unchanged.")
        return dict(config)
    return optuna_search(config, **kwargs)

def _decision_times(featured_5m: pd.DataFrame, window_len: int) -> pd.Series:
    times = pd.to_datetime(featured_5m["time"], utc=True)
    return times.iloc[int(window_len) :]

def build_dataset_from_ohlcv(
    frames: Dict[str, pd.DataFrame],
    *,
    window_len: int = FUSION_WINDOW_LEN,
    stride: int = 4,
    memmap_dir: Optional[Path] = None,
) -> Tuple[Dict[str, np.ndarray], np.ndarray, pd.Series]:
    """Native TF windows + 2-class train labels aligned on the 5m decision clock."""
    labeled = compute_fusion_horizon_labels(frames["5m"])
    labeled = trim_fusion_label_tail(labeled)
    featured = precompute_featured_frames(frames)
    feat5_time = featured["5m"][["time"]].copy()
    feat5_time["time"] = pd.to_datetime(feat5_time["time"], utc=True)
    labeled["time"] = pd.to_datetime(labeled["time"], utc=True)
    dir_cols = [c for c in FUSION_DIR_COLS if c in labeled.columns]
    merged = feat5_time.merge(labeled[["time", *dir_cols]], on="time", how="inner")
    merged = merged.iloc[int(window_len) :].reset_index(drop=True)
    if stride > 1:
        merged = merged.iloc[:: int(stride)].reset_index(drop=True)
    y = fusion_label_matrix(merged)
    mask = valid_label_mask(y)
    merged = merged.loc[mask].reset_index(drop=True)
    y = y[mask]
    decision_close = bar_close_time(merged["time"], 5)
    windows = collect_training_windows(
        featured,
        list(decision_close),
        window_len=window_len,
        zscore=True,
        memmap_dir=memmap_dir,
    )
    decision_times = merged["time"].copy()
    del featured, labeled, merged, feat5_time
    gc.collect()
    return windows, y, decision_times

def softmax_np(logits: np.ndarray) -> np.ndarray:
    z = logits - np.max(logits, axis=-1, keepdims=True)
    exp = np.exp(z)
    return exp / np.clip(exp.sum(axis=-1, keepdims=True), 1e-12, None)

def balanced_accuracy(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    n_classes: int = FUSION_DIRECTION_CARDINALITY,
) -> float:
    scores: List[float] = []
    for c in range(int(n_classes)):
        mask = y_true == c
        if not np.any(mask):
            continue
        scores.append(float(np.mean(y_pred[mask] == c)))
    if not scores:
        return 0.0
    return float(np.mean(scores))

def macro_f1(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    n_classes: int = FUSION_DIRECTION_CARDINALITY,
) -> float:
    f1s: List[float] = []
    for c in range(int(n_classes)):
        tp = float(np.sum((y_true == c) & (y_pred == c)))
        fp = float(np.sum((y_true != c) & (y_pred == c)))
        fn = float(np.sum((y_true == c) & (y_pred != c)))
        prec = tp / (tp + fp + 1e-9)
        rec = tp / (tp + fn + 1e-9)
        f1s.append(2 * prec * rec / (prec + rec + 1e-9))
    return float(np.mean(f1s)) if f1s else 0.0

def expected_calibration_error(
    probs: np.ndarray,
    y_true: np.ndarray,
    n_bins: int = 10,
) -> float:
    """ECE on max-class confidence vs correctness."""
    conf = probs.max(axis=-1)
    pred = probs.argmax(axis=-1)
    correct = (pred == y_true).astype(np.float64)
    edges = np.linspace(0.0, 1.0, int(n_bins) + 1)
    ece = 0.0
    n = max(len(y_true), 1)
    for i in range(int(n_bins)):
        lo, hi = edges[i], edges[i + 1]
        if i == n_bins - 1:
            sel = (conf >= lo) & (conf <= hi)
        else:
            sel = (conf >= lo) & (conf < hi)
        if not np.any(sel):
            continue
        acc = float(correct[sel].mean())
        avg_conf = float(conf[sel].mean())
        ece += (float(sel.sum()) / n) * abs(acc - avg_conf)
    return float(ece)

def brier_score(
    probs: np.ndarray,
    y_true: np.ndarray,
    n_classes: int = FUSION_DIRECTION_CARDINALITY,
) -> float:
    onehot = np.eye(int(n_classes), dtype=np.float64)[np.clip(y_true, 0, n_classes - 1)]
    return float(np.mean(np.sum((probs - onehot) ** 2, axis=-1)))

def paper_pnl(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Secondary metric only. +1 correct 2-class direction, -1 wrong. Skip < 0."""
    valid = (y_true >= 0) & (y_pred >= 0)
    if not np.any(valid):
        return 0.0
    wins = y_pred[valid] == y_true[valid]
    return float(np.mean(np.where(wins, 1.0, -1.0)))

def fit_temperature(logits: np.ndarray, y_true: np.ndarray) -> float:
    """One-parameter temperature on validation logits (NLL). Never uses test."""
    valid = y_true >= 0
    if not np.any(valid):
        return 1.0
    z = torch.tensor(logits[valid], dtype=torch.float32)
    y = torch.tensor(y_true[valid], dtype=torch.long)
    log_t = torch.nn.Parameter(torch.zeros(()))
    opt = torch.optim.LBFGS([log_t], lr=0.25, max_iter=50)

    def _closure() -> torch.Tensor:
        opt.zero_grad()
        temp = torch.exp(log_t).clamp(0.05, 10.0)
        loss = torch.nn.functional.cross_entropy(z / temp, y)
        loss.backward()
        return loss

    opt.step(_closure)
    return float(torch.exp(log_t).clamp(0.05, 10.0).detach().item())

def apply_temperature(logits: np.ndarray, temperature: float) -> np.ndarray:
    t = max(float(temperature), 1e-6)
    return softmax_np(logits / t)

def grade_horizon(
    *,
    balanced_acc: float,
    ece: float,
    fold_std: float = 0.0,
    high_acc: float = FUSION_HIGH_BALANCED_ACC,
    high_ece: float = FUSION_HIGH_MAX_ECE,
    medium_acc: float = FUSION_MEDIUM_BALANCED_ACC,
) -> str:
    """Map OOS metrics to HIGH / MEDIUM / LOW. Unstable folds cannot be HIGH."""
    if float(fold_std) > 0.08:
        if float(balanced_acc) >= float(medium_acc):
            return FUSION_GRADE_MEDIUM
        return FUSION_GRADE_LOW
    if float(balanced_acc) >= float(high_acc) and float(ece) <= float(high_ece):
        return FUSION_GRADE_HIGH
    if float(balanced_acc) >= float(medium_acc):
        return FUSION_GRADE_MEDIUM
    return FUSION_GRADE_LOW

def horizon_metrics(
    logits: np.ndarray,
    y_true: np.ndarray,
    *,
    temperature: float = 1.0,
) -> Dict[str, float]:
    valid = y_true >= 0
    if not np.any(valid):
        return {
            "balanced_acc": 0.0,
            "macro_f1": 0.0,
            "ece": 1.0,
            "brier": 1.0,
            "paper_pnl": 0.0,
            "n": 0.0,
        }
    probs = apply_temperature(logits[valid], temperature)
    pred = probs.argmax(axis=-1)
    yt = y_true[valid]
    return {
        "balanced_acc": balanced_accuracy(yt, pred, FUSION_DIRECTION_CARDINALITY),
        "macro_f1": macro_f1(yt, pred, FUSION_DIRECTION_CARDINALITY),
        "ece": expected_calibration_error(probs, yt),
        "brier": brier_score(probs, yt, FUSION_DIRECTION_CARDINALITY),
        "paper_pnl": paper_pnl(yt, pred),
        "n": float(len(yt)),
    }

@torch.no_grad()
def predict_logits(
    model: MtfFusionTransformer,
    loader: DataLoader,
    device: torch.device,
) -> Tuple[np.ndarray, np.ndarray]:
    """Return (n, n_horizons, 2) logits and (n, n_horizons) labels."""
    model.eval()
    logit_chunks: List[np.ndarray] = []
    label_chunks: List[np.ndarray] = []
    for batch in loader:
        batch_d = tuple(t.to(device) for t in batch)
        outs = model(*batch_d[:-1])
        dir_logits = [o.cpu().numpy() for o in outs[:N_HORIZONS_SAFE]]
        stacked = np.stack(dir_logits, axis=1)
        logit_chunks.append(stacked)
        label_chunks.append(batch_d[-1].cpu().numpy())
    return np.concatenate(logit_chunks, axis=0), np.concatenate(label_chunks, axis=0)

N_HORIZONS_SAFE = len(FUSION_HORIZON_KEYS)

def slice_windows(
    windows: Mapping[str, np.ndarray],
    labels: np.ndarray,
    sl: slice,
) -> Tuple[Dict[str, np.ndarray], np.ndarray]:
    return {k: v[sl] for k, v in windows.items()}, labels[sl]

def make_loader(
    windows: Mapping[str, np.ndarray],
    labels: np.ndarray,
    *,
    batch_size: int,
    shuffle: bool,
) -> DataLoader:
    ds = MtfFusionDataset(dict(windows), labels)
    return DataLoader(ds, batch_size=int(batch_size), shuffle=shuffle)

def run_walk_forward(
    windows: Mapping[str, np.ndarray],
    labels: np.ndarray,
    *,
    n_features: int,
    config: Mapping[str, Any],
    device: torch.device,
) -> Dict[str, Any]:
    """Expanding-window folds on the development set only."""
    n = len(labels)
    folds = int(
        config.get("walk_forward_folds") or config.get("walk_forward_folds") or 3
    )
    embargo = int(
        config.get("walk_forward_embargo") or config.get("walk_forward_embargo") or 24
    )
    fold_rows: List[Dict[str, Any]] = []
    per_h_acc: Dict[str, List[float]] = {k: [] for k in FUSION_HORIZON_KEYS}
    for train_sl, val_sl in walk_forward_slices(n, folds=folds, embargo=embargo):
        tw, ty = slice_windows(windows, labels, train_sl)
        vw, vy = slice_windows(windows, labels, val_sl)
        model = fusion_model_from_config(n_features, config).to(device)
        train_loader = make_loader(
            tw, ty, batch_size=int(config.get("batch_size") or 64), shuffle=True
        )
        val_loader = make_loader(
            vw, vy, batch_size=int(config.get("batch_size") or 64), shuffle=False
        )
        extra = _fusion_train_extra(config)
        train_mtf_fusion(
            model,
            train_loader,
            val_loader,
            device=device,
            epochs=max(1, int(config.get("epochs") or 8) // 4),
            lr=float(config.get("lr") or 1e-4),
            weight_decay=float(config.get("weight_decay") or 1e-3),
            patience=max(2, int(config.get("early_stop_patience") or 5) // 2),
            **extra,
        )
        logits, y = predict_logits(model, val_loader, device)
        row: Dict[str, Any] = {}
        for j, key in enumerate(FUSION_HORIZON_KEYS):
            m = horizon_metrics(logits[:, j, :], y[:, j])
            row[key] = m
            per_h_acc[key].append(float(m["balanced_acc"]))
        fold_rows.append(row)
    summary: Dict[str, Any] = {"folds": fold_rows, "mean": {}, "std": {}}
    for key in FUSION_HORIZON_KEYS:
        vals = per_h_acc[key]
        summary["mean"][key] = float(np.mean(vals)) if vals else 0.0
        summary["std"][key] = float(np.std(vals)) if vals else 0.0
    return summary

def freeze_horizon_gates(
    val_logits: np.ndarray,
    val_y: np.ndarray,
    walk_forward: Mapping[str, Any],
    *,
    config: Mapping[str, Any],
) -> Dict[str, Any]:
    """Calibrate and grade each horizon on validation. Do not touch test."""
    gates: Dict[str, Any] = {}
    temps: Dict[str, float] = {}
    for j, key in enumerate(FUSION_HORIZON_KEYS):
        temp = fit_temperature(val_logits[:, j, :], val_y[:, j])
        temps[key] = temp
        m = horizon_metrics(val_logits[:, j, :], val_y[:, j], temperature=temp)
        fold_std = float((walk_forward.get("std") or {}).get(key) or 0.0)
        grade = grade_horizon(
            balanced_acc=float(m["balanced_acc"]),
            ece=float(m["ece"]),
            fold_std=fold_std,
            high_acc=float(config.get("high_balanced_acc") or FUSION_HIGH_BALANCED_ACC),
            high_ece=float(config.get("high_max_ece") or FUSION_HIGH_MAX_ECE),
            medium_acc=float(config.get("medium_balanced_acc") or FUSION_MEDIUM_BALANCED_ACC),
        )
        gates[key] = {
            **m,
            "temperature": temp,
            "validation_confidence": grade,
            "min_probability": float(config.get("min_probability") or FUSION_MIN_PROBABILITY),
            "accepted_grades": [FUSION_GRADE_HIGH, FUSION_GRADE_MEDIUM],
        }
    return {"horizons": gates, "temperatures": temps}

def export_fusion_bundle(
    model: MtfFusionTransformer,
    export_dir: Path,
    *,
    n_features: int,
    window_len: int,
    config: Mapping[str, Any],
    gates: Mapping[str, Any],
    fusion_weights: Sequence[float],
    test_metrics: Mapping[str, Any] | None = None,
) -> Tuple[Path, Path, Path]:
    """Write ONNX + feature_config + metadata for the single fused bundle."""
    export_dir.mkdir(parents=True, exist_ok=True)
    onnx_path = export_dir / FUSION_ONNX_FILENAME
    cfg_path = export_dir / TRANSFORMER_FEATURE_CONFIG_FILENAME
    meta_path = export_dir / TRANSFORMER_METADATA_FILENAME
    dummy = [
        torch.randn(1, int(window_len), int(n_features)) for _ in FUSION_INPUT_RESOLUTIONS
    ]
    input_names = [f"features_{res}" for res in FUSION_INPUT_RESOLUTIONS]
    output_names = list(ONNX_OUTPUT_NAMES_V11)
    dynamic_axes = {name: {0: "batch"} for name in input_names + output_names}
    model.cpu().eval()
    export_kwargs: Dict[str, Any] = {
        "input_names": input_names,
        "output_names": output_names,
        "dynamic_axes": dynamic_axes,
        "opset_version": 17,
        "export_params": True,
    }
    last_error: Optional[BaseException] = None
    exported = False
    for opset in (17, 14):
        export_kwargs["opset_version"] = int(opset)
        try:
            try:
                torch.onnx.export(
                    model,
                    tuple(dummy),
                    str(onnx_path),
                    dynamo=False,
                    **export_kwargs,
                )
            except TypeError:
                torch.onnx.export(
                    model, tuple(dummy), str(onnx_path), **export_kwargs
                )
            exported = True
            break
        except (TypeError, RuntimeError, ValueError) as exc:
            last_error = exc
    if not exported:
        raise RuntimeError(
            f"ONNX export failed for opset 17 and 14: {last_error}"
        ) from last_error
    try:
        import onnx

        onnx.checker.check_model(onnx.load(str(onnx_path)))
    except ImportError:
        pass

    feature_config = {
        "feature_contract_version": FEATURE_CONTRACT_VERSION_V11,
        "feature_cols": list(fusion_feature_cols()),
        "window_len": int(window_len),
        "resolutions": list(FUSION_INPUT_RESOLUTIONS),
        "horizon_keys": list(FUSION_HORIZON_KEYS),
        "direction_names": {str(k): v for k, v in FUSION_DIRECTION_NAMES.items()},
        "onnx_output_names": output_names,
        "input_names": input_names,
        "config": dict(config),
        "horizon_gates": dict(gates),
        "tf_fusion_weights": [float(x) for x in fusion_weights],
    }
    cfg_path.write_text(
        json.dumps(feature_config, indent=2, default=str), encoding="utf-8"
    )
    meta = {
        "version": "transformer_mtf_fusion_v11",
        "model_name": "jacksparrow_transformer_BTCUSD_mtf_fusion",
        "model_family": FUSION_MODEL_FAMILY,
        "symbol": str(config.get("symbol") or "BTCUSD"),
        "resolution": "mtf_fusion",
        "onnx_filename": FUSION_ONNX_FILENAME,
        "feature_config_filename": TRANSFORMER_FEATURE_CONFIG_FILENAME,
        "onnx_output_names": output_names,
        "tf_fusion_weights": [float(x) for x in fusion_weights],
        "horizon_gates": dict(gates),
        "test_metrics": dict(test_metrics or {}),
        "training_config": dict(config),
        "primary_signal_mode": "multi_horizon_position",
    }
    meta_path.write_text(json.dumps(meta, indent=2, default=str), encoding="utf-8")
    return onnx_path, cfg_path, meta_path

def purged_dev_test_split(
    windows: Mapping[str, np.ndarray],
    labels: np.ndarray,
    *,
    train_frac: float,
    val_frac: float,
    embargo_bars: int,
) -> Dict[str, Dict[str, np.ndarray]]:
    """Time-ordered train/val/test with embargo. Test is the final untouched tail."""
    packed = {**{f"x_{k}": v for k, v in windows.items()}, "y": labels}
    splits = split_purged_windows(
        packed, train_frac=train_frac, val_frac=val_frac, embargo_bars=embargo_bars
    )
    out: Dict[str, Dict[str, np.ndarray]] = {}
    for name, part in splits.items():
        out[name] = {
            "windows": {res: part[f"x_{res}"] for res in FUSION_INPUT_RESOLUTIONS},
            "labels": part["y"],
        }
    return out

def fusion_ready_to_promote(
    walk_forward: Mapping[str, Any],
    test_metrics: Mapping[str, Any],
    gates: Mapping[str, Any],
) -> Dict[str, Any]:
    """True if at least one head is MEDIUM+ on walk-forward mean and frozen test.

    HIGH still requires ECE and fold std via ``grade_horizon``. Live must not
    load a bundle when this returns ready=False.
    """
    wf_mean = dict(walk_forward.get("mean") or {})
    wf_std = dict(walk_forward.get("std") or {})
    gate_map = dict(gates.get("horizons") or gates)
    ready_heads: List[str] = []
    detail: Dict[str, Any] = {}
    for key in FUSION_HORIZON_KEYS:
        test_m = dict(test_metrics.get(key) or {})
        test_acc = float(test_m.get("balanced_acc") or 0.0)
        test_ece = float(test_m.get("ece") or 1.0)
        wf_acc = float(wf_mean.get(key) or 0.0)
        fold_std = float(wf_std.get(key) or 0.0)
        wf_grade = grade_horizon(
            balanced_acc=wf_acc,
            ece=test_ece,
            fold_std=fold_std,
        )
        test_grade = str(
            (gate_map.get(key) or {}).get("validation_confidence") or ""
        )
        if not test_grade:
            test_grade = grade_horizon(balanced_acc=test_acc, ece=test_ece)
        accepted = {FUSION_GRADE_HIGH, FUSION_GRADE_MEDIUM}
        ok = wf_grade in accepted and test_grade in accepted
        if ok:
            ready_heads.append(key)
        detail[key] = {
            "walk_forward_mean_acc": wf_acc,
            "walk_forward_grade": wf_grade,
            "test_acc": test_acc,
            "test_grade": test_grade,
            "ok": ok,
        }
    ready = bool(ready_heads)
    return {
        "ready": ready,
        "heads": ready_heads,
        "detail": detail,
        "reason": (
            "at least one head MEDIUM+ on walk-forward mean and frozen test"
            if ready
            else "no head is MEDIUM on walk-forward mean and frozen test; do not promote"
        ),
    }

def default_config() -> Dict[str, Any]:
    cfg = default_fusion_training_config()
    cfg["window_len"] = FUSION_WINDOW_LEN
    cfg["n_classes"] = FUSION_DIRECTION_CARDINALITY
    return cfg


## 02 CONFIG

In [ ]:
CONFIG = default_fusion_training_config()
# Smoke overrides (comment out for a full research run):
# CONFIG["epochs"] = 2
# CONFIG["history_days"] = 120
CONFIG["run_optuna"] = True
CONFIG["optuna_trials"] = 8
CONFIG["run_shap"] = True
CONFIG["shap_background"] = 32
CONFIG["shap_explain_n"] = 64
CONFIG["run_walk_forward"] = True

_content = Path("/content")
_root = _content if _content.is_dir() else Path(".")
export_dir = _root / "export" / FUSION_BUNDLE_DIR_NAME
export_dir.mkdir(parents=True, exist_ok=True)
cache_dir = _root / "cache"
cache_dir.mkdir(parents=True, exist_ok=True)
# Section 11 writes TF window memmaps under cache_dir/fusion_windows.
print(json.dumps(CONFIG, indent=2, default=str))
print("contract", FEATURE_CONTRACT_VERSION_V11)
print("input TFs", list(FUSION_INPUT_RESOLUTIONS))
print("horizons", list(FUSION_HORIZON_KEYS))


## 03 Seeds

In [ ]:
set_research_seed(int(CONFIG.get("seed") or 42))
print("seed", CONFIG.get("seed") or 42)


## 04 Load OHLCV

In [ ]:
refresh_data = False
native_tfs = ("5m", "30m", "1h", "2h")
raw_frames = {}
for res in native_tfs:
    parquet = cache_dir / f"btcusd_{res}_raw.parquet"
    if parquet.is_file() and not refresh_data:
        raw_frames[res] = pd.read_parquet(parquet)
        print(f"Loaded cache {parquet} rows={len(raw_frames[res])}")
        continue
    raw_frames[res] = fetch_history_bundle(
        symbol=str(CONFIG.get("symbol") or "BTCUSD"),
        resolution=res,
        history_days=int(CONFIG.get("history_days") or 900),
        base_url=str(CONFIG.get("base_url") or "https://api.india.delta.exchange"),
    )
    raw_frames[res].to_parquet(parquet, index=False)
    print(f"Fetched and cached {parquet} rows={len(raw_frames[res])}")

frames = fusion_frames_from_fetch(
    raw_frames["5m"], raw_frames["30m"], raw_frames["1h"], raw_frames["2h"]
)
print("fusion frames", {k: len(v) for k, v in frames.items()})
print("10m is two closed 5m bars, not a model-side resample.")
assert set(FUSION_INPUT_RESOLUTIONS) <= set(frames)
assert "15m" not in frames


## 05 Data quality

In [ ]:
quality = {}
for res, df in frames.items():
    quality[res] = validate_ohlcv_completeness(
        df, res, symbol=str(CONFIG.get("symbol") or "BTCUSD")
    )
    print(res, json.dumps(quality[res], indent=2, default=str))
sample_t = pd.to_datetime(frames["5m"]["time"].iloc[-2], utc=True)
assert_no_lookahead(frames["30m"], sample_t, resolution_minutes=30)
assert_no_lookahead(frames["1h"], sample_t, resolution_minutes=60)
print("as-of join uses closed bars only; sample lookahead check passed.")


## 06 Native TF encodings

In [ ]:
preview = add_native_tf_features(
    frames["5m"].tail(400).reset_index(drop=True), resolution="5m"
)
feature_cols = list(fusion_feature_cols())
htf_cols = [c for c in preview.columns if str(c).startswith("htf_")]
print(f"preview rows={len(preview)} n_features={len(feature_cols)}")
print("htf_ resampled columns (must be empty):", htf_cols)
if htf_cols:
    raise RuntimeError("Fusion encodings must not resample HTF structure from 5m")
print("native TFs", list(FUSION_INPUT_RESOLUTIONS))
print(preview[feature_cols[:8]].tail(2))


## 07 Leakage audit

In [ ]:
leakage_audit(feature_cols)
print("Leakage audit passed: no t+1 / horizon dir / resampled HTF columns in X.")
print("last_swing_dir is a causal structure input, not a horizon target.")


## 08 Fusion horizon labels

In [ ]:
labeled_5m = compute_fusion_horizon_labels(frames["5m"])
labeled_5m = trim_fusion_label_tail(labeled_5m)
y_preview = fusion_label_matrix(labeled_5m)
print("label rows", len(labeled_5m), "shape", y_preview.shape)
print("2-class mix BEAR/BULL plus ignore_rate (NEUTRAL) per horizon:")
print(json.dumps(label_class_mix(y_preview), indent=2))
print("dead zone: h30m/h1h 0.5 ATR, h2h 0.75 ATR; NEUTRAL is ignore_index.")


## 09 Temporal split

In [ ]:
window_len = int(CONFIG.get("window_len") or FUSION_WINDOW_LEN)
stride = int(CONFIG.get("stride") or 4)
embargo = int(CONFIG.get("embargo_bars") or FUSION_EMBARGO_BARS)
print("window_len", window_len, "stride", stride, "embargo", embargo)
print("Test is the final untouched tail after a purged embargo.")


## 10 Scaler

In [ ]:
print("Scaler: per-window z-score inside collect_training_windows (zscore=True).")
print("No global scaler is fit on val or test.")
per_window = True


## 11 Sequence datasets

In [ ]:
windows, labels, decision_times = build_dataset_from_ohlcv(
    frames,
    window_len=window_len,
    stride=stride,
    memmap_dir=cache_dir / "fusion_windows",
)
print("windows", {k: v.shape for k, v in windows.items()})
print("labels", labels.shape, "decisions", len(decision_times))
splits = purged_dev_test_split(
    windows,
    labels,
    train_frac=float(CONFIG.get("train_frac") or 0.70),
    val_frac=float(CONFIG.get("val_frac") or 0.15),
    embargo_bars=embargo,
)
print("split sizes", {k: len(v["labels"]) for k, v in splits.items()})
n_features = len(fusion_feature_cols())
feature_finite_report(splits["train"]["windows"]["5m"], feature_cols)
class_w = inverse_frequency_class_weights(
    splits["train"]["labels"], FUSION_DIRECTION_CARDINALITY
)
print("dir class weights", np.round(class_w, 3).tolist())


## 12 Optuna

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_loader = make_loader(
    splits["train"]["windows"],
    splits["train"]["labels"],
    batch_size=int(CONFIG.get("batch_size") or 64),
    shuffle=True,
)
val_loader = make_loader(
    splits["val"]["windows"],
    splits["val"]["labels"],
    batch_size=int(CONFIG.get("batch_size") or 64),
    shuffle=False,
)
test_loader = make_loader(
    splits["test"]["windows"],
    splits["test"]["labels"],
    batch_size=int(CONFIG.get("batch_size") or 64),
    shuffle=False,
)
print("Loaders ready. Test is frozen until the final test section.")
CONFIG = optuna_search(
    CONFIG,
    n_features=n_features,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    class_weights=torch.tensor(class_w, dtype=torch.float32),
)
print(
    "post-optuna",
    json.dumps(
        {
            "lr": CONFIG.get("lr"),
            "dropout": CONFIG.get("dropout"),
            "weight_decay": CONFIG.get("weight_decay"),
            "optuna_best": CONFIG.get("optuna_best"),
        },
        indent=2,
        default=str,
    ),
)


## 13 Fusion transformer

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = fusion_model_from_config(n_features, CONFIG).to(device)
print(model)
print("device", device)
print("dropout", CONFIG.get("dropout"), "lr", CONFIG.get("lr"))
print("heads", list(FUSION_HORIZON_KEYS), "classes BEAR/BULL (NEUTRAL ignored)")


## 14 Cross-entropy loss

In [ ]:
print("Loss is weighted CE across three 2-class horizon heads.")
print("horizon_loss_weights", CONFIG.get("horizon_loss_weights"))
print("label_smoothing", CONFIG.get("label_smoothing"))
print("No PnL loss. NEUTRAL labels (<0) are ignored in CE.")
print("class weights", np.round(class_w, 3).tolist())


## 15 Train + early stopping

In [ ]:
train_hist = train_mtf_fusion(
    model,
    train_loader,
    val_loader,
    device=device,
    epochs=int(CONFIG.get("epochs") or 40),
    lr=float(CONFIG.get("lr") or 1e-4),
    weight_decay=float(CONFIG.get("weight_decay") or 1e-3),
    patience=int(CONFIG.get("early_stop_patience") or 5),
    class_weights=torch.tensor(class_w, dtype=torch.float32),
    label_smoothing=float(CONFIG.get("label_smoothing") or 0.0),
    horizon_weights=list(CONFIG.get("horizon_loss_weights") or [1.0, 0.8, 0.4]),
    lr_schedule=str(CONFIG.get("lr_schedule") or "cosine"),
)
print(train_hist)
if not train_hist.get("ok"):
    raise RuntimeError(f"Training aborted with non-finite loss: {train_hist}")


## 16 Validation metrics

In [ ]:
print("Validation (used for early stopping, temperature, and grades):")
val_logits, val_y = predict_logits(model, val_loader, device)
val_metrics = {}
for j, key in enumerate(FUSION_HORIZON_KEYS):
    val_metrics[key] = horizon_metrics(val_logits[:, j, :], val_y[:, j])
    m = val_metrics[key]
    print(
        f"  {key}: acc={m['balanced_acc']:.3f} f1={m['macro_f1']:.3f} "
        f"ece={m['ece']:.3f} n={int(m['n'])}"
    )


## 17 Test hold

In [ ]:
print("Test split is frozen until section 24. Do not score it during search.")
print("test windows", {k: v.shape for k, v in splits["test"]["windows"].items()})


## 18 Walk-forward

In [ ]:
tf_keys = tuple(splits["train"]["windows"].keys())
n_dev = int(len(splits["train"]["labels"]) + len(splits["val"]["labels"]))
n_folds = int(CONFIG.get("walk_forward_folds") or 3)
fold_embargo = int(CONFIG.get("walk_forward_embargo") or embargo)
print("walk-forward TFs", tf_keys)
print("dev samples (train+val)", n_dev, "folds", n_folds, "embargo", fold_embargo)
print("test is excluded from walk-forward")
if CONFIG.get("run_walk_forward"):
    dev_windows = {
        res: np.concatenate(
            [splits["train"]["windows"][res], splits["val"]["windows"][res]],
            axis=0,
        )
        for res in tf_keys
    }
    dev_labels = np.concatenate(
        [splits["train"]["labels"], splits["val"]["labels"]], axis=0
    )
    walk_forward = run_walk_forward(
        dev_windows,
        dev_labels,
        n_features=n_features,
        config=CONFIG,
        device=device,
    )
    print(json.dumps(walk_forward.get("mean") or {}, indent=2, default=str))
    del dev_windows, dev_labels
else:
    folds = walk_forward_slices(n_dev, folds=n_folds, embargo=fold_embargo)
    fold_spans = [(s.start, s.stop, v.start, v.stop) for s, v in folds]
    print("Walk-forward skipped (CONFIG run_walk_forward=False). Fold plan:", fold_spans)
    walk_forward = {"folds": [], "mean": {}, "std": {}}


## 19 Horizon confusion

In [ ]:
print("Validation confusion (true x pred) per horizon, 0=BEAR 1=BULL:")
for j, key in enumerate(FUSION_HORIZON_KEYS):
    pred = val_logits[:, j, :].argmax(axis=-1)
    mat = confusion_counts(pred, val_y[:, j], FUSION_DIRECTION_CARDINALITY)
    print(key)
    print(mat)


## 20 Fusion weights

In [ ]:
fusion_w = model.fusion_weights().detach().cpu().numpy()
tf_keys = tuple(splits["train"]["windows"].keys())
print("softmax TF fusion weights:")
for res, w in zip(tf_keys, fusion_w):
    print(f"  {res}: {float(w):.4f}")


## 21 Temperature calibration

In [ ]:
print("Fit one temperature per horizon on validation logits. Never uses test.")
gates = freeze_horizon_gates(val_logits, val_y, walk_forward, config=CONFIG)
for key, row in gates["horizons"].items():
    print(
        f"  {key}: T={row['temperature']:.3f} grade={row['validation_confidence']} "
        f"acc={row['balanced_acc']:.3f} ece={row['ece']:.3f}"
    )


## 22 Horizon grades

In [ ]:
print("HIGH / MEDIUM may trade; LOW is telemetry-only.")
print("Each head is gated independently. Do not pick max-prob across horizons.")
print("Duration = longest accepted same-side horizon; SL/TP = ATR at that duration.")
for key, row in gates["horizons"].items():
    print(key, row["validation_confidence"], "min_p", row["min_probability"])


## 23 Re-train

In [ ]:
print("Main train already uses Optuna winners when run_optuna=True.")
print("No second fit on train+val. Test split stays frozen.")


## 24 Final untouched test

In [ ]:
print("Final untouched test (frozen weights + frozen gates):")
test_logits, test_y = predict_logits(model, test_loader, device)
final_test = {}
for j, key in enumerate(FUSION_HORIZON_KEYS):
    temp = float(gates["horizons"][key]["temperature"])
    final_test[key] = horizon_metrics(
        test_logits[:, j, :], test_y[:, j], temperature=temp
    )
    m = final_test[key]
    print(
        f"  {key}: acc={m['balanced_acc']:.3f} f1={m['macro_f1']:.3f} "
        f"ece={m['ece']:.3f} pnl={m['paper_pnl']:.3f}"
    )
promo = fusion_ready_to_promote(walk_forward, final_test, gates)
print(json.dumps(promo, indent=2, default=str))
if not promo["ready"]:
    print("DO NOT PROMOTE: no head is MEDIUM on walk-forward mean and frozen test.")


## 25 Save

In [ ]:
shap_report = {}
if not train_hist.get("ok"):
    print("Skip save: training did not succeed", train_hist)
else:
    try:
        shap_report = shap_grouped_stub(
            feature_cols,
            enabled=bool(CONFIG.get("run_shap")),
            model=model,
            val_windows=splits["val"]["windows"],
            device=device,
            config=CONFIG,
        )
    except Exception as exc:
        print("SHAP failed; continuing export:", type(exc).__name__, exc)
        shap_report = {
            "ok": False,
            "enabled": True,
            "reason": f"{type(exc).__name__}: {exc}",
            "horizons": {},
        }
    artifact = {
        "feature_cols": list(feature_cols),
        "seed": CONFIG.get("seed"),
        "window_len": window_len,
        "feature_contract_version": FEATURE_CONTRACT_VERSION_V11,
        "resolutions": list(FUSION_INPUT_RESOLUTIONS),
        "horizon_keys": list(FUSION_HORIZON_KEYS),
        "tf_fusion_weights": [float(x) for x in fusion_w],
        "val_metrics": val_metrics,
        "horizon_gates": gates,
        "test_metrics": final_test,
        "shap_report": shap_report,
        "optuna_best": CONFIG.get("optuna_best"),
    }
    (export_dir / "research_run.json").write_text(
        json.dumps(artifact, indent=2, default=str), encoding="utf-8"
    )
    print("Wrote", export_dir / "research_run.json")


## 26 JackSparrow export

In [ ]:
if not train_hist.get("ok"):
    print("Skip ONNX export: training did not succeed", train_hist)
else:
    if not shap_report:
        try:
            shap_report = shap_grouped_stub(
                feature_cols,
                enabled=bool(CONFIG.get("run_shap")),
                model=model,
                val_windows=splits["val"]["windows"],
                device=device,
                config=CONFIG,
            )
        except Exception as exc:
            print("SHAP retry failed; exporting anyway:", type(exc).__name__, exc)
            shap_report = {"ok": False, "reason": str(exc), "horizons": {}}
    onnx_path, cfg_path, meta_path = export_fusion_bundle(
        model,
        export_dir,
        n_features=n_features,
        window_len=window_len,
        config=CONFIG,
        gates=gates,
        fusion_weights=fusion_w.tolist(),
        test_metrics=final_test,
    )
    print("Exported", onnx_path)
    print("feature_config", cfg_path)
    print("metadata", meta_path)
    print("Copy this directory into agent/model_storage/ after validation.")
    import shutil
    zip_stem = export_dir.parent / FUSION_BUNDLE_DIR_NAME
    zip_path = Path(shutil.make_archive(str(zip_stem), "zip", root_dir=export_dir))
    print("Wrote zip", zip_path)
    try:
        from google.colab import files
        files.download(str(zip_path))
    except Exception as exc:
        print("Browser download skipped:", exc)
        print("Zip remains at", zip_path)


## Notes

- Historical OHLCV is **immutable**. Training updates **weights only**.
- 10m is assembled from two closed 5m bars **outside** the model.
- Each TF runs candle/chart/structure engines on its **native** grid. No HTF resample.
- Walk-forward, Optuna, and temperature fitting never see the final test split.
- Gate each horizon independently. Duration is the longest accepted same-side head.
- SL/TP are ATR scaled to that duration (path heads were dropped).
- Promote **only** if at least one head is MEDIUM on walk-forward **mean** and
  frozen test (HIGH still needs ECE and fold std). Until then live stays on the
  current gated stack. Paper PnL is a secondary diagnostic, not the training loss.
- Section 26 zips the export folder and starts a Colab download on success.
